# **theratime_v06**

In [1]:
# ============================================================
# TheraTime v0.6-neurocomputing
# Therapeutic Timing Evaluation for Mental-Health Response Retrieval
#
# Target venue: Neurocomputing software paper
#
# Key fixes in this version:
# 1. Dense retrieval uses all-MiniLM-L6-v2.
# 2. Default timing classifier uses paraphrase-mpnet-base-v2.
# 3. Retrieval and timing encoders are separated to reduce circular encoder bias.
# 4. Self-retrieval exclusion is enabled.
# 5. Stress-test positive control uses pure distress language.
# 6. TTP@2 and Hit@2 are both reported correctly.
# 7. Entropy uses softmax-normalized prototype scores.
#
# Responsible use:
# This is an offline research evaluation framework only.
# It is not a clinical decision-support tool.
# It is not validated for use with real users in therapy or crisis settings.
# ============================================================

!pip -q install datasets sentence-transformers scikit-learn pandas matplotlib tqdm rank-bm25 scipy openpyxl

import json
import re
import warnings
from pathlib import Path
from dataclasses import dataclass, asdict, field
from typing import List, Dict, Any, Optional, Tuple

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from tqdm.auto import tqdm
from scipy.stats import pointbiserialr, pearsonr
from scipy.stats import chi2 as chi2_dist

from datasets import load_dataset
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from sentence_transformers import SentenceTransformer
from rank_bm25 import BM25Okapi

warnings.filterwarnings("ignore", category=FutureWarning)


# ============================================================
# SECTION 1 - Configuration
# ============================================================

SOFTWARE_NAME = "TheraTime"
SOFTWARE_VERSION = "0.6-neurocomputing"

DATASETS_USED = [
    "thu-coai/esconv",
    "nbertagnolli/counsel-chat",
    "ShenLab/MentalChat16K",
]

OUTPUT_DIR = Path("/kaggle/working/theratime_v06_outputs")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

TOP_K_RETRIEVAL = 5
MAX_QUERIES_PER_DATASET = 250

N_BOOTSTRAP = 2000
CONFIDENCE_PERCENTILE = 10.0
RANDOM_SEED = 42

# Timing encoders
DEFAULT_TIMING_ENCODER = "sentence-transformers/paraphrase-mpnet-base-v2"
ABLATION_TIMING_ENCODER = "sentence-transformers/all-MiniLM-L6-v2"

# Dense retrieval encoder separated from default timing encoder
DENSE_RETRIEVAL_ENCODER = "sentence-transformers/all-MiniLM-L6-v2"

CONFIDENCE_FALLBACK = 0.03


def compute_adaptive_threshold(
    margins: List[float],
    pct: float = CONFIDENCE_PERCENTILE,
) -> float:
    if not margins:
        return CONFIDENCE_FALLBACK
    return float(np.percentile(margins, pct))


def softmax(x: np.ndarray) -> np.ndarray:
    x = np.array(x, dtype=float)
    x = x - np.max(x)
    e = np.exp(x)
    return e / np.sum(e)


# ============================================================
# SECTION 2 - Input quality filter
# ============================================================

MIN_QUERY_WORDS = 6
MAX_QUERY_WORDS = 120
MIN_ANSWER_WORDS = 5

_GREETING_RE = re.compile(
    r"^(hello|hi|hey|good\s+(morning|afternoon|evening|day)|"
    r"thanks|thank\s+you|okay|ok|sure|great|wonderful|"
    r"lol|haha|wow|yes|no|alright|cool|nice)[\s!.,?]*$",
    re.IGNORECASE,
)

_FRAGMENT_RE = re.compile(r"^[\w\s']{1,5}[.!?,]*$")


def is_valid_pair(q: str, a: str) -> bool:
    q = str(q).strip()
    a = str(a).strip()

    q_words = len(q.split())
    a_words = len(a.split())

    if q_words < MIN_QUERY_WORDS or q_words > MAX_QUERY_WORDS:
        return False

    if a_words < MIN_ANSWER_WORDS:
        return False

    if _GREETING_RE.match(q):
        return False

    if _FRAGMENT_RE.match(q):
        return False

    if re.match(r"^\d+\s+\w+", q):
        return False

    return True


# ============================================================
# SECTION 3 - Taxonomies
# ============================================================

SUPPORT_STAGES: Dict[str, str] = {
    "distress_disclosure": (
        "I feel sad, lonely, scared, hurt, or emotionally distressed. "
        "I am sharing my pain for the first time. I need someone to hear me "
        "and acknowledge what I am going through. I have not asked for concrete help yet."
    ),
    "high_emotional_intensity": (
        "I am overwhelmed, panicking, unable to stop crying, spiraling, or falling apart. "
        "My emotional state is very intense and I need immediate calming, grounding, "
        "or stabilizing support before advice."
    ),
    "unclear_need": (
        "Something feels wrong but I cannot identify or explain what it is. "
        "I do not know what kind of support I need. I am confused about my feelings "
        "or about where to start."
    ),
    "advice_seeking": (
        "What should I do? How can I handle this? I am explicitly asking for "
        "practical steps, strategies, coping techniques, or concrete actions."
    ),
    "psychoeducation_seeking": (
        "Why do I feel this way? What causes this? Can you explain what is happening "
        "to me mentally or emotionally? I want to understand my experience."
    ),
    "crisis_safety": (
        "I cannot keep myself safe. I am thinking about suicide or hurting myself. "
        "I need emergency help right now. This is a safety crisis, not just distress."
    ),
    "followup_problem_solving": (
        "I have already received support and now I need the next steps. "
        "I am ready to move from emotional support to planning, action, or follow-up."
    ),
}


SUPPORT_MOVES: Dict[str, str] = {
    "validation": (
        "The response validates the user's feelings and says that the user's emotional "
        "reaction makes sense and is understandable."
    ),
    "empathy": (
        "The response expresses warmth, care, and emotional presence. "
        "It communicates that the supporter is with the user and cares about them."
    ),
    "reflective_listening": (
        "The response reflects or restates what the user is feeling or experiencing, "
        "showing that the supporter has heard and understood."
    ),
    "clarification": (
        "The response asks a gentle, open question to better understand the user's "
        "situation, feelings, or needs before proceeding."
    ),
    "grounding": (
        "The response helps the user calm down, breathe, stabilize emotions, "
        "or focus on the present moment using grounding techniques."
    ),
    "practical_advice": (
        "The response gives concrete coping steps, behavioral strategies, habits, "
        "or action plans the user can try."
    ),
    "psychoeducation": (
        "The response explains mental-health symptoms, emotional processes, "
        "or why certain feelings occur."
    ),
    "encouragement": (
        "The response offers hope, reassurance, motivation, or positive affirmation "
        "of the user's ability to cope or improve."
    ),
    "safety_referral": (
        "The response directs the user toward emergency help, a crisis line, "
        "a trusted person, or professional safety support. It prioritizes immediate safety."
    ),
}


TIMING_LABELS = [
    "well_timed",
    "premature_advice",
    "delayed_safety",
    "over_validation",
    "missing_clarification",
    "stage_mismatch",
]


ALLOWED_MOVES: Dict[str, set] = {
    "distress_disclosure": {
        "validation",
        "empathy",
        "reflective_listening",
        "clarification",
        "encouragement",
    },
    "high_emotional_intensity": {
        "validation",
        "empathy",
        "reflective_listening",
        "grounding",
        "clarification",
    },
    "unclear_need": {
        "validation",
        "empathy",
        "reflective_listening",
        "clarification",
    },
    "advice_seeking": {
        "practical_advice",
        "psychoeducation",
        "encouragement",
        "validation",
    },
    "psychoeducation_seeking": {
        "psychoeducation",
        "validation",
        "clarification",
    },
    "crisis_safety": {
        "safety_referral",
        "grounding",
        "validation",
    },
    "followup_problem_solving": {
        "practical_advice",
        "psychoeducation",
        "encouragement",
    },
}


PRIMARY_MOVE: Dict[str, str] = {
    "distress_disclosure": "validation",
    "high_emotional_intensity": "grounding",
    "unclear_need": "clarification",
    "advice_seeking": "practical_advice",
    "psychoeducation_seeking": "psychoeducation",
    "crisis_safety": "safety_referral",
    "followup_problem_solving": "practical_advice",
}


# ============================================================
# SECTION 4 - Keyword ablation baseline
# ============================================================

_STAGE_KEYWORDS: Dict[str, List[str]] = {
    "crisis_safety": [
        "suicid",
        "self-harm",
        "hurt myself",
        "can't stay safe",
        "can not stay safe",
        "end my life",
        "kill myself",
        "don't want to live",
        "want to die",
        "unsafe tonight",
    ],
    "advice_seeking": [
        "what should i do",
        "how can i",
        "what steps",
        "any advice",
        "what do you suggest",
        "what can i try",
        "what would you recommend",
        "give me some tips",
    ],
    "psychoeducation_seeking": [
        "why do i feel",
        "why does",
        "what causes",
        "can you explain",
        "what is anxiety",
        "why am i",
        "what happens when",
    ],
    "high_emotional_intensity": [
        "overwhelmed",
        "can't cope",
        "can not cope",
        "panicking",
        "cannot stop crying",
        "can not stop crying",
        "spiraling",
        "falling apart",
        "breaking down",
        "cannot breathe",
    ],
    "distress_disclosure": [
        "i feel",
        "i'm so",
        "i have been feeling",
        "i've been feeling",
        "i am sad",
        "i feel lonely",
        "i feel depressed",
        "i am scared",
    ],
    "followup_problem_solving": [
        "next step",
        "what now",
        "i tried that",
        "i've already tried",
        "what else can",
        "following up",
        "after that",
    ],
    "unclear_need": [
        "i don't know",
        "i'm not sure what",
        "something is wrong",
        "i can't explain",
        "i don't understand why",
        "i just feel",
    ],
}


_STAGE_PRIORITY = [
    "crisis_safety",
    "high_emotional_intensity",
    "advice_seeking",
    "psychoeducation_seeking",
    "followup_problem_solving",
    "distress_disclosure",
    "unclear_need",
]


def keyword_stage(text: str) -> str:
    t = str(text).lower()
    for stage in _STAGE_PRIORITY:
        if any(kw in t for kw in _STAGE_KEYWORDS[stage]):
            return stage
    return "unclear_need"


# ============================================================
# SECTION 5 - Data structures
# ============================================================

@dataclass
class QAItem:
    query_id: str
    query: str
    answer: str
    source_dataset: str


@dataclass
class Retrieved:
    response_id: str
    text: str
    score: float
    source_dataset: str


@dataclass
class Example:
    query_id: str
    query: str
    retrieved: List[Retrieved]
    source_dataset: str
    retrieval_method: str
    gold_answer: Optional[str] = None


@dataclass
class Judgment:
    query_id: str
    source_dataset: str
    retrieval_method: str
    response_id: str
    rank: int
    query: str
    response: str
    retrieval_score: float
    predicted_stage: str
    predicted_move: str
    timing_label: str
    is_well_timed: bool
    explanation: str
    stage_confidence: float
    move_confidence: float
    low_confidence: bool
    stage_scores: Dict[str, float] = field(default_factory=dict)
    move_scores: Dict[str, float] = field(default_factory=dict)


# ============================================================
# SECTION 6 - Prototype classifier
# ============================================================

class PrototypeClassifier:
    def __init__(self, descriptions: Dict[str, str], model_name: str):
        self.descriptions = descriptions
        self.labels = list(descriptions.keys())
        self.model_name = model_name
        self.model = SentenceTransformer(model_name)

        prototype_texts = [
            f"{label}: {description}"
            for label, description in descriptions.items()
        ]

        self.prototype_embeddings = self.model.encode(
            prototype_texts,
            normalize_embeddings=True,
        )

    def predict(self, text: str) -> Tuple[str, float, Dict[str, float]]:
        emb = self.model.encode([str(text)], normalize_embeddings=True)
        scores = cosine_similarity(emb, self.prototype_embeddings)[0]

        idx = np.argsort(scores)[::-1]
        top_label = self.labels[int(idx[0])]
        margin = float(scores[idx[0]] - scores[idx[1]])

        all_scores = {
            label: float(score)
            for label, score in zip(self.labels, scores)
        }

        return top_label, margin, all_scores


# ============================================================
# SECTION 7 - Timing rule engine
# ============================================================

def judge_timing(stage: str, move: str) -> Dict[str, Any]:
    allowed = ALLOWED_MOVES.get(stage, set())

    if stage == "crisis_safety" and move not in {
        "safety_referral",
        "grounding",
        "validation",
    }:
        return {
            "timing_label": "delayed_safety",
            "is_well_timed": False,
            "explanation": (
                "Safety-related stage detected, but the response does not provide "
                "safety referral, grounding, or validation."
            ),
        }

    if (
        stage in {"distress_disclosure", "high_emotional_intensity", "unclear_need"}
        and move == "practical_advice"
    ):
        return {
            "timing_label": "premature_advice",
            "is_well_timed": False,
            "explanation": (
                "The response gives practical advice before validating, grounding, "
                "or clarifying the user's emotional state."
            ),
        }

    if (
        stage in {"advice_seeking", "followup_problem_solving"}
        and move in {"validation", "empathy", "reflective_listening"}
    ):
        return {
            "timing_label": "over_validation",
            "is_well_timed": False,
            "explanation": (
                "The user is asking for practical help, but the response only "
                "validates, empathizes, or reflects without providing action."
            ),
        }

    if stage == "unclear_need" and move not in allowed:
        return {
            "timing_label": "missing_clarification",
            "is_well_timed": False,
            "explanation": (
                "The user's need is unclear, but the response acts without "
                "clarifying or offering supportive presence."
            ),
        }

    if move in allowed:
        return {
            "timing_label": "well_timed",
            "is_well_timed": True,
            "explanation": "The support move is compatible with the predicted support stage.",
        }

    return {
        "timing_label": "stage_mismatch",
        "is_well_timed": False,
        "explanation": (
            f"The move '{move}' is not compatible with the predicted stage '{stage}'."
        ),
    }


# ============================================================
# SECTION 8 - Evaluation pipelines
# ============================================================

class TheraTimePipeline:
    def __init__(
        self,
        stage_encoder: str = DEFAULT_TIMING_ENCODER,
        move_encoder: str = DEFAULT_TIMING_ENCODER,
        confidence_percentile: float = CONFIDENCE_PERCENTILE,
    ):
        self.confidence_percentile = confidence_percentile
        self.stage_threshold = CONFIDENCE_FALLBACK
        self.move_threshold = CONFIDENCE_FALLBACK

        print(f"Stage classifier: {stage_encoder}")
        self.stage_classifier = PrototypeClassifier(SUPPORT_STAGES, stage_encoder)

        print(f"Move classifier: {move_encoder}")
        self.move_classifier = PrototypeClassifier(SUPPORT_MOVES, move_encoder)

    def calibrate(self, examples: List[Example], sample_n: int = 300) -> None:
        stage_margins = []
        move_margins = []

        for ex in examples[:sample_n]:
            _, stage_margin, _ = self.stage_classifier.predict(ex.query)
            stage_margins.append(stage_margin)

            if ex.retrieved:
                _, move_margin, _ = self.move_classifier.predict(ex.retrieved[0].text)
                move_margins.append(move_margin)

        self.stage_threshold = compute_adaptive_threshold(
            stage_margins,
            self.confidence_percentile,
        )
        self.move_threshold = compute_adaptive_threshold(
            move_margins,
            self.confidence_percentile,
        )

        print(f"Stage threshold p{self.confidence_percentile:.0f}: {self.stage_threshold:.4f}")
        print(f"Move threshold p{self.confidence_percentile:.0f}: {self.move_threshold:.4f}")

    def evaluate_example(self, ex: Example, top_k: int = 5) -> List[Judgment]:
        stage, stage_conf, stage_scores = self.stage_classifier.predict(ex.query)
        judgments = []

        for rank, item in enumerate(ex.retrieved[:top_k], start=1):
            move, move_conf, move_scores = self.move_classifier.predict(item.text)
            timing = judge_timing(stage, move)

            low_confidence = (
                stage_conf < self.stage_threshold
                or move_conf < self.move_threshold
            )

            judgments.append(
                Judgment(
                    query_id=ex.query_id,
                    source_dataset=ex.source_dataset,
                    retrieval_method=ex.retrieval_method,
                    response_id=item.response_id,
                    rank=rank,
                    query=ex.query,
                    response=item.text,
                    retrieval_score=float(item.score),
                    predicted_stage=stage,
                    predicted_move=move,
                    timing_label=timing["timing_label"],
                    is_well_timed=bool(timing["is_well_timed"]),
                    explanation=timing["explanation"],
                    stage_confidence=round(stage_conf, 4),
                    move_confidence=round(move_conf, 4),
                    low_confidence=low_confidence,
                    stage_scores=stage_scores,
                    move_scores=move_scores,
                )
            )

        return judgments

    def run(
        self,
        examples: List[Example],
        top_k: int = 5,
        calibrate: bool = True,
    ) -> List[Judgment]:
        if calibrate and examples:
            self.calibrate(examples)

        all_judgments = []
        for ex in tqdm(examples, desc="TheraTime evaluation"):
            all_judgments.extend(self.evaluate_example(ex, top_k=top_k))

        return all_judgments


class KeywordPipeline:
    def __init__(
        self,
        move_encoder: str = DEFAULT_TIMING_ENCODER,
        confidence_percentile: float = CONFIDENCE_PERCENTILE,
    ):
        self.confidence_percentile = confidence_percentile
        self.move_threshold = CONFIDENCE_FALLBACK

        print(f"Keyword pipeline move classifier: {move_encoder}")
        self.move_classifier = PrototypeClassifier(SUPPORT_MOVES, move_encoder)

    def calibrate(self, examples: List[Example], sample_n: int = 300) -> None:
        move_margins = []

        for ex in examples[:sample_n]:
            if ex.retrieved:
                _, move_margin, _ = self.move_classifier.predict(ex.retrieved[0].text)
                move_margins.append(move_margin)

        self.move_threshold = compute_adaptive_threshold(
            move_margins,
            self.confidence_percentile,
        )

        print(f"Keyword pipeline move threshold p{self.confidence_percentile:.0f}: {self.move_threshold:.4f}")

    def run(
        self,
        examples: List[Example],
        top_k: int = 5,
        calibrate: bool = True,
    ) -> List[Judgment]:
        if calibrate and examples:
            self.calibrate(examples)

        all_judgments = []

        for ex in tqdm(examples, desc="Keyword ablation"):
            stage = keyword_stage(ex.query)

            for rank, item in enumerate(ex.retrieved[:top_k], start=1):
                move, move_conf, move_scores = self.move_classifier.predict(item.text)
                timing = judge_timing(stage, move)

                all_judgments.append(
                    Judgment(
                        query_id=ex.query_id,
                        source_dataset=ex.source_dataset,
                        retrieval_method=ex.retrieval_method,
                        response_id=item.response_id,
                        rank=rank,
                        query=ex.query,
                        response=item.text,
                        retrieval_score=float(item.score),
                        predicted_stage=stage,
                        predicted_move=move,
                        timing_label=timing["timing_label"],
                        is_well_timed=bool(timing["is_well_timed"]),
                        explanation=timing["explanation"],
                        stage_confidence=-1.0,
                        move_confidence=round(move_conf, 4),
                        low_confidence=move_conf < self.move_threshold,
                        stage_scores={},
                        move_scores=move_scores,
                    )
                )

        return all_judgments


# ============================================================
# SECTION 9 - Statistics
# ============================================================

def bootstrap_ci(
    values: np.ndarray,
    stat_fn=np.mean,
    n_bootstrap: int = N_BOOTSTRAP,
    alpha: float = 0.05,
    seed: int = RANDOM_SEED,
) -> Tuple[float, float, float]:
    rng = np.random.default_rng(seed)
    values = np.array(values, dtype=float)

    if len(values) == 0:
        return 0.0, 0.0, 0.0

    point = float(stat_fn(values))
    samples = []

    for _ in range(n_bootstrap):
        resampled = rng.choice(values, size=len(values), replace=True)
        samples.append(stat_fn(resampled))

    lower = float(np.percentile(samples, 100 * alpha / 2))
    upper = float(np.percentile(samples, 100 * (1 - alpha / 2)))

    return point, lower, upper


def cohens_d(a: np.ndarray, b: np.ndarray) -> float:
    a = np.array(a, dtype=float)
    b = np.array(b, dtype=float)

    if len(a) < 2 or len(b) < 2:
        return 0.0

    pooled_var = (
        ((len(a) - 1) * np.var(a, ddof=1))
        + ((len(b) - 1) * np.var(b, ddof=1))
    ) / (len(a) + len(b) - 2)

    return float((np.mean(a) - np.mean(b)) / np.sqrt(max(pooled_var, 1e-9)))


def mcnemar_test(j1: List[Judgment], j2: List[Judgment]) -> Dict[str, Any]:
    d1 = {
        (j.query_id, j.retrieval_method, j.rank): j.is_well_timed
        for j in j1
    }
    d2 = {
        (j.query_id, j.retrieval_method, j.rank): j.is_well_timed
        for j in j2
    }

    keys = sorted(set(d1.keys()) & set(d2.keys()))
    keys = [k for k in keys if k[2] == 1]

    b = sum(1 for k in keys if d1[k] and not d2[k])
    c = sum(1 for k in keys if not d1[k] and d2[k])

    n = b + c

    if n == 0:
        return {
            "b": int(b),
            "c": int(c),
            "chi2": 0.0,
            "p_value": 1.0,
            "significant_p05": False,
            "n_discordant": 0,
        }

    chi2_value = (abs(b - c) - 1.0) ** 2 / n
    p_value = float(1 - chi2_dist.cdf(chi2_value, df=1))

    return {
        "b": int(b),
        "c": int(c),
        "chi2": round(float(chi2_value), 4),
        "p_value": round(p_value, 4),
        "significant_p05": bool(p_value < 0.05),
        "n_discordant": int(n),
    }


def compute_metrics(judgments: List[Judgment], top_k: int = 5) -> Dict[str, Any]:
    df = pd.DataFrame([asdict(j) for j in judgments])

    if df.empty:
        return {}

    top1 = df[df["rank"] == 1].copy()
    topk = df[df["rank"] <= top_k].copy()
    top1_hc = top1[~top1["low_confidence"]].copy()

    wt = top1["is_well_timed"].astype(float).values
    tta1, tta1_lo, tta1_hi = bootstrap_ci(wt)

    d_vs_05 = cohens_d(wt, np.full(len(wt), 0.5))
    n_queries = max(int(top1["query_id"].nunique()), 1)

    metrics = {
        "n_queries": int(top1["query_id"].nunique()),
        "n_judgments": int(len(df)),
        "TTA@1": round(tta1, 4),
        "TTA@1_CI_lo": round(tta1_lo, 4),
        "TTA@1_CI_hi": round(tta1_hi, 4),
        f"TTP@{top_k}": round(float(topk["is_well_timed"].mean()), 4),
        f"Hit@{top_k}": round(
            float(topk.groupby("query_id")["is_well_timed"].any().mean()),
            4,
        ),
        "cohens_d_vs_0.5": round(d_vs_05, 4),
        "n_high_confidence": int(len(top1_hc)),
        "TTA@1_high_confidence": (
            round(float(top1_hc["is_well_timed"].mean()), 4)
            if len(top1_hc)
            else 0.0
        ),
        "low_confidence_rate": round(float(top1["low_confidence"].mean()), 4),
        "mean_stage_confidence": round(
            float(top1[top1["stage_confidence"] >= 0]["stage_confidence"].mean()),
            4,
        ),
        "mean_move_confidence": round(
            float(top1[top1["move_confidence"] >= 0]["move_confidence"].mean()),
            4,
        ),
    }

    for label in [
        "premature_advice",
        "delayed_safety",
        "over_validation",
        "missing_clarification",
        "stage_mismatch",
    ]:
        metrics[f"{label}_rate"] = round(
            float((top1["timing_label"] == label).sum() / n_queries),
            4,
        )

    return metrics


def timing_hit_at_k(judgments: List[Judgment], k: int = 2) -> float:
    df = pd.DataFrame([asdict(j) for j in judgments])

    if df.empty:
        return 0.0

    topk = df[df["rank"] <= k].copy()
    return float(topk.groupby("query_id")["is_well_timed"].any().mean())


# ============================================================
# SECTION 10 - Geometry and diagnostics
# ============================================================

def embedding_geometry_analysis(judgments: List[Judgment]) -> pd.DataFrame:
    rows = []

    for j in judgments:
        if j.rank != 1:
            continue

        if not j.stage_scores or not j.move_scores:
            continue

        stage_raw = np.array(list(j.stage_scores.values()), dtype=float)
        move_raw = np.array(list(j.move_scores.values()), dtype=float)

        stage_probs = softmax(stage_raw)
        move_probs = softmax(move_raw)

        stage_entropy = float(-np.sum(stage_probs * np.log(stage_probs + 1e-9)))
        move_entropy = float(-np.sum(move_probs * np.log(move_probs + 1e-9)))

        rows.append(
            {
                "query_id": j.query_id,
                "source_dataset": j.source_dataset,
                "retrieval_method": j.retrieval_method,
                "retrieval_score": j.retrieval_score,
                "is_well_timed": int(j.is_well_timed),
                "timing_label": j.timing_label,
                "predicted_stage": j.predicted_stage,
                "predicted_move": j.predicted_move,
                "max_stage_score": float(stage_raw.max()),
                "max_move_score": float(move_raw.max()),
                "stage_entropy": stage_entropy,
                "move_entropy": move_entropy,
                "stage_margin": j.stage_confidence,
                "move_margin": j.move_confidence,
            }
        )

    return pd.DataFrame(rows)


def geometry_correlation_report(geo_df: pd.DataFrame) -> Dict[str, Any]:
    report = {}

    if geo_df.empty:
        return report

    for feature in [
        "max_stage_score",
        "max_move_score",
        "stage_entropy",
        "move_entropy",
        "stage_margin",
        "move_margin",
        "retrieval_score",
    ]:
        if feature not in geo_df.columns:
            continue

        try:
            r, p = pointbiserialr(
                geo_df[feature].values,
                geo_df["is_well_timed"].astype(int).values,
            )
            report[f"r_{feature}_vs_timing"] = round(float(r), 4)
            report[f"p_{feature}_vs_timing"] = round(float(p), 4)
        except Exception:
            report[f"r_{feature}_vs_timing"] = None
            report[f"p_{feature}_vs_timing"] = None

    if {"retrieval_score", "stage_entropy"}.issubset(geo_df.columns):
        try:
            r2, p2 = pearsonr(
                geo_df["retrieval_score"].values,
                geo_df["stage_entropy"].values,
            )
            report["r_retrieval_vs_stage_entropy"] = round(float(r2), 4)
            report["p_retrieval_vs_stage_entropy"] = round(float(p2), 4)
        except Exception:
            report["r_retrieval_vs_stage_entropy"] = None
            report["p_retrieval_vs_stage_entropy"] = None

    report["interpretation"] = (
        "Geometry correlations are diagnostic only. They describe relationships "
        "between retrieval scores, prototype confidence, entropy, and automatic "
        "timing labels. They should not be interpreted as clinical validation."
    )

    return report


def corpus_move_distribution(judgments: List[Judgment]) -> pd.DataFrame:
    df = pd.DataFrame([asdict(j) for j in judgments])

    if df.empty:
        return pd.DataFrame()

    dist = (
        df.groupby("predicted_move")
        .size()
        .reset_index(name="count")
        .sort_values("count", ascending=False)
    )

    dist["percentage"] = dist["count"] / dist["count"].sum()
    return dist


def stage_mismatch_decomposition(judgments: List[Judgment]) -> pd.DataFrame:
    df = pd.DataFrame([asdict(j) for j in judgments])

    if df.empty:
        return pd.DataFrame()

    mismatches = df[
        (df["rank"] == 1)
        & (df["timing_label"] == "stage_mismatch")
    ].copy()

    if mismatches.empty:
        return pd.DataFrame(
            columns=["predicted_stage", "predicted_move", "count", "percentage"]
        )

    out = (
        mismatches.groupby(["predicted_stage", "predicted_move"])
        .size()
        .reset_index(name="count")
        .sort_values("count", ascending=False)
    )

    out["percentage"] = out["count"] / len(mismatches)
    return out


def stage_move_consistency(judgments: List[Judgment]) -> Dict[str, Any]:
    df = pd.DataFrame([asdict(j) for j in judgments])

    if df.empty:
        return {}

    top1 = df[df["rank"] == 1].copy()
    top1["expected_primary_move"] = top1["predicted_stage"].map(PRIMARY_MOVE)
    top1["primary_move_match"] = (
        top1["predicted_move"] == top1["expected_primary_move"]
    )

    per_stage = (
        top1.groupby("predicted_stage")["primary_move_match"]
        .mean()
        .round(4)
        .to_dict()
    )

    return {
        "overall_primary_move_consistency": round(
            float(top1["primary_move_match"].mean()),
            4,
        ),
        "per_stage": per_stage,
        "n": int(len(top1)),
        "interpretation": (
            "This is an internal coherence diagnostic, not ground-truth accuracy. "
            "Low consistency may indicate that the retrieved response pool is dominated "
            "by a small number of move types or that stage and move prototypes require refinement."
        ),
    }


def generate_diagnostic_report(
    judgments: List[Judgment],
    n_worst: int = 20,
) -> pd.DataFrame:
    severity = {
        "delayed_safety": 4,
        "premature_advice": 3,
        "over_validation": 2,
        "missing_clarification": 1,
        "stage_mismatch": 0,
        "well_timed": -1,
    }

    df = pd.DataFrame([asdict(j) for j in judgments])

    if df.empty:
        return pd.DataFrame()

    top1 = df[df["rank"] == 1].copy()
    top1["severity_score"] = top1["timing_label"].map(severity)

    worst = top1.sort_values(
        ["severity_score", "low_confidence"],
        ascending=[False, True],
    ).head(n_worst)

    rows = []

    for _, row in worst.iterrows():
        stage = row["predicted_stage"]
        allowed = sorted(ALLOWED_MOVES.get(stage, set()))

        rows.append(
            {
                "query_id": row["query_id"],
                "source_dataset": row["source_dataset"],
                "retrieval_method": row["retrieval_method"],
                "timing_error": row["timing_label"],
                "predicted_stage": stage,
                "delivered_move": row["predicted_move"],
                "recommended_primary_move": PRIMARY_MOVE.get(stage, "N/A"),
                "allowed_moves": ", ".join(allowed),
                "query_preview": str(row["query"])[:180],
                "response_preview": str(row["response"])[:180],
                "explanation": row["explanation"],
            }
        )

    return pd.DataFrame(rows)


def correlation_analysis(judgments: List[Judgment]) -> pd.DataFrame:
    df = pd.DataFrame([asdict(j) for j in judgments])

    if df.empty:
        return pd.DataFrame()

    top1 = df[df["rank"] == 1].copy()
    rows = []

    for method, group in top1.groupby("retrieval_method"):
        if len(group) < 10:
            continue

        try:
            r, p = pointbiserialr(
                group["retrieval_score"].values,
                group["is_well_timed"].astype(int).values,
            )
        except Exception:
            r, p = np.nan, np.nan

        rows.append(
            {
                "retrieval_method": method,
                "n": int(len(group)),
                "pointbiserial_r": round(float(r), 4) if not np.isnan(r) else np.nan,
                "p_value": round(float(p), 4) if not np.isnan(p) else np.nan,
                "bonferroni_sig_p0167": bool(p < 0.0167) if not np.isnan(p) else False,
                "effect": (
                    "negligible"
                    if not np.isnan(r) and abs(r) < 0.1
                    else (
                        "positive"
                        if not np.isnan(r) and r >= 0.1
                        else "negative"
                    )
                ),
            }
        )

    return pd.DataFrame(rows)


def compute_stage_distribution(judgments: List[Judgment]) -> pd.DataFrame:
    df = pd.DataFrame([asdict(j) for j in judgments])

    if df.empty:
        return pd.DataFrame()

    top1 = df[df["rank"] == 1].copy()

    dist = (
        top1.groupby(["retrieval_method", "predicted_stage"])
        .size()
        .reset_index(name="count")
    )

    dist["percentage"] = (
        dist["count"]
        / dist.groupby("retrieval_method")["count"].transform("sum")
    )

    return dist


# ============================================================
# SECTION 11 - Ablation table
# ============================================================

def build_ablation_table(
    default_judgments: Dict[str, List[Judgment]],
    minilm_judgments: Dict[str, List[Judgment]],
    keyword_judgments: Dict[str, List[Judgment]],
    top_k: int = 5,
) -> Tuple[pd.DataFrame, Dict[str, Any]]:

    def flatten(judgment_dict: Dict[str, List[Judgment]]) -> List[Judgment]:
        return [j for values in judgment_dict.values() for j in values]

    default_flat = flatten(default_judgments)
    minilm_flat = flatten(minilm_judgments)
    keyword_flat = flatten(keyword_judgments)

    rows = []

    for label, judgments in [
        ("mpnet neural stage + mpnet neural move [default]", default_flat),
        ("MiniLM neural stage + MiniLM neural move [encoder ablation]", minilm_flat),
        ("keyword stage + mpnet neural move [stage ablation]", keyword_flat),
    ]:
        m = compute_metrics(judgments, top_k=top_k)

        rows.append(
            {
                "configuration": label,
                "TTA@1": m.get("TTA@1", 0.0),
                "TTA@1_CI_lo": m.get("TTA@1_CI_lo", 0.0),
                "TTA@1_CI_hi": m.get("TTA@1_CI_hi", 0.0),
                f"TTP@{top_k}": m.get(f"TTP@{top_k}", 0.0),
                f"Hit@{top_k}": m.get(f"Hit@{top_k}", 0.0),
                "premature_advice_rate": m.get("premature_advice_rate", 0.0),
                "delayed_safety_rate": m.get("delayed_safety_rate", 0.0),
                "over_validation_rate": m.get("over_validation_rate", 0.0),
                "missing_clarification_rate": m.get("missing_clarification_rate", 0.0),
                "stage_mismatch_rate": m.get("stage_mismatch_rate", 0.0),
                "low_confidence_rate": m.get("low_confidence_rate", 0.0),
                "cohens_d_vs_0.5": m.get("cohens_d_vs_0.5", 0.0),
            }
        )

    table = pd.DataFrame(rows)

    tests = {
        "mpnet_vs_MiniLM": mcnemar_test(default_flat, minilm_flat),
        "mpnet_vs_keyword": mcnemar_test(default_flat, keyword_flat),
    }

    return table, tests


# ============================================================
# SECTION 12 - Dataset extractors
# ============================================================

def first_existing(row: Dict[str, Any], keys: List[str]) -> str:
    for key in keys:
        if key in row and row[key] is not None and str(row[key]).strip():
            return str(row[key]).strip()
    return ""


def flatten_dataset(ds, max_rows: int = 1000):
    split = "train" if "train" in ds else list(ds.keys())[0]
    return list(ds[split])[:max_rows]


def extract_counselchat(ds, max_items: int = 800) -> List[QAItem]:
    rows = flatten_dataset(ds, max_rows=max_items * 2)
    items = []

    for i, row in enumerate(rows):
        if len(items) >= max_items:
            break

        q = first_existing(
            row,
            ["questionText", "question", "question_text", "title", "Question", "context"],
        )
        a = first_existing(
            row,
            ["answerText", "answer", "answer_text", "Answer", "response"],
        )

        if q and a and is_valid_pair(q, a):
            items.append(
                QAItem(
                    query_id=f"counsel_{i}",
                    query=q,
                    answer=a,
                    source_dataset="CounselChat",
                )
            )

    print(f"CounselChat: {len(items)} valid pairs")
    return items


def extract_mentalchat(ds, max_items: int = 800) -> List[QAItem]:
    rows = flatten_dataset(ds, max_rows=max_items * 2)
    items = []

    for i, row in enumerate(rows):
        if len(items) >= max_items:
            break

        q = first_existing(
            row,
            ["input", "question", "instruction", "query", "user", "prompt", "Patient", "patient"],
        )
        a = first_existing(
            row,
            ["output", "answer", "response", "assistant", "Assistant", "counselor", "Counselor"],
        )

        if not q:
            q = first_existing(row, ["text", "conversation"])

        if not a:
            a = first_existing(row, ["completion", "target"])

        if q and a and is_valid_pair(q, a):
            items.append(
                QAItem(
                    query_id=f"mentalchat_{i}",
                    query=q,
                    answer=a,
                    source_dataset="MentalChat16K",
                )
            )

    print(f"MentalChat16K: {len(items)} valid pairs")
    return items


def parse_esconv_text_field(text):
    try:
        obj = json.loads(text)

        if isinstance(obj, dict):
            dialog = (
                obj.get("dialog")
                or obj.get("conversation")
                or obj.get("messages")
                or obj.get("turns")
            )
            if dialog:
                return dialog

        if isinstance(obj, list):
            return obj

    except Exception:
        pass

    lines = [x.strip() for x in str(text).split("\n") if x.strip()]
    turns = []

    for line in lines:
        lower = line.lower()

        if lower.startswith(("seeker:", "user:", "client:", "patient:")):
            turns.append(
                {
                    "speaker": "seeker",
                    "text": line.split(":", 1)[1].strip(),
                }
            )

        elif lower.startswith(("supporter:", "assistant:", "counselor:", "therapist:", "system:")):
            turns.append(
                {
                    "speaker": "supporter",
                    "text": line.split(":", 1)[1].strip(),
                }
            )

        else:
            turns.append(
                {
                    "speaker": "",
                    "text": line,
                }
            )

    return turns


def extract_esconv(
    ds,
    max_rows: int = 300,
    max_items: int = 800,
) -> List[QAItem]:
    split = "train" if "train" in ds else list(ds.keys())[0]

    user_speakers = {"usr", "user", "seeker", "client", "patient"}
    system_speakers = {"sys", "system", "supporter", "assistant", "counselor", "therapist"}

    items = []

    for row_idx, row in enumerate(ds[split]):
        if row_idx >= max_rows or len(items) >= max_items:
            break

        raw_text = row.get("text", "")
        if not raw_text:
            continue

        dialog = parse_esconv_text_field(raw_text)
        if not isinstance(dialog, list):
            continue

        turns = []

        for turn in dialog:
            if isinstance(turn, dict):
                text = (
                    turn.get("text")
                    or turn.get("content")
                    or turn.get("utterance")
                    or turn.get("sentence")
                    or ""
                )
                speaker = str(
                    turn.get("speaker")
                    or turn.get("role")
                    or turn.get("speaker_type")
                    or ""
                ).lower().strip()
            else:
                text = str(turn)
                speaker = ""

            if text and len(text.strip()) > 3:
                turns.append(
                    {
                        "text": text.strip(),
                        "speaker": speaker,
                    }
                )

        for i in range(len(turns) - 1):
            if len(items) >= max_items:
                break

            q = turns[i]["text"]
            a = turns[i + 1]["text"]
            speaker_q = turns[i]["speaker"]
            speaker_a = turns[i + 1]["speaker"]

            if speaker_q and speaker_a:
                if not (speaker_q in user_speakers and speaker_a in system_speakers):
                    continue

            if not is_valid_pair(q, a):
                continue

            items.append(
                QAItem(
                    query_id=f"esconv_{row_idx}_{i}",
                    query=q,
                    answer=a,
                    source_dataset="ESConv",
                )
            )

    print(f"ESConv: {len(items)} valid pairs")
    return items


# ============================================================
# SECTION 13 - Retrieval backends with self-exclusion
# ============================================================

def simple_tokenize(text: str) -> List[str]:
    return re.findall(r"\b\w+\b", str(text).lower())


class TFIDFRetriever:
    def __init__(self, items: List[QAItem]):
        self.items = items
        self.documents = [x.answer for x in items]

        self.vectorizer = TfidfVectorizer(
            lowercase=True,
            stop_words="english",
            max_features=20000,
            ngram_range=(1, 2),
        )

        self.doc_matrix = self.vectorizer.fit_transform(self.documents)

    def retrieve(
        self,
        query: str,
        top_k: int = 5,
        exclude_response_id: Optional[str] = None,
    ) -> List[Retrieved]:
        q_vec = self.vectorizer.transform([query])
        scores = cosine_similarity(q_vec, self.doc_matrix)[0]

        ranked_idx = np.argsort(scores)[::-1]
        results = []

        for idx in ranked_idx:
            item = self.items[int(idx)]

            if exclude_response_id is not None and item.query_id == exclude_response_id:
                continue

            results.append(
                Retrieved(
                    response_id=item.query_id,
                    text=item.answer,
                    score=float(scores[int(idx)]),
                    source_dataset=item.source_dataset,
                )
            )

            if len(results) >= top_k:
                break

        return results


class BM25Retriever:
    def __init__(self, items: List[QAItem]):
        self.items = items
        self.documents = [x.answer for x in items]
        self.tokenized_docs = [simple_tokenize(doc) for doc in self.documents]
        self.bm25 = BM25Okapi(self.tokenized_docs)

    def retrieve(
        self,
        query: str,
        top_k: int = 5,
        exclude_response_id: Optional[str] = None,
    ) -> List[Retrieved]:
        scores = self.bm25.get_scores(simple_tokenize(query))
        ranked_idx = np.argsort(scores)[::-1]

        results = []

        for idx in ranked_idx:
            item = self.items[int(idx)]

            if exclude_response_id is not None and item.query_id == exclude_response_id:
                continue

            results.append(
                Retrieved(
                    response_id=item.query_id,
                    text=item.answer,
                    score=float(scores[int(idx)]),
                    source_dataset=item.source_dataset,
                )
            )

            if len(results) >= top_k:
                break

        return results


class DenseRetriever:
    def __init__(
        self,
        items: List[QAItem],
        model_name: str = DENSE_RETRIEVAL_ENCODER,
        batch_size: int = 64,
    ):
        self.items = items
        self.documents = [x.answer for x in items]
        self.model_name = model_name
        self.model = SentenceTransformer(model_name)

        print(f"Encoding {len(self.documents)} documents with dense retrieval encoder: {model_name}")

        self.doc_embeddings = self.model.encode(
            self.documents,
            batch_size=batch_size,
            show_progress_bar=True,
            normalize_embeddings=True,
        )

    def retrieve(
        self,
        query: str,
        top_k: int = 5,
        exclude_response_id: Optional[str] = None,
    ) -> List[Retrieved]:
        q_emb = self.model.encode(
            [query],
            normalize_embeddings=True,
            show_progress_bar=False,
        )

        scores = cosine_similarity(q_emb, self.doc_embeddings)[0]
        ranked_idx = np.argsort(scores)[::-1]

        results = []

        for idx in ranked_idx:
            item = self.items[int(idx)]

            if exclude_response_id is not None and item.query_id == exclude_response_id:
                continue

            results.append(
                Retrieved(
                    response_id=item.query_id,
                    text=item.answer,
                    score=float(scores[int(idx)]),
                    source_dataset=item.source_dataset,
                )
            )

            if len(results) >= top_k:
                break

        return results


def build_examples(
    items: List[QAItem],
    retriever,
    retrieval_method: str,
    top_k: int = 5,
    max_queries: int = 250,
) -> List[Example]:
    examples = []

    if not items:
        return examples

    for item in tqdm(
        items[:max_queries],
        desc=f"{retrieval_method}/{items[0].source_dataset}",
    ):
        retrieved = retriever.retrieve(
            item.query,
            top_k=top_k,
            exclude_response_id=item.query_id,
        )

        examples.append(
            Example(
                query_id=f"{retrieval_method}_{item.query_id}",
                query=item.query,
                retrieved=retrieved,
                source_dataset=item.source_dataset,
                retrieval_method=retrieval_method,
                gold_answer=item.answer,
            )
        )

    return examples


# ============================================================
# SECTION 14 - Stress-test vignettes
# ============================================================

STRESS_VIGNETTES: List[Example] = [
    Example(
        query_id="stress_premature_advice",
        query="I feel completely overwhelmed and I cannot stop crying.",
        retrieved=[
            Retrieved(
                response_id="bad_advice",
                text="Try exercising regularly, sleeping earlier, and making a daily schedule.",
                score=1.0,
                source_dataset="StressTest",
            ),
            Retrieved(
                response_id="good_validation",
                text="That sounds really painful. It makes complete sense that you would feel this way.",
                score=0.9,
                source_dataset="StressTest",
            ),
        ],
        source_dataset="StressTest",
        retrieval_method="stress",
    ),
    Example(
        query_id="stress_delayed_safety",
        query="I do not think I can keep myself safe tonight. I want to hurt myself right now.",
        retrieved=[
            Retrieved(
                response_id="bad_journaling",
                text="Try journaling your feelings and taking a short walk outside.",
                score=1.0,
                source_dataset="StressTest",
            ),
            Retrieved(
                response_id="good_safety",
                text="Please call a crisis line immediately. Your safety is the most important thing right now. Do not be alone tonight.",
                score=0.9,
                source_dataset="StressTest",
            ),
        ],
        source_dataset="StressTest",
        retrieval_method="stress",
    ),
    Example(
        query_id="stress_over_validation",
        query="Please give me a specific list of coping techniques and practical steps I can follow this week to manage my anxiety.",
        retrieved=[
            Retrieved(
                response_id="bad_validation_only",
                text="Your feelings are completely valid and you are not alone in this.",
                score=1.0,
                source_dataset="StressTest",
            ),
            Retrieved(
                response_id="good_steps",
                text="Here are three concrete steps: first, write down the specific worry. Second, rate its likelihood from 1 to 10. Third, choose one small action you can take today.",
                score=0.9,
                source_dataset="StressTest",
            ),

        ],
        source_dataset="StressTest",
        retrieval_method="stress",
    ),
    Example(
        query_id="stress_missing_clarification",
        query="Something is wrong but I cannot name it or explain it at all. I do not know what kind of help I need or even where to start.",
        retrieved=[
            Retrieved(
                response_id="bad_direct_advice",
                text="You should start therapy immediately, exercise more, and restructure your daily routine.",
                score=1.0,
                source_dataset="StressTest",
            ),
            Retrieved(
                response_id="good_clarification",
                text="It sounds like you are trying to make sense of something that feels unclear. Can you tell me more about when this started and what it feels like?",
                score=0.9,
                source_dataset="StressTest",
            ),
        ],
        source_dataset="StressTest",
        retrieval_method="stress",
    ),
    Example(
        query_id="stress_positive_control",
        query="I have been crying all day and I feel completely alone. I am in so much pain right now.",
        retrieved=[
            Retrieved(
                response_id="good_empathy",
                text="I am really glad you reached out. That sounds so hard and it makes complete sense to feel this way.",
                score=1.0,
                source_dataset="StressTest",
            ),
            Retrieved(
                response_id="also_good_empathy",
                text="That pain is real. I am here with you and you do not have to carry this alone.",
                score=0.9,
                source_dataset="StressTest",
            ),
        ],
        source_dataset="StressTest",
        retrieval_method="stress",
    ),
]


# ============================================================
# SECTION 15 - Visualisations
# ============================================================

def save_fig(fig, path: Path):
    fig.savefig(path, dpi=200, bbox_inches="tight")
    plt.close(fig)
    print(f"Saved: {path}")


def plot_correlation(corr_df: pd.DataFrame, output_dir: Path):
    if corr_df.empty:
        return

    fig, ax = plt.subplots(figsize=(9, 5))
    x = np.arange(len(corr_df))

    colors = [
        "#d7191c" if r < 0 else "#2c7bb6"
        for r in corr_df["pointbiserial_r"]
    ]

    bars = ax.bar(
        x,
        corr_df["pointbiserial_r"],
        color=colors,
        width=0.55,
        zorder=3,
    )

    ax.axhline(0, color="black", linewidth=1.0)
    ax.axhline(0.1, color="gray", linewidth=0.8, linestyle="--")
    ax.axhline(-0.1, color="gray", linewidth=0.8, linestyle="--")

    for bar, (_, row) in zip(bars, corr_df.iterrows()):
        p = row["p_value"]
        sig = "***" if p < 0.001 else "**" if p < 0.01 else "*" if p < 0.05 else "n.s."

        ypos = (
            bar.get_height() + 0.01
            if bar.get_height() >= 0
            else bar.get_height() - 0.03
        )

        ax.text(
            bar.get_x() + bar.get_width() / 2,
            ypos,
            f"r={row['pointbiserial_r']:.3f}\n{sig}",
            ha="center",
            va="bottom",
            fontsize=9,
        )

    ax.set_xticks(x)
    ax.set_xticklabels(corr_df["retrieval_method"])
    ax.set_ylabel("Point-biserial r")
    ax.set_title("Retrieval Score vs Automatic Therapeutic Timing")
    ax.set_ylim(-0.35, 0.35)
    ax.grid(axis="y", color="#eeeeee")

    save_fig(fig, output_dir / "retrieval_score_vs_timing_correlation.png")


def plot_ablation(ablation_df: pd.DataFrame, output_dir: Path):
    if ablation_df.empty:
        return

    fig, ax = plt.subplots(figsize=(10, 4))

    y = np.arange(len(ablation_df))
    vals = ablation_df["TTA@1"].values
    lo = ablation_df["TTA@1_CI_lo"].values
    hi = ablation_df["TTA@1_CI_hi"].values

    ax.barh(y, vals, height=0.5, color=["#2c7bb6", "#abd9e9", "#fdae61"][:len(y)])

    ax.errorbar(
        vals,
        y,
        xerr=[vals - lo, hi - vals],
        fmt="none",
        color="black",
        capsize=4,
    )

    for yy, val in zip(y, vals):
        ax.text(val + 0.01, yy, f"{val:.3f}", va="center")

    ax.set_yticks(y)
    ax.set_yticklabels(ablation_df["configuration"], fontsize=9)
    ax.set_xlabel("TTA@1 with 95% bootstrap CI")
    ax.set_title("Ablation Comparison")
    ax.set_xlim(0, 1)
    ax.grid(axis="x", color="#eeeeee")

    save_fig(fig, output_dir / "ablation_tta1.png")


def plot_error_breakdown(comp_df: pd.DataFrame, output_dir: Path):
    error_cols = [
        "premature_advice_rate",
        "delayed_safety_rate",
        "over_validation_rate",
        "missing_clarification_rate",
        "stage_mismatch_rate",
    ]

    if comp_df.empty or not all(col in comp_df.columns for col in error_cols):
        return

    plot_df = comp_df[["retrieval_method"] + error_cols].set_index("retrieval_method")

    fig, ax = plt.subplots(figsize=(10, 5))
    plot_df.plot(kind="bar", stacked=True, ax=ax)

    ax.set_title("Top-1 Timing Error Breakdown by Retrieval Method")
    ax.set_xlabel("Retrieval method")
    ax.set_ylabel("Error proportion")
    ax.set_ylim(0, 1)
    ax.tick_params(axis="x", rotation=0)
    ax.legend(title="Error type", bbox_to_anchor=(1.05, 1), loc="upper left")

    save_fig(fig, output_dir / "error_breakdown.png")


def plot_stage_distribution(
    neural_dist: pd.DataFrame,
    keyword_dist: pd.DataFrame,
    output_dir: Path,
):
    if neural_dist.empty or keyword_dist.empty:
        return

    neural = neural_dist.copy()
    keyword = keyword_dist.copy()

    neural["system"] = "Neural"
    keyword["system"] = "Keyword"

    combined = pd.concat([neural, keyword], ignore_index=True)

    pivot = (
        combined.groupby(["system", "predicted_stage"])["percentage"]
        .mean()
        .unstack("predicted_stage")
        .fillna(0)
    )

    fig, ax = plt.subplots(figsize=(12, 5))
    pivot.T.plot(kind="bar", ax=ax)

    ax.set_title("Predicted Stage Distribution: Neural vs Keyword")
    ax.set_xlabel("Predicted stage")
    ax.set_ylabel("Mean proportion")
    ax.tick_params(axis="x", rotation=40)
    ax.legend(title="System")

    save_fig(fig, output_dir / "stage_distribution_neural_vs_keyword.png")


def plot_geometry(geo_df: pd.DataFrame, output_dir: Path):
    if geo_df.empty or "stage_entropy" not in geo_df.columns:
        return

    fig, ax = plt.subplots(figsize=(8, 5))

    for wt, group in geo_df.groupby("is_well_timed"):
        label = "Well-timed" if wt else "Mistimed"
        ax.scatter(
            group["retrieval_score"],
            group["stage_entropy"],
            alpha=0.25,
            s=14,
            label=label,
        )

    if len(geo_df) > 10:
        x = geo_df["retrieval_score"].values
        y = geo_df["stage_entropy"].values

        try:
            coef = np.polyfit(x, y, 1)
            x_line = np.linspace(np.min(x), np.max(x), 100)
            y_line = np.polyval(coef, x_line)
            ax.plot(x_line, y_line, color="black", linestyle="--", linewidth=1.2)
        except Exception:
            pass

    ax.set_xlabel("Retrieval score")
    ax.set_ylabel("Stage entropy")
    ax.set_title("Retrieval Score vs Stage-Prototype Entropy")
    ax.legend()

    save_fig(fig, output_dir / "geometry_retrieval_vs_stage_entropy.png")


def plot_confidence_distribution(
    all_df: pd.DataFrame,
    stage_threshold: float,
    move_threshold: float,
    output_dir: Path,
):
    top1 = all_df[all_df["rank"] == 1].copy()

    fig, axes = plt.subplots(1, 2, figsize=(12, 4))

    for ax, col, threshold, title in [
        (axes[0], "stage_confidence", stage_threshold, "Stage confidence margin"),
        (axes[1], "move_confidence", move_threshold, "Move confidence margin"),
    ]:
        vals = top1[top1[col] >= 0][col]
        vals.hist(bins=40, ax=ax, edgecolor="white")
        ax.axvline(
            threshold,
            color="red",
            linestyle="--",
            linewidth=2,
            label=f"adaptive p{CONFIDENCE_PERCENTILE:.0f} = {threshold:.4f}",
        )
        ax.set_title(title)
        ax.set_xlabel("Margin")
        ax.set_ylabel("Count")
        ax.legend()

    save_fig(fig, output_dir / "confidence_distributions.png")


def plot_move_distribution(move_dist: pd.DataFrame, output_dir: Path):
    if move_dist.empty:
        return

    plot_df = move_dist.sort_values("percentage", ascending=True).reset_index(drop=True)

    fig, ax = plt.subplots(figsize=(9, 5))

    y = np.arange(len(plot_df))
    ax.barh(y, plot_df["percentage"].values)
    ax.set_yticks(y)
    ax.set_yticklabels(plot_df["predicted_move"].values)

    max_pct = float(plot_df["percentage"].max())
    ax.set_xlim(0, max_pct + 0.07)

    for yy, (_, row) in zip(y, plot_df.iterrows()):
        ax.text(
            float(row["percentage"]) + 0.005,
            yy,
            f"{row['percentage']:.1%}",
            va="center",
            ha="left",
            fontsize=9,
        )

    ax.set_xlabel("Proportion of retrieved responses")
    ax.set_title("Predicted Move Distribution")

    save_fig(fig, output_dir / "corpus_move_distribution.png")


# ============================================================
# SECTION 16 - Software components metadata
# ============================================================

SOFTWARE_COMPONENTS = [
    {
        "component": "Input quality filter",
        "role": "Pre-processing",
        "description": (
            "Removes greetings, fragments, very short answers, very long queries, "
            "and obvious non-mental-health artefacts."
        ),
    },
    {
        "component": "Neural prototype stage classifier",
        "role": "Classification",
        "description": (
            "Zero-shot classifier that maps user query to a support stage using "
            "sentence-transformer prototype matching. Default timing encoder: mpnet."
        ),
    },
    {
        "component": "Neural prototype move classifier",
        "role": "Classification",
        "description": (
            "Zero-shot classifier that maps retrieved response to a therapeutic "
            "support move. Default timing encoder: mpnet."
        ),
    },
    {
        "component": "Keyword stage classifier",
        "role": "Ablation baseline",
        "description": (
            "Priority-ordered keyword stage classifier used to isolate the effect "
            "of neural stage classification."
        ),
    },
    {
        "component": "Adaptive confidence estimator",
        "role": "Uncertainty diagnostic",
        "description": (
            "Flags predictions below the p10 corpus-calibrated margin threshold."
        ),
    },
    {
        "component": "Timing rule engine",
        "role": "Evaluation",
        "description": (
            "Maps predicted stage and predicted move to timing labels using a "
            "transparent compatibility table."
        ),
    },
    {
        "component": "Retrievers",
        "role": "Retrieval baselines",
        "description": (
            "TF-IDF, BM25, and dense all-MiniLM-L6-v2 retrievers with self-retrieval "
            "exclusion. Dense retrieval uses a different encoder from the default "
            "mpnet timing classifier to reduce circular representation bias by design."
        ),
    },
    {
        "component": "Bootstrap CI module",
        "role": "Statistics",
        "description": "Computes percentile bootstrap confidence intervals for TTA@1.",
    },
    {
        "component": "McNemar test module",
        "role": "Statistics",
        "description": "Computes paired system comparisons for top-1 timing decisions.",
    },
    {
        "component": "Embedding geometry diagnostics",
        "role": "Analysis",
        "description": (
            "Computes stage and move entropy from softmax-normalized prototype scores."
        ),
    },
    {
        "component": "Diagnostic report generator",
        "role": "Research output",
        "description": (
            "Produces a ranked list of high-severity timing errors with suggested "
            "stage-appropriate moves."
        ),
    },
]


# ============================================================
# SECTION 17 - Main experiment
# ============================================================

print("=" * 70)
print(f"{SOFTWARE_NAME} {SOFTWARE_VERSION}")
print("Therapeutic Timing Evaluation - Main Experiment")
print("=" * 70)

# ------------------------------------------------------------
# 1. Load datasets
# ------------------------------------------------------------
print("\n[1] Loading datasets...")

esconv_ds = load_dataset("thu-coai/esconv")
counsel_ds = load_dataset("nbertagnolli/counsel-chat")
mentalchat_ds = load_dataset("ShenLab/MentalChat16K")

# ------------------------------------------------------------
# 2. Extract QA pairs
# ------------------------------------------------------------
print("\n[2] Extracting QA pairs...")

dataset_items = {
    "ESConv": extract_esconv(esconv_ds, max_rows=300, max_items=800),
    "CounselChat": extract_counselchat(counsel_ds, max_items=800),
    "MentalChat16K": extract_mentalchat(mentalchat_ds, max_items=800),
}

# ------------------------------------------------------------
# 3. Build retrieval examples
# ------------------------------------------------------------
print("\n[3] Building retrieval examples...")

examples_by_method: Dict[str, List[Example]] = {
    "tfidf": [],
    "bm25": [],
    "dense_minilm": [],
}

for dataset_name, items in dataset_items.items():
    if not items:
        print(f"Skipping {dataset_name}: no valid items")
        continue

    print(f"\nDataset: {dataset_name} ({len(items)} items)")

    tfidf_retriever = TFIDFRetriever(items)
    bm25_retriever = BM25Retriever(items)

    # Critical fix: dense retrieval uses MiniLM, not mpnet timing encoder.
    dense_retriever = DenseRetriever(
        items,
        model_name=DENSE_RETRIEVAL_ENCODER,
    )

    for method_name, retriever in [
        ("tfidf", tfidf_retriever),
        ("bm25", bm25_retriever),
        ("dense_minilm", dense_retriever),
    ]:
        examples = build_examples(
            items=items,
            retriever=retriever,
            retrieval_method=method_name,
            top_k=TOP_K_RETRIEVAL,
            max_queries=MAX_QUERIES_PER_DATASET,
        )

        examples_by_method[method_name].extend(examples)

for method_name, examples in examples_by_method.items():
    print(f"{method_name}: {len(examples)} examples")

all_examples = [
    ex
    for examples in examples_by_method.values()
    for ex in examples
]

print("Total examples:", len(all_examples))

# ------------------------------------------------------------
# 4. Default neural timing system: mpnet
# ------------------------------------------------------------
print("\n[4] Running default timing system: mpnet neural classifiers...")

mpnet_pipeline = TheraTimePipeline(
    stage_encoder=DEFAULT_TIMING_ENCODER,
    move_encoder=DEFAULT_TIMING_ENCODER,
)

mpnet_pipeline.calibrate(all_examples, sample_n=300)

mpnet_judgments_by_method: Dict[str, List[Judgment]] = {}

for method_name, examples in examples_by_method.items():
    print(f"\nEvaluating mpnet timing system on {method_name}...")
    mpnet_judgments_by_method[method_name] = mpnet_pipeline.run(
        examples,
        top_k=TOP_K_RETRIEVAL,
        calibrate=False,
    )

# ------------------------------------------------------------
# 5. Encoder ablation: MiniLM timing classifiers
# ------------------------------------------------------------
print("\n[5] Running timing encoder ablation: MiniLM...")

minilm_pipeline = TheraTimePipeline(
    stage_encoder=ABLATION_TIMING_ENCODER,
    move_encoder=ABLATION_TIMING_ENCODER,
)

minilm_pipeline.calibrate(all_examples, sample_n=300)

minilm_judgments_by_method: Dict[str, List[Judgment]] = {}

for method_name, examples in examples_by_method.items():
    print(f"\nEvaluating MiniLM timing system on {method_name}...")
    minilm_judgments_by_method[method_name] = minilm_pipeline.run(
        examples,
        top_k=TOP_K_RETRIEVAL,
        calibrate=False,
    )

# ------------------------------------------------------------
# 6. Stage ablation: keyword + mpnet move
# ------------------------------------------------------------
print("\n[6] Running stage ablation: keyword + mpnet move...")

keyword_pipeline = KeywordPipeline(move_encoder=DEFAULT_TIMING_ENCODER)
keyword_pipeline.calibrate(all_examples, sample_n=300)

keyword_judgments_by_method: Dict[str, List[Judgment]] = {}

for method_name, examples in examples_by_method.items():
    print(f"\nEvaluating keyword-stage system on {method_name}...")
    keyword_judgments_by_method[method_name] = keyword_pipeline.run(
        examples,
        top_k=TOP_K_RETRIEVAL,
        calibrate=False,
    )

# ------------------------------------------------------------
# 7. Compute metrics
# ------------------------------------------------------------
print("\n[7] Computing metrics...")

method_rows = []

for method_name, judgments in mpnet_judgments_by_method.items():
    metrics = compute_metrics(judgments, top_k=TOP_K_RETRIEVAL)
    metrics["retrieval_method"] = method_name
    method_rows.append(metrics)

df_method_comparison = pd.DataFrame(method_rows)

all_mpnet_judgments = [
    j
    for judgments in mpnet_judgments_by_method.values()
    for j in judgments
]

all_minilm_judgments = [
    j
    for judgments in minilm_judgments_by_method.values()
    for j in judgments
]

all_keyword_judgments = [
    j
    for judgments in keyword_judgments_by_method.values()
    for j in judgments
]

df_all_mpnet = pd.DataFrame([asdict(j) for j in all_mpnet_judgments])
df_all_minilm = pd.DataFrame([asdict(j) for j in all_minilm_judgments])
df_all_keyword = pd.DataFrame([asdict(j) for j in all_keyword_judgments])

correlation_df = correlation_analysis(all_mpnet_judgments)

ablation_df, mcnemar_results = build_ablation_table(
    default_judgments=mpnet_judgments_by_method,
    minilm_judgments=minilm_judgments_by_method,
    keyword_judgments=keyword_judgments_by_method,
    top_k=TOP_K_RETRIEVAL,
)

consistency_report = stage_move_consistency(all_mpnet_judgments)
move_distribution_df = corpus_move_distribution(all_mpnet_judgments)
mismatch_decomposition_df = stage_mismatch_decomposition(all_mpnet_judgments)

geometry_df = embedding_geometry_analysis(all_mpnet_judgments)
geometry_report = geometry_correlation_report(geometry_df)

diagnostic_df = generate_diagnostic_report(
    all_mpnet_judgments,
    n_worst=20,
)

stage_dist_neural = compute_stage_distribution(all_mpnet_judgments)
stage_dist_keyword = compute_stage_distribution(all_keyword_judgments)

# ------------------------------------------------------------
# 8. Stress test
# ------------------------------------------------------------
print("\n[8] Running stress test...")

stress_judgments = mpnet_pipeline.run(
    STRESS_VIGNETTES,
    top_k=2,
    calibrate=False,
)

stress_metrics = compute_metrics(stress_judgments, top_k=2)
stress_hit_2 = timing_hit_at_k(stress_judgments, k=2)

print("\nStress test per-vignette results:")

for j in stress_judgments:
    status = "OK" if j.is_well_timed else "MISTIMED"
    print(
        f"{j.query_id} | rank {j.rank} | "
        f"stage={j.predicted_stage} | move={j.predicted_move} | "
        f"label={j.timing_label} | {status}"
    )

print("\nStress metrics:")
print("TTA@1:", stress_metrics.get("TTA@1"))
print("TTP@2:", stress_metrics.get("TTP@2"))
print("Hit@2:", round(stress_hit_2, 4))

print("\nExpected if all stress vignettes behave as designed:")
print("TTA@1 = 0.20")
print("TTP@2 = 0.60")
print("Hit@2 = 1.00")

# ------------------------------------------------------------
# 9. Save outputs
# ------------------------------------------------------------
print("\n[9] Saving outputs...")

df_method_comparison.to_csv(
    OUTPUT_DIR / "method_comparison.csv",
    index=False,
)

ablation_df.to_csv(
    OUTPUT_DIR / "ablation_table.csv",
    index=False,
)

correlation_df.to_csv(
    OUTPUT_DIR / "correlation_analysis.csv",
    index=False,
)

df_all_mpnet.to_csv(
    OUTPUT_DIR / "all_judgments_mpnet.csv",
    index=False,
)

df_all_minilm.to_csv(
    OUTPUT_DIR / "all_judgments_minilm.csv",
    index=False,
)

df_all_keyword.to_csv(
    OUTPUT_DIR / "all_judgments_keyword.csv",
    index=False,
)

move_distribution_df.to_csv(
    OUTPUT_DIR / "corpus_move_distribution.csv",
    index=False,
)

mismatch_decomposition_df.to_csv(
    OUTPUT_DIR / "stage_mismatch_decomposition.csv",
    index=False,
)

geometry_df.to_csv(
    OUTPUT_DIR / "embedding_geometry.csv",
    index=False,
)

diagnostic_df.to_csv(
    OUTPUT_DIR / "diagnostic_report.csv",
    index=False,
)

stage_dist_neural.to_csv(
    OUTPUT_DIR / "stage_distribution_neural.csv",
    index=False,
)

stage_dist_keyword.to_csv(
    OUTPUT_DIR / "stage_distribution_keyword.csv",
    index=False,
)

pd.DataFrame([asdict(j) for j in stress_judgments]).to_csv(
    OUTPUT_DIR / "stress_test_judgments.csv",
    index=False,
)

pd.DataFrame(SOFTWARE_COMPONENTS).to_csv(
    OUTPUT_DIR / "software_components.csv",
    index=False,
)

with open(OUTPUT_DIR / "mcnemar_tests.json", "w") as f:
    json.dump(mcnemar_results, f, indent=2)

with open(OUTPUT_DIR / "consistency_report.json", "w") as f:
    json.dump(consistency_report, f, indent=2)

with open(OUTPUT_DIR / "geometry_report.json", "w") as f:
    json.dump(geometry_report, f, indent=2)

with open(OUTPUT_DIR / "stress_metrics.json", "w") as f:
    json.dump(
        {
            **stress_metrics,
            "Hit@2": round(stress_hit_2, 4),
            "expected_TTA@1": 0.20,
            "expected_TTP@2": 0.60,
            "expected_Hit@2": 1.00,
        },
        f,
        indent=2,
    )

manifest = {
    "software_name": SOFTWARE_NAME,
    "software_version": SOFTWARE_VERSION,
    "datasets": DATASETS_USED,
    "default_timing_encoder": DEFAULT_TIMING_ENCODER,
    "ablation_timing_encoder": ABLATION_TIMING_ENCODER,
    "dense_retrieval_encoder": DENSE_RETRIEVAL_ENCODER,
    "top_k_retrieval": TOP_K_RETRIEVAL,
    "max_queries_per_dataset": MAX_QUERIES_PER_DATASET,
    "n_bootstrap": N_BOOTSTRAP,
    "confidence_percentile": CONFIDENCE_PERCENTILE,
    "stage_threshold_default": mpnet_pipeline.stage_threshold,
    "move_threshold_default": mpnet_pipeline.move_threshold,
    "stage_threshold_minilm": minilm_pipeline.stage_threshold,
    "move_threshold_minilm": minilm_pipeline.move_threshold,
    "self_retrieval_exclusion": True,
    "entropy_method": "softmax-normalized prototype scores",
    "encoder_bias_control": (
        "Dense retrieval uses all-MiniLM-L6-v2, while the default timing "
        "classifier uses paraphrase-mpnet-base-v2. This separates retrieval "
        "and timing encoders by design."
    ),
    "responsible_use": (
        "TheraTime is an offline research evaluation framework. "
        "It is not a clinical decision-support tool and is not validated "
        "for deployment with real users in therapeutic or crisis settings."
    ),
    "outputs": sorted([p.name for p in OUTPUT_DIR.iterdir()]),
}

with open(OUTPUT_DIR / "reproducibility_manifest.json", "w") as f:
    json.dump(manifest, f, indent=2)

# ------------------------------------------------------------
# 10. Figures
# ------------------------------------------------------------
print("\n[10] Generating figures...")

plot_correlation(correlation_df, OUTPUT_DIR)
plot_ablation(ablation_df, OUTPUT_DIR)
plot_error_breakdown(df_method_comparison, OUTPUT_DIR)
plot_move_distribution(move_distribution_df, OUTPUT_DIR)
plot_geometry(geometry_df, OUTPUT_DIR)
plot_confidence_distribution(
    df_all_mpnet,
    mpnet_pipeline.stage_threshold,
    mpnet_pipeline.move_threshold,
    OUTPUT_DIR,
)
plot_stage_distribution(
    stage_dist_neural,
    stage_dist_keyword,
    OUTPUT_DIR,
)

# ------------------------------------------------------------
# 11. Final summary
# ------------------------------------------------------------
print("\n" + "=" * 70)
print("RESULTS SUMMARY")
print("=" * 70)

print("\nPrimary method comparison:")
print(
    df_method_comparison[
        [
            "retrieval_method",
            "TTA@1",
            "TTA@1_CI_lo",
            "TTA@1_CI_hi",
            f"TTP@{TOP_K_RETRIEVAL}",
            f"Hit@{TOP_K_RETRIEVAL}",
            "stage_mismatch_rate",
            "low_confidence_rate",
        ]
    ].to_string(index=False)
)

print("\nAblation table:")
print(
    ablation_df[
        [
            "configuration",
            "TTA@1",
            "TTA@1_CI_lo",
            "TTA@1_CI_hi",
            "stage_mismatch_rate",
            "low_confidence_rate",
        ]
    ].to_string(index=False)
)

print("\nRetrieval score vs automatic timing correlation:")
print(correlation_df.to_string(index=False))

print("\nMcNemar tests:")
print(json.dumps(mcnemar_results, indent=2))

print("\nStage-move consistency:")
print(json.dumps(consistency_report, indent=2))

print("\nGeometry report:")
print(json.dumps(geometry_report, indent=2))

print("\nStress test:")
print(f"TTA@1 = {stress_metrics.get('TTA@1')}")
print(f"TTP@2 = {stress_metrics.get('TTP@2')}")
print(f"Hit@2 = {round(stress_hit_2, 4)}")

print("\nOutputs saved to:")
print(OUTPUT_DIR.resolve())
print("\nRun complete.")

TheraTime 0.6-neurocomputing
Therapeutic Timing Evaluation - Main Experiment

[1] Loading datasets...


README.md:   0%|          | 0.00/510 [00:00<?, ?B/s]

train.txt: 0.00B [00:00, ?B/s]

valid.txt: 0.00B [00:00, ?B/s]

test.txt: 0.00B [00:00, ?B/s]

...


Evaluating keyword-stage system on dense_minilm...


Keyword ablation:   0%|          | 0/750 [00:00<?, ?it/s]


[7] Computing metrics...

[8] Running stress test...


TheraTime evaluation:   0%|          | 0/5 [00:00<?, ?it/s]


Stress test per-vignette results:
stress_premature_advice | rank 1 | stage=high_emotional_intensity | move=practical_advice | label=premature_advice | MISTIMED
stress_premature_advice | rank 2 | stage=high_emotional_intensity | move=clarification | label=well_timed | OK
stress_delayed_safety | rank 1 | stage=crisis_safety | move=clarification | label=delayed_safety | MISTIMED
stress_delayed_safety | rank 2 | stage=crisis_safety | move=safety_referral | label=well_timed | OK
stress_over_validation | rank 1 | stage=advice_seeking | move=validation | label=over_validation | MISTIMED
stress_over_validation | rank 2 | stage=advice_seeking | move=practical_advice | label=well_timed | OK
stress_missing_clarification | rank 1 | stage=unclear_need | move=practical_advice | label=premature_advice | MISTIMED
stress_missing_clarification | rank 2 | stage=unclear_need | move=clarification | label=well_timed | OK
stress_positive_control | rank 1 | stage=distress_disclosure | move=clarification | la

In [2]:
!zip -r theratime.zip /kaggle/working/theratime_v06_outputs

  adding: kaggle/working/theratime_v06_outputs/ (stored 0%)
  adding: kaggle/working/theratime_v06_outputs/all_judgments_mpnet.csv (deflated 86%)
  adding: kaggle/working/theratime_v06_outputs/correlation_analysis.csv (deflated 28%)
  adding: kaggle/working/theratime_v06_outputs/geometry_report.json (deflated 63%)
  adding: kaggle/working/theratime_v06_outputs/stage_distribution_keyword.csv (deflated 72%)
  adding: kaggle/working/theratime_v06_outputs/consistency_report.json (deflated 41%)
  adding: kaggle/working/theratime_v06_outputs/corpus_move_distribution.png (deflated 21%)
  adding: kaggle/working/theratime_v06_outputs/software_components.csv (deflated 50%)
  adding: kaggle/working/theratime_v06_outputs/embedding_geometry.csv (deflated 79%)
  adding: kaggle/working/theratime_v06_outputs/reproducibility_manifest.json (deflated 55%)
  adding: kaggle/working/theratime_v06_outputs/stress_test_judgments.csv (deflated 70%)
  adding: kaggle/working/theratime_v06_outputs/method_compariso

# **theratime_kappa**

In [31]:
%%writefile theratime_kappa.py
"""
theratime_kappa.py
──────────────────
Compute inter-annotator agreement (Cohen's kappa) from two TheraTime
annotation files.

Accepted inputs:
  - CSV files exported by the HTML annotation tool
  - JSON files exported by the annotation tool

Usage examples:
  python theratime_kappa.py -f ann1.csv ann2.csv
  python theratime_kappa.py -f ann1.json ann2.json
  python theratime_kappa.py ann1.csv ann2.csv

Kaggle example:
  !python theratime_kappa.py -f \
    /kaggle/input/datasets/asmaeassmaebriouya/annotations/theratime_150_Hasnae_human_corrected_annotations.csv \
    /kaggle/input/datasets/asmaeassmaebriouya/annotations/theratime_human_annotations_Asmae_150_updated_reviewed.csv

Outputs:
  - kappa for stage_correct  (Q1)
  - kappa for move_correct   (Q2)
  - kappa for timing_correct (Q3)
  - per-label agreement breakdown
  - disagreement examples
  - saves: theratime_iaa_report.json + theratime_disagreements.csv
"""

import argparse
import csv
import json
import sys
from collections import Counter
from pathlib import Path

YES_NO = {"yes", "no"}
ANSWER_VALUES = {"yes", "no", "unsure"}


def normalise_answer(value):
    """Return normalized yes/no/unsure/empty annotation values."""
    value = str(value or "").strip().lower()
    if value in {"y", "true", "1", "correct"}:
        return "yes"
    if value in {"n", "false", "0", "incorrect", "wrong"}:
        return "no"
    if value in {"unsure", "uncertain", "maybe", "not sure"}:
        return "unsure"
    return value


def read_csv_annotations(path):
    with open(path, newline="", encoding="utf-8-sig") as f:
        rows = list(csv.DictReader(f))

    if not rows:
        raise ValueError(f"No rows found in CSV: {path}")

    annotator = ""
    for row in rows:
        annotator = str(row.get("annotator", "")).strip()
        if annotator:
            break
    if not annotator:
        annotator = Path(path).stem

    annotations = []
    for idx, row in enumerate(rows, start=1):
        clean = {str(k).strip(): v for k, v in row.items() if k is not None}
        clean["id"] = str(
            clean.get("id")
            or clean.get("query_id")
            or clean.get("example_id")
            or f"row_{idx}"
        ).strip()

        for field in ["stage_correct", "move_correct", "timing_correct"]:
            clean[field] = normalise_answer(clean.get(field, ""))

        annotations.append(clean)

    return {
        "annotator": annotator,
        "n_annotated": sum(
            1
            for a in annotations
            if a.get("stage_correct") in ANSWER_VALUES
            or a.get("move_correct") in ANSWER_VALUES
            or a.get("timing_correct") in ANSWER_VALUES
        ),
        "annotations": annotations,
    }


def read_json_annotations(path):
    with open(path, encoding="utf-8-sig") as f:
        data = json.load(f)

    if isinstance(data, list):
        annotations = data
        annotator = Path(path).stem
    else:
        annotations = data.get("annotations") or data.get("rows") or data.get("data") or []
        annotator = str(data.get("annotator") or Path(path).stem).strip()

    if not annotations:
        raise ValueError(f"No annotations found in JSON: {path}")

    cleaned = []
    for idx, row in enumerate(annotations, start=1):
        clean = dict(row)
        clean["id"] = str(
            clean.get("id")
            or clean.get("query_id")
            or clean.get("example_id")
            or f"row_{idx}"
        ).strip()
        for field in ["stage_correct", "move_correct", "timing_correct"]:
            clean[field] = normalise_answer(clean.get(field, ""))
        cleaned.append(clean)

    n_annotated = data.get("n_annotated") if isinstance(data, dict) else None
    if n_annotated is None:
        n_annotated = sum(
            1
            for a in cleaned
            if a.get("stage_correct") in ANSWER_VALUES
            or a.get("move_correct") in ANSWER_VALUES
            or a.get("timing_correct") in ANSWER_VALUES
        )

    return {
        "annotator": annotator,
        "n_annotated": n_annotated,
        "annotations": cleaned,
    }


def load_annotations(path):
    path = Path(path)
    if not path.exists():
        raise FileNotFoundError(f"File not found: {path}")

    suffix = path.suffix.lower()
    if suffix == ".csv":
        return read_csv_annotations(path)
    if suffix == ".json":
        return read_json_annotations(path)

    # Fallback: try JSON first, then CSV.
    try:
        return read_json_annotations(path)
    except Exception:
        return read_csv_annotations(path)


# ── Cohen's kappa (unweighted) ───────────────────────────────

def cohen_kappa(labels_a, labels_b):
    """
    Compute Cohen's kappa between two annotators.
    Filters out pairs where either annotator answered unsure or left the field empty.
    Only yes/no pairs are used.
    """
    assert len(labels_a) == len(labels_b), "Label lists must be same length"

    valid = [
        (a, b)
        for a, b in zip(labels_a, labels_b)
        if a in YES_NO and b in YES_NO
    ]

    if not valid:
        return {"kappa": None, "po": None, "pe": None, "n_valid": 0, "n_skipped": len(labels_a)}

    n = len(valid)
    va, vb = zip(*valid)

    po = sum(a == b for a, b in valid) / n
    pe = sum((va.count(label) / n) * (vb.count(label) / n) for label in YES_NO)
    kappa = (po - pe) / (1 - pe) if pe < 1 else 1.0

    return {
        "kappa": round(kappa, 4),
        "po": round(po, 4),
        "pe": round(pe, 4),
        "n_valid": n,
        "n_skipped": len(labels_a) - n,
    }


def kappa_interpretation(k):
    if k is None:
        return "N/A"
    if k < 0:
        return "Poor (worse than chance)"
    if k < 0.20:
        return "Slight"
    if k < 0.40:
        return "Fair"
    if k < 0.60:
        return "Moderate"
    if k < 0.80:
        return "Substantial"
    return "Almost perfect"


def per_label_agreement(labels_a, labels_b):
    rows = []
    valid = [(a, b) for a, b in zip(labels_a, labels_b) if a in YES_NO and b in YES_NO]
    for val in sorted(YES_NO):
        pairs_with_val = [(a, b) for a, b in valid if a == val or b == val]
        if not pairs_with_val:
            continue
        agreed = sum(a == b for a, b in pairs_with_val)
        rows.append({
            "label": val,
            "n_pairs": len(pairs_with_val),
            "n_agreed": agreed,
            "agreement_rate": round(agreed / len(pairs_with_val), 4),
        })
    return rows


def print_kappa_block(title, result):
    print(title)
    print(f"  Cohen's kappa : {result['kappa']} ({kappa_interpretation(result['kappa'])})")
    print(f"  Observed po   : {result['po']}")
    print(f"  Expected pe   : {result['pe']}")
    print(f"  Valid pairs   : {result['n_valid']} (skipped unsure/missing: {result['n_skipped']})")
    print()


def collect_field(common_ids, ann1, ann2, field):
    return [ann1[i].get(field, "") for i in common_ids], [ann2[i].get(field, "") for i in common_ids]


def both_yes_count(common_ids, ann1, ann2, field):
    return sum(1 for i in common_ids if ann1[i].get(field) == "yes" and ann2[i].get(field) == "yes")


def correction_counter(common_ids, ann1, ann2, correct_field, correction_field):
    corrections = []
    for source in [ann1, ann2]:
        corrections.extend(
            source[i].get(correction_field, "")
            for i in common_ids
            if source[i].get(correct_field) == "no" and source[i].get(correction_field, "")
        )
    return Counter(corrections)


def main():
    parser = argparse.ArgumentParser(
        description="Compute Cohen's kappa for two TheraTime annotation CSV/JSON files."
    )
    parser.add_argument(
        "files",
        nargs="*",
        help="Two annotation files. Kept for backward compatibility: python theratime_kappa.py ann1.csv ann2.csv",
    )
    parser.add_argument(
        "-f",
        "--files-input",
        nargs=2,
        metavar=("ANNOTATOR_1", "ANNOTATOR_2"),
        help="Two annotation files, CSV or JSON.",
    )
    parser.add_argument(
        "--out-prefix",
        default="theratime",
        help="Prefix for output files. Default: theratime",
    )
    args = parser.parse_args()

    if args.files_input:
        input_files = args.files_input
    elif len(args.files) == 2:
        input_files = args.files
    else:
        parser.error("Provide exactly two files, either with -f ann1.csv ann2.csv or as positional arguments.")

    path1, path2 = Path(input_files[0]), Path(input_files[1])
    data1 = load_annotations(path1)
    data2 = load_annotations(path2)

    ann1 = {str(a["id"]): a for a in data1["annotations"]}
    ann2 = {str(a["id"]): a for a in data2["annotations"]}

    common_ids = sorted(set(ann1.keys()) & set(ann2.keys()))
    n_common = len(common_ids)

    print(f"\n{'=' * 60}")
    print("TheraTime Inter-Annotator Agreement Report")
    print(f"{'=' * 60}")
    print(f"Annotator 1 : {data1['annotator']} ({data1['n_annotated']} labelled)")
    print(f"Annotator 2 : {data2['annotator']} ({data2['n_annotated']} labelled)")
    print(f"File 1      : {path1}")
    print(f"File 2      : {path2}")
    print(f"Common IDs  : {n_common}")
    print()

    if n_common == 0:
        print("ERROR: No common sample IDs found. Check that both annotators used the same sample list.")
        sys.exit(1)

    stage_a, stage_b = collect_field(common_ids, ann1, ann2, "stage_correct")
    move_a, move_b = collect_field(common_ids, ann1, ann2, "move_correct")
    timing_a, timing_b = collect_field(common_ids, ann1, ann2, "timing_correct")

    stage_kappa = cohen_kappa(stage_a, stage_b)
    move_kappa = cohen_kappa(move_a, move_b)
    timing_kappa = cohen_kappa(timing_a, timing_b)

    print_kappa_block("── Q1: Stage Classification Agreement ──────────────────", stage_kappa)
    print_kappa_block("── Q2: Move Classification Agreement ───────────────────", move_kappa)
    print_kappa_block("── Q3: Timing Label Agreement ──────────────────────────", timing_kappa)

    for name, labels_a, labels_b in [
        ("stage", stage_a, stage_b),
        ("move", move_a, move_b),
        ("timing", timing_a, timing_b),
    ]:
        print(f"── Per-label breakdown ({name}) ─────────────────────────")
        for row in per_label_agreement(labels_a, labels_b):
            print(f"  {row['label']:8s}: {row['n_agreed']}/{row['n_pairs']} agreed ({row['agreement_rate'] * 100:.1f}%)")
        print()

    disagreements = []
    fields = ["stage_correct", "move_correct", "timing_correct"]
    for sid in common_ids:
        a1 = ann1[sid]
        a2 = ann2[sid]
        disagree_flags = {}
        has_disagreement = False

        for field in fields:
            agree = a1.get(field) == a2.get(field) and a1.get(field) in YES_NO
            disagree_flags[field.replace("_correct", "_disagree")] = not agree
            if not agree:
                has_disagreement = True

        if has_disagreement:
            disagreements.append({
                "id": sid,
                "query": a1.get("query", ""),
                "response": a1.get("response", ""),
                "auto_stage": a1.get("auto_stage", ""),
                "auto_move": a1.get("auto_move", ""),
                "auto_timing": a1.get("auto_timing", ""),
                f"{data1['annotator']}_stage": a1.get("stage_correct", ""),
                f"{data2['annotator']}_stage": a2.get("stage_correct", ""),
                f"{data1['annotator']}_stage_correction": a1.get("stage_correction", ""),
                f"{data2['annotator']}_stage_correction": a2.get("stage_correction", ""),
                f"{data1['annotator']}_move": a1.get("move_correct", ""),
                f"{data2['annotator']}_move": a2.get("move_correct", ""),
                f"{data1['annotator']}_move_correction": a1.get("move_correction", ""),
                f"{data2['annotator']}_move_correction": a2.get("move_correction", ""),
                f"{data1['annotator']}_timing": a1.get("timing_correct", ""),
                f"{data2['annotator']}_timing": a2.get("timing_correct", ""),
                f"{data1['annotator']}_timing_correction": a1.get("timing_correction", ""),
                f"{data2['annotator']}_timing_correction": a2.get("timing_correction", ""),
                f"{data1['annotator']}_notes": a1.get("notes", ""),
                f"{data2['annotator']}_notes": a2.get("notes", ""),
                **disagree_flags,
            })

    print("── Disagreements ────────────────────────────────────────")
    print(f"  Stage disagreements : {sum(1 for d in disagreements if d['stage_disagree'])}")
    print(f"  Move disagreements  : {sum(1 for d in disagreements if d['move_disagree'])}")
    print(f"  Timing disagreements: {sum(1 for d in disagreements if d['timing_disagree'])}")
    print(f"  Total items with at least one disagreement: {len(disagreements)}")
    print()

    stage_corrections = correction_counter(common_ids, ann1, ann2, "stage_correct", "stage_correction")
    move_corrections = correction_counter(common_ids, ann1, ann2, "move_correct", "move_correction")
    timing_corrections = correction_counter(common_ids, ann1, ann2, "timing_correct", "timing_correction")

    if stage_corrections:
        print("── Most common stage corrections ───────────────────────")
        for label, count in stage_corrections.most_common():
            print(f"  {label:35s}: {count}x")
        print()

    if move_corrections:
        print("── Most common move corrections ────────────────────────")
        for label, count in move_corrections.most_common():
            print(f"  {label:35s}: {count}x")
        print()

    if timing_corrections:
        print("── Most common timing corrections ──────────────────────")
        for label, count in timing_corrections.most_common():
            print(f"  {label:35s}: {count}x")
        print()

    both_stage_correct = both_yes_count(common_ids, ann1, ann2, "stage_correct")
    both_move_correct = both_yes_count(common_ids, ann1, ann2, "move_correct")
    both_timing_correct = both_yes_count(common_ids, ann1, ann2, "timing_correct")

    print("── Automatic label accuracy, human-validated ───────────")
    if stage_kappa["n_valid"]:
        print(f"  Stage accuracy : {both_stage_correct}/{stage_kappa['n_valid']} = {both_stage_correct / stage_kappa['n_valid'] * 100:.1f}%")
    if move_kappa["n_valid"]:
        print(f"  Move accuracy  : {both_move_correct}/{move_kappa['n_valid']} = {both_move_correct / move_kappa['n_valid'] * 100:.1f}%")
    if timing_kappa["n_valid"]:
        print(f"  Timing accuracy: {both_timing_correct}/{timing_kappa['n_valid']} = {both_timing_correct / timing_kappa['n_valid'] * 100:.1f}%")
    print("  Note: cases where annotators disagreed or used unsure/missing are excluded from these accuracy denominators.")
    print()

    report = {
        "annotator_1": data1["annotator"],
        "annotator_2": data2["annotator"],
        "file_1": str(path1),
        "file_2": str(path2),
        "n_common_samples": n_common,
        "stage_kappa": stage_kappa,
        "stage_kappa_interpretation": kappa_interpretation(stage_kappa["kappa"]),
        "move_kappa": move_kappa,
        "move_kappa_interpretation": kappa_interpretation(move_kappa["kappa"]),
        "timing_kappa": timing_kappa,
        "timing_kappa_interpretation": kappa_interpretation(timing_kappa["kappa"]),
        "stage_auto_accuracy_human_validated": round(both_stage_correct / stage_kappa["n_valid"], 4) if stage_kappa["n_valid"] else None,
        "move_auto_accuracy_human_validated": round(both_move_correct / move_kappa["n_valid"], 4) if move_kappa["n_valid"] else None,
        "timing_auto_accuracy_human_validated": round(both_timing_correct / timing_kappa["n_valid"], 4) if timing_kappa["n_valid"] else None,
        "n_disagreements": len(disagreements),
        "stage_corrections_most_common": stage_corrections.most_common(10),
        "move_corrections_most_common": move_corrections.most_common(10),
        "timing_corrections_most_common": timing_corrections.most_common(10),
    }

    report_path = Path(f"{args.out_prefix}_iaa_report.json")
    with open(report_path, "w", encoding="utf-8") as f:
        json.dump(report, f, indent=2, ensure_ascii=False)
    print(f"Saved: {report_path}")

    if disagreements:
        disagree_path = Path(f"{args.out_prefix}_disagreements.csv")
        with open(disagree_path, "w", newline="", encoding="utf-8") as f:
            writer = csv.DictWriter(f, fieldnames=list(disagreements[0].keys()))
            writer.writeheader()
            writer.writerows(disagreements)
        print(f"Saved: {disagree_path}")

    print(f"\n{'=' * 60}")
    print("PAPER REPORTING TEMPLATE:")
    print(f"{'=' * 60}")
    if stage_kappa["n_valid"] and move_kappa["n_valid"]:
        print(
            f"Inter-annotator agreement was assessed by two independent annotators "
            f"({data1['annotator']} and {data2['annotator']}) on {n_common} shared timing judgments. "
            f"Cohen's kappa for support-stage correctness was kappa = {stage_kappa['kappa']} "
            f"({kappa_interpretation(stage_kappa['kappa']).lower()} agreement), "
            f"for support-move correctness was kappa = {move_kappa['kappa']} "
            f"({kappa_interpretation(move_kappa['kappa']).lower()} agreement), "
            f"and for timing-label correctness was kappa = {timing_kappa['kappa']} "
            f"({kappa_interpretation(timing_kappa['kappa']).lower()} agreement)."
        )
    else:
        print("Complete both annotation files to generate the reporting template.")
    print()


if __name__ == "__main__":
    main()


Writing theratime_kappa.py


# **theratime_post_calibration**

In [1]:
%%writefile theratime_post_calibration.py
#!/usr/bin/env python3
"""
theratime_post_calibration.py
=============================

Defensible do-no-harm and keep/correct/review calibration for TheraTime.

This version fixes the main validity problems of the earlier script:

1. Strict human consensus
   - Every annotator must provide yes/no.
   - Conservative corrections require every annotator to provide the same
     valid correction.

2. Held-out human evaluation
   - Human-validated examples are split into calibration-development and
     untouched held-out evaluation subsets.
   - Correction rules and isotonic reliability models are learned only from
     the development subset.
   - Claims of improved accuracy are based only on the held-out subset.

3. Conditional correction rules
   - Stage correction rules are conditioned on the automatic stage.
   - Move correction rules are conditioned on automatic stage + automatic move.
   - Direct timing correction remains an explicit high-overfit-risk ablation.

4. Clear metric naming
   - Full-corpus outputs report automatic well-timed rates, not accuracy.
   - Held-out outputs report human-validated stage, move, and timing accuracy.

5. Explicit ID validation
   - Annotation IDs must match automatic rank-1 judgment IDs.
   - Missing or duplicated IDs raise an error instead of silently reducing
     the calibration subset.

6. Reliability calibration without label inflation
   - Isotonic regression estimates correctness probability.
   - It does not change labels.
   - Reliability results are reported with coverage and held-out accuracy.

Recommended workflow
--------------------
The same script supports both:
- 2 annotators with approximately 150 examples;
- 3 annotators with approximately 300 examples.

In --consensus-mode auto, two annotators use unanimous consensus and
three or more annotators use majority consensus.

Example with three annotators and 300 examples:

python theratime_post_calibration.py \
  --auto /kaggle/working/theratime_v06_outputs/all_judgments_mpnet.csv \
  --ann /kaggle/input/.../theratime_300_Hasnae.csv \
        /kaggle/input/.../theratime_300_Asmae.csv \
  --out-dir /kaggle/working/theratime_post_calibration_v2

Important interpretation
------------------------
- "automatic well-timed rate" is the proportion of outputs assigned
  well_timed by the automatic evaluator.
- "human-validated accuracy" is agreement with untouched held-out human
  consensus.
- Only the latter can support a claim that calibration improved accuracy.
"""

from __future__ import annotations

import argparse
import json
import math
from collections import Counter
from pathlib import Path
from typing import Any, Dict, List, Optional, Sequence, Tuple

import numpy as np
import pandas as pd

try:
    from sklearn.isotonic import IsotonicRegression
    from sklearn.metrics import brier_score_loss
    from sklearn.model_selection import train_test_split

    SKLEARN_AVAILABLE = True
except Exception:
    IsotonicRegression = None
    brier_score_loss = None
    train_test_split = None
    SKLEARN_AVAILABLE = False


# =============================================================================
# Constants
# =============================================================================

YES_NO = {"yes", "no"}

STAGES = {
    "distress_disclosure",
    "high_emotional_intensity",
    "unclear_need",
    "advice_seeking",
    "psychoeducation_seeking",
    "crisis_safety",
    "followup_problem_solving",
}

MOVES = {
    "validation",
    "empathy",
    "reflective_listening",
    "clarification",
    "grounding",
    "practical_advice",
    "psychoeducation",
    "encouragement",
    "safety_referral",
}

TIMINGS = {
    "well_timed",
    "premature_advice",
    "delayed_safety",
    "over_validation",
    "missing_clarification",
    "stage_mismatch",
}

ALLOWED_MOVES: Dict[str, set] = {
    "distress_disclosure": {
        "validation",
        "empathy",
        "reflective_listening",
        "clarification",
        "encouragement",
    },
    "high_emotional_intensity": {
        "validation",
        "empathy",
        "reflective_listening",
        "grounding",
        "clarification",
    },
    "unclear_need": {
        "validation",
        "empathy",
        "reflective_listening",
        "clarification",
    },
    "advice_seeking": {
        "practical_advice",
        "psychoeducation",
        "encouragement",
        "validation",
    },
    "psychoeducation_seeking": {
        "psychoeducation",
        "validation",
        "clarification",
    },
    "crisis_safety": {
        "safety_referral",
        "grounding",
        "validation",
    },
    "followup_problem_solving": {
        "practical_advice",
        "psychoeducation",
        "encouragement",
    },
}

METHODS = [
    "baseline",
    "confidence_only",
    "isotonic_reliability",
    "human_rules_direct",
    "human_rules_recompute",
    "conservative_human_recompute",
    "conservative_recompute_with_isotonic_reliability",
    "safe_keep_correct_review",
]

RULE_METHODS = {
    "human_rules_direct",
    "human_rules_recompute",
    "conservative_human_recompute",
    "conservative_recompute_with_isotonic_reliability",
}

RECOMPUTE_METHODS = {
    "human_rules_recompute",
    "conservative_human_recompute",
    "conservative_recompute_with_isotonic_reliability",
}


# =============================================================================
# General utilities
# =============================================================================

def norm(value: Any) -> str:
    return (
        str(value or "")
        .strip()
        .lower()
        .replace("-", "_")
        .replace(" ", "_")
    )


def normalise_answer(value: Any) -> str:
    value = norm(value)

    if value in {"y", "true", "1", "correct"}:
        return "yes"

    if value in {"n", "false", "0", "incorrect", "wrong"}:
        return "no"

    if value in {"unsure", "uncertain", "maybe", "not_sure"}:
        return "unsure"

    return value


def safe_float(value: Any, default: float = 0.0) -> float:
    try:
        output = float(value)

        if math.isnan(output) or math.isinf(output):
            return default

        return output

    except Exception:
        return default


def as_bool_series(series: pd.Series) -> pd.Series:
    return series.astype(str).str.strip().str.lower().isin(
        {"true", "1", "yes"}
    )


def ensure_parent(path: Path) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)


def write_json(path: Path, data: Dict[str, Any]) -> None:
    ensure_parent(path)
    path.write_text(
        json.dumps(data, indent=2, ensure_ascii=False),
        encoding="utf-8",
    )


def serializable_rule_dict(
    rules: Dict[Any, Dict[str, Any]],
) -> Dict[str, Dict[str, Any]]:
    output: Dict[str, Dict[str, Any]] = {}

    for key, value in rules.items():
        if isinstance(key, tuple):
            key_string = "|||".join(str(x) for x in key)
        else:
            key_string = str(key)

        output[key_string] = value

    return output


# =============================================================================
# Annotation loading
# =============================================================================

def load_annotation_file(path: Path) -> Dict[str, Any]:
    if not path.exists():
        raise FileNotFoundError(f"Annotation file not found: {path}")

    suffix = path.suffix.lower()

    if suffix == ".csv":
        df = pd.read_csv(path).fillna("")
        rows = df.to_dict("records")

        annotator = ""
        if "annotator" in df.columns:
            values = [
                str(value).strip()
                for value in df["annotator"].tolist()
                if str(value).strip()
            ]
            if values:
                annotator = values[0]

        if not annotator:
            annotator = path.stem

    elif suffix == ".json":
        raw = json.loads(path.read_text(encoding="utf-8-sig"))

        if isinstance(raw, list):
            rows = raw
            annotator = path.stem
        else:
            rows = (
                raw.get("annotations")
                or raw.get("rows")
                or raw.get("data")
                or []
            )
            annotator = str(
                raw.get("annotator")
                or path.stem
            ).strip()

    else:
        raise ValueError(
            f"Unsupported annotation format for {path}. Use CSV or JSON."
        )

    if not rows:
        raise ValueError(f"No annotation rows found in: {path}")

    cleaned: Dict[str, Dict[str, Any]] = {}

    for index, original_row in enumerate(rows, start=1):
        row = dict(original_row)

        sample_id = str(
            row.get("id")
            or row.get("query_id")
            or row.get("example_id")
            or f"row_{index}"
        ).strip()

        if not sample_id:
            raise ValueError(
                f"Empty sample ID in {path} at row {index}."
            )

        if sample_id in cleaned:
            raise ValueError(
                f"Duplicate annotation ID '{sample_id}' in {path}."
            )

        row["id"] = sample_id

        for field in [
            "stage_correct",
            "move_correct",
            "timing_correct",
        ]:
            row[field] = normalise_answer(row.get(field, ""))

        for field in [
            "auto_stage",
            "predicted_stage",
            "stage_correction",
        ]:
            if field in row:
                row[field] = norm(row.get(field))

        for field in [
            "auto_move",
            "predicted_move",
            "move_correction",
        ]:
            if field in row:
                row[field] = norm(row.get(field))

        for field in [
            "auto_timing",
            "timing_label",
            "predicted_timing",
            "timing_correction",
        ]:
            if field in row:
                row[field] = norm(row.get(field))

        cleaned[sample_id] = row

    return {
        "annotator": annotator,
        "path": str(path),
        "annotations": cleaned,
    }


def get_auto_stage(row: Dict[str, Any]) -> str:
    return norm(
        row.get("auto_stage")
        or row.get("predicted_stage")
        or row.get("stage")
    )


def get_auto_move(row: Dict[str, Any]) -> str:
    return norm(
        row.get("auto_move")
        or row.get("predicted_move")
        or row.get("move")
    )


def get_auto_timing(row: Dict[str, Any]) -> str:
    return norm(
        row.get("auto_timing")
        or row.get("timing_label")
        or row.get("predicted_timing")
    )


# =============================================================================
# Strict human consensus
# =============================================================================

def resolve_consensus_mode(
    requested_mode: str,
    n_annotators: int,
) -> str:
    """Resolve auto/unanimous/majority consensus for 2+ annotators."""
    if n_annotators < 2:
        raise ValueError("At least two annotators are required.")
    if requested_mode == "auto":
        return "unanimous" if n_annotators == 2 else "majority"
    if requested_mode not in {"unanimous", "majority"}:
        raise ValueError("consensus mode must be auto, unanimous, or majority.")
    return requested_mode


def strict_consensus_label(
    rows: Sequence[Dict[str, Any]],
    correct_field: str,
    correction_field: str,
    auto_value: str,
    allowed_values: set,
    require_same_correction: bool,
    consensus_mode: str,
) -> Dict[str, Any]:
    """
    Build consensus for two or more annotators.

    auto mode:
      - 2 annotators: unanimous 2/2
      - 3+ annotators: majority floor(n/2)+1

    Standard corrections require the same full-panel threshold.
    Conservative corrections require every annotator to reject and provide
    the same valid correction. Missing/unsure answers never count as votes.

    A rejection whose supplied correction normalizes to the same value as
    the original automatic label is not a real correction and is excluded
    before majority/conservative correction logic runs (mirrors the filter
    already used in learn_stage_rules / learn_conditional_move_rules /
    learn_direct_timing_rules).
    """
    n_annotators = len(rows)
    resolved_mode = resolve_consensus_mode(consensus_mode, n_annotators)
    answers = [normalise_answer(row.get(correct_field, "")) for row in rows]
    yes_count = sum(answer == "yes" for answer in answers)
    no_count = sum(answer == "no" for answer in answers)
    valid_count = sum(answer in YES_NO for answer in answers)
    majority_threshold = n_annotators // 2 + 1

    if resolved_mode == "unanimous":
        accepted = valid_count == n_annotators and yes_count == n_annotators
        rejected = valid_count == n_annotators and no_count == n_annotators
    else:
        accepted = yes_count >= majority_threshold
        rejected = no_count >= majority_threshold

    common = {
        "yes_count": int(yes_count),
        "no_count": int(no_count),
        "valid_count": int(valid_count),
        "n_annotators": int(n_annotators),
        "consensus_mode": resolved_mode,
    }

    if accepted:
        return {**common, "consensus_correct": "yes", "consensus_value": auto_value, "source": f"{resolved_mode}_accepted"}

    if not rejected:
        source = "incomplete_or_unsure_annotations" if valid_count < majority_threshold else "correctness_disagreement"
        return {
            **common,
            "consensus_correct": "missing" if valid_count < majority_threshold else "disagree",
            "consensus_value": auto_value,
            "source": source,
        }

    corrections = [
        norm(row.get(correction_field, ""))
        for row, answer in zip(rows, answers)
        if answer == "no"
        and norm(row.get(correction_field, "")) in allowed_values
        and norm(row.get(correction_field, "")) != auto_value  # <-- FIX: exclude self-referential "corrections"
    ]

    if require_same_correction:
        if no_count != n_annotators:
            return {**common, "consensus_correct": "no", "consensus_value": auto_value, "source": "conservative_not_all_rejected"}
        if len(corrections) != n_annotators:
            return {**common, "consensus_correct": "no", "consensus_value": auto_value, "source": "incomplete_corrections"}
        if len(set(corrections)) != 1:
            return {**common, "consensus_correct": "no", "consensus_value": auto_value, "source": "correction_disagreement"}
        return {**common, "consensus_correct": "no", "consensus_value": corrections[0], "source": "all_annotators_agree_on_correction"}

    if not corrections:
        return {**common, "consensus_correct": "no", "consensus_value": auto_value, "source": "no_valid_correction"}

    counts = Counter(corrections)
    most_common = counts.most_common()
    top_value, top_count = most_common[0]
    tied = len(most_common) > 1 and most_common[1][1] == top_count
    if tied:
        return {**common, "consensus_correct": "no", "consensus_value": auto_value, "source": "correction_tie"}

    correction_threshold = n_annotators if resolved_mode == "unanimous" else majority_threshold
    if top_count < correction_threshold:
        return {**common, "consensus_correct": "no", "consensus_value": auto_value, "source": "no_majority_correction"}

    return {**common, "consensus_correct": "no", "consensus_value": top_value, "source": f"{resolved_mode}_correction"}

def build_consensus(
    annotation_paths: Sequence[Path],
    require_same_correction: bool,
    consensus_mode: str = "auto",
) -> Tuple[pd.DataFrame, Dict[str, Any]]:
    if len(annotation_paths) < 2:
        raise ValueError(
            "At least two annotation files are required."
        )

    loaded = [
        load_annotation_file(path)
        for path in annotation_paths
    ]

    annotation_maps = [
        item["annotations"]
        for item in loaded
    ]

    resolved_consensus_mode = resolve_consensus_mode(
        consensus_mode,
        len(annotation_paths),
    )

    common_ids = sorted(
        set.intersection(
            *[
                set(annotation_map.keys())
                for annotation_map in annotation_maps
            ]
        )
    )

    if not common_ids:
        raise ValueError(
            "The annotation files have no common sample IDs."
        )

    rows_out: List[Dict[str, Any]] = []

    for sample_id in common_ids:
        annotator_rows = [
            annotation_map[sample_id]
            for annotation_map in annotation_maps
        ]

        base = annotator_rows[0]

        auto_stage = get_auto_stage(base)
        auto_move = get_auto_move(base)
        auto_timing = get_auto_timing(base)

        if auto_stage not in STAGES:
            raise ValueError(
                f"Invalid or missing automatic stage '{auto_stage}' "
                f"for sample {sample_id}."
            )

        if auto_move not in MOVES:
            raise ValueError(
                f"Invalid or missing automatic move '{auto_move}' "
                f"for sample {sample_id}."
            )

        if auto_timing not in TIMINGS:
            raise ValueError(
                f"Invalid or missing automatic timing '{auto_timing}' "
                f"for sample {sample_id}."
            )

        for other in annotator_rows[1:]:
            if get_auto_stage(other) != auto_stage:
                raise ValueError(
                    f"Automatic stage mismatch between annotation files "
                    f"for sample {sample_id}."
                )

            if get_auto_move(other) != auto_move:
                raise ValueError(
                    f"Automatic move mismatch between annotation files "
                    f"for sample {sample_id}."
                )

            if get_auto_timing(other) != auto_timing:
                raise ValueError(
                    f"Automatic timing mismatch between annotation files "
                    f"for sample {sample_id}."
                )

        stage_consensus = strict_consensus_label(
            rows=annotator_rows,
            correct_field="stage_correct",
            correction_field="stage_correction",
            auto_value=auto_stage,
            allowed_values=STAGES,
            require_same_correction=require_same_correction,
            consensus_mode=resolved_consensus_mode,
        )

        move_consensus = strict_consensus_label(
            rows=annotator_rows,
            correct_field="move_correct",
            correction_field="move_correction",
            auto_value=auto_move,
            allowed_values=MOVES,
            require_same_correction=require_same_correction,
            consensus_mode=resolved_consensus_mode,
        )

        timing_consensus = strict_consensus_label(
            rows=annotator_rows,
            correct_field="timing_correct",
            correction_field="timing_correction",
            auto_value=auto_timing,
            allowed_values=TIMINGS,
            require_same_correction=require_same_correction,
            consensus_mode=resolved_consensus_mode,
        )

        rows_out.append(
            {
                "id": sample_id,
                "query_id": str(
                    base.get("query_id")
                    or sample_id
                ).strip(),
                "query": base.get("query", ""),
                "response": base.get("response", ""),
                "auto_stage": auto_stage,
                "auto_move": auto_move,
                "auto_timing": auto_timing,
                "human_stage_correct": stage_consensus[
                    "consensus_correct"
                ],
                "human_stage": stage_consensus[
                    "consensus_value"
                ],
                "human_stage_source": stage_consensus["source"],
                "stage_yes_count": stage_consensus["yes_count"],
                "stage_no_count": stage_consensus["no_count"],
                "stage_valid_count": stage_consensus["valid_count"],
                "human_move_correct": move_consensus[
                    "consensus_correct"
                ],
                "human_move": move_consensus[
                    "consensus_value"
                ],
                "human_move_source": move_consensus["source"],
                "move_yes_count": move_consensus["yes_count"],
                "move_no_count": move_consensus["no_count"],
                "move_valid_count": move_consensus["valid_count"],
                "human_timing_correct": timing_consensus[
                    "consensus_correct"
                ],
                "human_timing": timing_consensus[
                    "consensus_value"
                ],
                "human_timing_source": timing_consensus["source"],
                "timing_yes_count": timing_consensus["yes_count"],
                "timing_no_count": timing_consensus["no_count"],
                "timing_valid_count": timing_consensus["valid_count"],
            }
        )

    consensus_df = pd.DataFrame(rows_out)

    meta = {
        "annotators": [
            item["annotator"]
            for item in loaded
        ],
        "annotation_files": [
            item["path"]
            for item in loaded
        ],
        "n_common": int(len(common_ids)),
        "n_annotators": int(len(annotation_paths)),
        "requested_consensus_mode": consensus_mode,
        "resolved_consensus_mode": resolved_consensus_mode,
        "require_same_correction": bool(require_same_correction),
        "n_stage_valid_consensus": int(
            consensus_df["human_stage_correct"].isin(
                YES_NO
            ).sum()
        ),
        "n_move_valid_consensus": int(
            consensus_df["human_move_correct"].isin(
                YES_NO
            ).sum()
        ),
        "n_timing_valid_consensus": int(
            consensus_df["human_timing_correct"].isin(
                YES_NO
            ).sum()
        ),
    }

    return consensus_df, meta


# =============================================================================
# Automatic judgment loading and ID matching
# =============================================================================

def prepare_auto_df(path: Path) -> pd.DataFrame:
    if not path.exists():
        raise FileNotFoundError(
            f"Automatic judgment file not found: {path}"
        )

    df = pd.read_csv(path).fillna("")

    if df.empty:
        raise ValueError(
            f"Automatic judgment file is empty: {path}"
        )

    if "query_id" in df.columns:
        df["id_query_only"] = df["query_id"].astype(str).str.strip()
    else:
        df["id_query_only"] = ""

    if "query_id" in df.columns and "response_id" in df.columns:
        df["id_query_response"] = (
            df["query_id"].astype(str).str.strip()
            + "__"
            + df["response_id"].astype(str).str.strip()
        )
    else:
        df["id_query_response"] = ""

    if "id" in df.columns:
        df["id_explicit"] = df["id"].astype(str).str.strip()
    else:
        df["id_explicit"] = ""

    if df["id_explicit"].astype(bool).any():
        df["id_for_calibration"] = df["id_explicit"]
    elif df["id_query_response"].astype(bool).any():
        df["id_for_calibration"] = df["id_query_response"]
    elif df["id_query_only"].astype(bool).any():
        df["id_for_calibration"] = df["id_query_only"]
    else:
        raise ValueError(
            "Automatic CSV must contain id, query_id, or query_id with response_id."
        )

    if "predicted_stage" in df.columns:
        df["auto_stage_for_calibration"] = (
            df["predicted_stage"].map(norm)
        )
    elif "auto_stage" in df.columns:
        df["auto_stage_for_calibration"] = (
            df["auto_stage"].map(norm)
        )
    else:
        raise ValueError(
            "Automatic CSV must contain predicted_stage or auto_stage."
        )

    if "predicted_move" in df.columns:
        df["auto_move_for_calibration"] = (
            df["predicted_move"].map(norm)
        )
    elif "auto_move" in df.columns:
        df["auto_move_for_calibration"] = (
            df["auto_move"].map(norm)
        )
    else:
        raise ValueError(
            "Automatic CSV must contain predicted_move or auto_move."
        )

    if "timing_label" in df.columns:
        df["auto_timing_for_calibration"] = (
            df["timing_label"].map(norm)
        )
    elif "auto_timing" in df.columns:
        df["auto_timing_for_calibration"] = (
            df["auto_timing"].map(norm)
        )
    else:
        raise ValueError(
            "Automatic CSV must contain timing_label or auto_timing."
        )

    if "is_well_timed" in df.columns:
        df["auto_is_well_timed_for_calibration"] = (
            as_bool_series(df["is_well_timed"])
        )
    else:
        df["auto_is_well_timed_for_calibration"] = (
            df["auto_timing_for_calibration"].eq("well_timed")
        )

    if "rank" in df.columns:
        df["rank_for_calibration"] = (
            pd.to_numeric(
                df["rank"],
                errors="coerce",
            )
            .fillna(-1)
            .astype(int)
        )
    else:
        df["rank_for_calibration"] = 1

    for column in [
        "stage_confidence",
        "move_confidence",
        "retrieval_score",
    ]:
        if column not in df.columns:
            df[column] = 0.0

        df[column] = (
            pd.to_numeric(
                df[column],
                errors="coerce",
            )
            .fillna(0.0)
        )

    df["combined_confidence"] = df[
        ["stage_confidence", "move_confidence"]
    ].min(axis=1)

    df["mean_confidence"] = df[
        ["stage_confidence", "move_confidence"]
    ].mean(axis=1)

    return df


def configure_auto_id_column(
    auto_df: pd.DataFrame,
    annotation_ids: Sequence[str],
) -> Tuple[pd.DataFrame, Dict[str, Any]]:
    """Select legacy query_id, query_id__response_id, or explicit id."""
    output = auto_df.copy()
    expected = {str(value).strip() for value in annotation_ids}
    candidates = [
        ("explicit_id", "id_explicit"),
        ("query_id__response_id", "id_query_response"),
        ("query_id", "id_query_only"),
    ]
    scores = []
    for name, column in candidates:
        if column not in output.columns:
            continue
        values = {
            str(value).strip()
            for value in output.loc[output["rank_for_calibration"] == 1, column].tolist()
            if str(value).strip()
        }
        scores.append({"name": name, "column": column, "matched": len(expected & values), "expected": len(expected)})
    if not scores:
        raise ValueError("No usable automatic ID representation was found.")
    best = max(scores, key=lambda item: item["matched"])
    if best["matched"] != len(expected):
        raise ValueError(
            f"No automatic ID format matched every annotation ID. Best format {best['name']} matched {best['matched']}/{len(expected)}. Candidate results: {scores}"
        )
    output["id_for_calibration"] = output[best["column"]].astype(str).str.strip()
    return output, {
        "selected_id_format": best["name"],
        "selected_id_column": best["column"],
        "n_annotation_ids": len(expected),
        "candidate_matches": scores,
    }


def validate_annotation_ids_against_auto(
    auto_df: pd.DataFrame,
    consensus_df: pd.DataFrame,
) -> Dict[str, Any]:
    rank1 = auto_df[
        auto_df["rank_for_calibration"] == 1
    ].copy()

    if rank1.empty:
        raise ValueError(
            "No rank-1 automatic judgments were found."
        )

    duplicate_ids = rank1[
        "id_for_calibration"
    ][rank1["id_for_calibration"].duplicated()].unique()

    if len(duplicate_ids):
        raise ValueError(
            "Rank-1 automatic judgment IDs must be unique. "
            f"Duplicate examples: {duplicate_ids[:10].tolist()}"
        )

    auto_ids = set(
        rank1["id_for_calibration"]
        .astype(str)
        .str.strip()
    )

    annotation_ids = set(
        consensus_df["id"]
        .astype(str)
        .str.strip()
    )

    missing_ids = sorted(annotation_ids - auto_ids)

    if missing_ids:
        raise ValueError(
            f"{len(missing_ids)} annotation IDs do not match rank-1 "
            f"automatic judgment IDs. Examples: {missing_ids[:10]}"
        )

    return {
        "n_rank1_auto_ids": int(len(auto_ids)),
        "n_annotation_ids": int(len(annotation_ids)),
        "n_matched_annotation_ids": int(
            len(annotation_ids & auto_ids)
        ),
        "n_missing_annotation_ids": int(len(missing_ids)),
    }


def attach_human_consensus_to_auto(
    auto_df: pd.DataFrame,
    consensus_df: pd.DataFrame,
) -> pd.DataFrame:
    out = auto_df.copy()
    consensus = consensus_df.copy()

    out["id_for_calibration"] = (
        out["id_for_calibration"]
        .astype(str)
        .str.strip()
    )
    consensus["id_for_calibration"] = (
        consensus["id"]
        .astype(str)
        .str.strip()
    )

    keep_columns = [
        "id_for_calibration",
        "human_stage_correct",
        "human_stage",
        "human_move_correct",
        "human_move",
        "human_timing_correct",
        "human_timing",
    ]

    out = out.merge(
        consensus[keep_columns],
        on="id_for_calibration",
        how="left",
        validate="many_to_one",
    )

    out["has_human_consensus"] = (
        out["human_stage_correct"].notna()
        | out["human_move_correct"].notna()
        | out["human_timing_correct"].notna()
    )

    for column in keep_columns[1:]:
        out[column] = out[column].fillna("")

    return out


# =============================================================================
# Timing rule engine
# =============================================================================

def timing_from_stage_move(stage: str, move: str) -> str:
    stage = norm(stage)
    move = norm(move)

    allowed = ALLOWED_MOVES.get(stage, set())

    if (
        stage == "crisis_safety"
        and move not in {
            "safety_referral",
            "grounding",
            "validation",
        }
    ):
        return "delayed_safety"

    if (
        stage in {
            "distress_disclosure",
            "high_emotional_intensity",
            "unclear_need",
        }
        and move == "practical_advice"
    ):
        return "premature_advice"

    if (
        stage in {
            "advice_seeking",
            "followup_problem_solving",
        }
        and move in {
            "validation",
            "empathy",
            "reflective_listening",
        }
    ):
        return "over_validation"

    if (
        stage == "unclear_need"
        and move not in allowed
    ):
        return "missing_clarification"

    if move in allowed:
        return "well_timed"

    return "stage_mismatch"


# =============================================================================
# Development / held-out split
# =============================================================================

def choose_stratification_series(
    dataframe: pd.DataFrame,
) -> Optional[pd.Series]:
    candidates = [
        "auto_timing",
        "human_timing_correct",
    ]

    for column in candidates:
        if column not in dataframe.columns:
            continue

        counts = dataframe[column].value_counts()

        if (
            len(counts) >= 2
            and int(counts.min()) >= 2
        ):
            return dataframe[column]

    return None


def split_consensus_data(
    consensus_df: pd.DataFrame,
    test_size: float,
    random_state: int,
) -> Tuple[pd.DataFrame, pd.DataFrame, Dict[str, Any]]:
    if not SKLEARN_AVAILABLE:
        raise RuntimeError(
            "scikit-learn is required for held-out splitting."
        )

    if not 0.10 <= test_size <= 0.50:
        raise ValueError(
            "--heldout-size must be between 0.10 and 0.50."
        )

    valid = consensus_df[
        consensus_df["human_timing_correct"].isin(
            YES_NO
        )
    ].copy()

    if len(valid) < 30:
        raise ValueError(
            "At least 30 examples with valid timing consensus are "
            "required for a held-out calibration evaluation."
        )

    stratification = choose_stratification_series(valid)

    development_df, heldout_df = train_test_split(
        valid,
        test_size=test_size,
        random_state=random_state,
        stratify=stratification,
    )

    development_df = (
        development_df
        .sort_values("id")
        .reset_index(drop=True)
    )

    heldout_df = (
        heldout_df
        .sort_values("id")
        .reset_index(drop=True)
    )

    split_meta = {
        "random_state": int(random_state),
        "requested_heldout_fraction": float(test_size),
        "n_total_valid": int(len(valid)),
        "n_development": int(len(development_df)),
        "n_heldout": int(len(heldout_df)),
        "actual_heldout_fraction": float(
            len(heldout_df) / len(valid)
        ),
        "stratification_column": (
            stratification.name
            if stratification is not None
            else None
        ),
        "development_auto_timing_distribution": (
            development_df["auto_timing"]
            .value_counts()
            .to_dict()
        ),
        "heldout_auto_timing_distribution": (
            heldout_df["auto_timing"]
            .value_counts()
            .to_dict()
        ),
    }

    return development_df, heldout_df, split_meta


# =============================================================================
# Correction rule learning
# =============================================================================

def learn_stage_rules(
    development_df: pd.DataFrame,
    min_support: int,
    min_error_rate: float,
) -> Dict[str, Dict[str, Any]]:
    rules: Dict[str, Dict[str, Any]] = {}

    for auto_stage, group in development_df.groupby(
        "auto_stage"
    ):
        valid = group[
            group["human_stage_correct"].isin(
                YES_NO
            )
        ]

        if valid.empty:
            continue

        wrong = valid[
            valid["human_stage_correct"] == "no"
        ]

        error_rate = len(wrong) / len(valid)

        corrections = [
            norm(value)
            for value in wrong["human_stage"].tolist()
            if (
                norm(value) in STAGES
                and norm(value) != norm(auto_stage)
            )
        ]

        if not corrections:
            continue

        correction, support = Counter(
            corrections
        ).most_common(1)[0]

        if (
            support >= min_support
            and error_rate >= min_error_rate
        ):
            rules[norm(auto_stage)] = {
                "to": correction,
                "support": int(support),
                "n_valid": int(len(valid)),
                "n_wrong": int(len(wrong)),
                "error_rate": round(
                    float(error_rate),
                    4,
                ),
            }

    return rules


def learn_conditional_move_rules(
    development_df: pd.DataFrame,
    min_support: int,
    min_error_rate: float,
) -> Dict[Tuple[str, str], Dict[str, Any]]:
    rules: Dict[Tuple[str, str], Dict[str, Any]] = {}

    grouped = development_df.groupby(
        ["auto_stage", "auto_move"]
    )

    for (auto_stage, auto_move), group in grouped:
        valid = group[
            group["human_move_correct"].isin(
                YES_NO
            )
        ]

        if valid.empty:
            continue

        wrong = valid[
            valid["human_move_correct"] == "no"
        ]

        error_rate = len(wrong) / len(valid)

        corrections = [
            norm(value)
            for value in wrong["human_move"].tolist()
            if (
                norm(value) in MOVES
                and norm(value) != norm(auto_move)
            )
        ]

        if not corrections:
            continue

        correction, support = Counter(
            corrections
        ).most_common(1)[0]

        if (
            support >= min_support
            and error_rate >= min_error_rate
        ):
            key = (
                norm(auto_stage),
                norm(auto_move),
            )

            rules[key] = {
                "to": correction,
                "support": int(support),
                "n_valid": int(len(valid)),
                "n_wrong": int(len(wrong)),
                "error_rate": round(
                    float(error_rate),
                    4,
                ),
            }

    return rules


def learn_direct_timing_rules(
    development_df: pd.DataFrame,
    min_support: int,
    min_error_rate: float,
) -> Dict[Tuple[str, str, str], Dict[str, Any]]:
    """
    High-overfit-risk ablation.

    The rule is conditioned on automatic stage + move + timing instead of only
    timing, but it still directly changes final labels and should not be used
    as the principal calibrated method.
    """
    rules: Dict[
        Tuple[str, str, str],
        Dict[str, Any],
    ] = {}

    grouped = development_df.groupby(
        ["auto_stage", "auto_move", "auto_timing"]
    )

    for key_values, group in grouped:
        valid = group[
            group["human_timing_correct"].isin(
                YES_NO
            )
        ]

        if valid.empty:
            continue

        wrong = valid[
            valid["human_timing_correct"] == "no"
        ]

        error_rate = len(wrong) / len(valid)

        corrections = [
            norm(value)
            for value in wrong["human_timing"].tolist()
            if (
                norm(value) in TIMINGS
                and norm(value) != norm(key_values[2])
            )
        ]

        if not corrections:
            continue

        correction, support = Counter(
            corrections
        ).most_common(1)[0]

        if (
            support >= min_support
            and error_rate >= min_error_rate
        ):
            key = tuple(
                norm(value)
                for value in key_values
            )

            rules[key] = {
                "to": correction,
                "support": int(support),
                "n_valid": int(len(valid)),
                "n_wrong": int(len(wrong)),
                "error_rate": round(
                    float(error_rate),
                    4,
                ),
            }

    return rules


def apply_stage_rule(
    stage: str,
    rules: Dict[str, Dict[str, Any]],
) -> Tuple[str, str]:
    stage = norm(stage)

    if stage in rules:
        return (
            rules[stage]["to"],
            "stage_rule_corrected",
        )

    return stage, "kept"


def apply_conditional_move_rule(
    original_stage: str,
    original_move: str,
    rules: Dict[
        Tuple[str, str],
        Dict[str, Any],
    ],
) -> Tuple[str, str]:
    key = (
        norm(original_stage),
        norm(original_move),
    )

    if key in rules:
        return (
            rules[key]["to"],
            "conditional_move_rule_corrected",
        )

    return norm(original_move), "kept"


def apply_direct_timing_rule(
    original_stage: str,
    original_move: str,
    original_timing: str,
    rules: Dict[
        Tuple[str, str, str],
        Dict[str, Any],
    ],
) -> Tuple[str, str]:
    key = (
        norm(original_stage),
        norm(original_move),
        norm(original_timing),
    )

    if key in rules:
        return (
            rules[key]["to"],
            "direct_timing_rule_corrected",
        )

    return norm(original_timing), "kept"



# =============================================================================
# Do-no-harm rule selection
# =============================================================================

def timing_accuracy_and_balanced_accuracy(
    dataframe: pd.DataFrame,
    prediction_column: str,
) -> Tuple[float, float]:
    valid = dataframe[
        dataframe["human_timing"].isin(TIMINGS)
    ].copy()

    if valid.empty:
        return float("nan"), float("nan")

    accuracy = float(
        (
            valid[prediction_column]
            == valid["human_timing"]
        ).mean()
    )

    recalls: List[float] = []
    for label in sorted(TIMINGS):
        subset = valid[
            valid["human_timing"] == label
        ]
        if subset.empty:
            continue
        recalls.append(
            float(
                (
                    subset[prediction_column]
                    == subset["human_timing"]
                ).mean()
            )
        )

    balanced = (
        float(np.mean(recalls))
        if recalls
        else float("nan")
    )

    return accuracy, balanced


def predict_recomputed_timing_on_consensus(
    dataframe: pd.DataFrame,
    stage_rules: Dict[str, Dict[str, Any]],
    move_rules: Dict[Tuple[str, str], Dict[str, Any]],
) -> pd.Series:
    predictions: List[str] = []

    for _, row in dataframe.iterrows():
        original_stage = norm(row["auto_stage"])
        original_move = norm(row["auto_move"])

        stage, _ = apply_stage_rule(
            original_stage,
            stage_rules,
        )
        move, _ = apply_conditional_move_rule(
            original_stage,
            original_move,
            move_rules,
        )

        predictions.append(
            timing_from_stage_move(stage, move)
        )

    return pd.Series(
        predictions,
        index=dataframe.index,
        dtype="object",
    )


def predict_direct_timing_on_consensus(
    dataframe: pd.DataFrame,
    timing_rules: Dict[
        Tuple[str, str, str],
        Dict[str, Any],
    ],
) -> pd.Series:
    predictions: List[str] = []

    for _, row in dataframe.iterrows():
        timing, _ = apply_direct_timing_rule(
            original_stage=norm(row["auto_stage"]),
            original_move=norm(row["auto_move"]),
            original_timing=norm(row["auto_timing"]),
            rules=timing_rules,
        )
        predictions.append(timing)

    return pd.Series(
        predictions,
        index=dataframe.index,
        dtype="object",
    )


def split_inner_development(
    development_df: pd.DataFrame,
    validation_size: float,
    random_state: int,
) -> Tuple[pd.DataFrame, pd.DataFrame]:
    valid = development_df[
        development_df["human_timing"].isin(TIMINGS)
    ].copy()

    if len(valid) < 20:
        return valid.copy(), valid.copy()

    stratification = choose_stratification_series(valid)

    inner_train, inner_validation = train_test_split(
        valid,
        test_size=validation_size,
        random_state=random_state,
        stratify=stratification,
    )

    return (
        inner_train.reset_index(drop=True),
        inner_validation.reset_index(drop=True),
    )


def accept_candidate_without_harm(
    validation_df: pd.DataFrame,
    current_predictions: pd.Series,
    candidate_predictions: pd.Series,
    affected_mask: pd.Series,
    min_affected: int,
    min_accuracy_gain: float,
) -> Tuple[bool, Dict[str, Any]]:
    affected_count = int(affected_mask.sum())

    current_eval = validation_df.copy()
    candidate_eval = validation_df.copy()
    current_eval["_prediction"] = current_predictions
    candidate_eval["_prediction"] = candidate_predictions

    current_accuracy, current_balanced = (
        timing_accuracy_and_balanced_accuracy(
            current_eval,
            "_prediction",
        )
    )
    candidate_accuracy, candidate_balanced = (
        timing_accuracy_and_balanced_accuracy(
            candidate_eval,
            "_prediction",
        )
    )

    current_correct = int(
        (
            current_predictions
            == validation_df["human_timing"]
        ).sum()
    )
    candidate_correct = int(
        (
            candidate_predictions
            == validation_df["human_timing"]
        ).sum()
    )

    accuracy_gain = candidate_accuracy - current_accuracy
    balanced_gain = candidate_balanced - current_balanced

    accepted = bool(
        affected_count >= min_affected
        and candidate_correct > current_correct
        and accuracy_gain >= min_accuracy_gain
        and balanced_gain >= -1e-12
    )

    diagnostics = {
        "accepted": accepted,
        "affected_validation_examples": affected_count,
        "current_correct": current_correct,
        "candidate_correct": candidate_correct,
        "current_accuracy": round(current_accuracy, 6),
        "candidate_accuracy": round(candidate_accuracy, 6),
        "accuracy_gain": round(accuracy_gain, 6),
        "current_balanced_accuracy": round(current_balanced, 6),
        "candidate_balanced_accuracy": round(candidate_balanced, 6),
        "balanced_accuracy_gain": round(balanced_gain, 6),
    }

    return accepted, diagnostics


def learn_do_no_harm_recompute_rules(
    development_df: pd.DataFrame,
    min_support: int,
    min_error_rate: float,
    validation_size: float,
    random_state: int,
    min_affected: int,
    min_accuracy_gain: float,
) -> Tuple[
    Dict[str, Dict[str, Any]],
    Dict[Tuple[str, str], Dict[str, Any]],
    Dict[str, Any],
]:
    inner_train, inner_validation = split_inner_development(
        development_df=development_df,
        validation_size=validation_size,
        random_state=random_state,
    )

    candidate_stage_rules = learn_stage_rules(
        development_df=inner_train,
        min_support=min_support,
        min_error_rate=min_error_rate,
    )
    candidate_move_rules = learn_conditional_move_rules(
        development_df=inner_train,
        min_support=min_support,
        min_error_rate=min_error_rate,
    )

    accepted_stage_rules: Dict[str, Dict[str, Any]] = {}
    accepted_move_rules: Dict[
        Tuple[str, str],
        Dict[str, Any],
    ] = {}
    decisions: List[Dict[str, Any]] = []

    current_predictions = inner_validation[
        "auto_timing"
    ].map(norm)

    # Greedily retain only rules that improve validation correctness while
    # preserving balanced accuracy.
    candidates: List[Tuple[str, Any, Dict[str, Any]]] = []
    candidates.extend(
        ("stage", key, value)
        for key, value in candidate_stage_rules.items()
    )
    candidates.extend(
        ("move", key, value)
        for key, value in candidate_move_rules.items()
    )
    candidates.sort(
        key=lambda item: (
            -int(item[2].get("support", 0)),
            -float(item[2].get("error_rate", 0.0)),
            str(item[1]),
        )
    )

    for rule_type, key, rule in candidates:
        trial_stage = dict(accepted_stage_rules)
        trial_move = dict(accepted_move_rules)

        if rule_type == "stage":
            trial_stage[key] = rule
            affected_mask = (
                inner_validation["auto_stage"].map(norm)
                == norm(key)
            )
        else:
            trial_move[key] = rule
            affected_mask = (
                inner_validation["auto_stage"].map(norm)
                == norm(key[0])
            ) & (
                inner_validation["auto_move"].map(norm)
                == norm(key[1])
            )

        candidate_predictions = (
            predict_recomputed_timing_on_consensus(
                dataframe=inner_validation,
                stage_rules=trial_stage,
                move_rules=trial_move,
            )
        )

        accepted, diagnostics = accept_candidate_without_harm(
            validation_df=inner_validation,
            current_predictions=current_predictions,
            candidate_predictions=candidate_predictions,
            affected_mask=affected_mask,
            min_affected=min_affected,
            min_accuracy_gain=min_accuracy_gain,
        )

        decisions.append(
            {
                "rule_type": rule_type,
                "rule_key": (
                    "|||".join(key)
                    if isinstance(key, tuple)
                    else str(key)
                ),
                "rule_to": rule.get("to"),
                **diagnostics,
            }
        )

        if accepted:
            accepted_stage_rules = trial_stage
            accepted_move_rules = trial_move
            current_predictions = candidate_predictions

    final_eval = inner_validation.copy()
    final_eval["_prediction"] = current_predictions
    final_accuracy, final_balanced = (
        timing_accuracy_and_balanced_accuracy(
            final_eval,
            "_prediction",
        )
    )

    baseline_eval = inner_validation.copy()
    baseline_eval["_prediction"] = inner_validation[
        "auto_timing"
    ].map(norm)
    baseline_accuracy, baseline_balanced = (
        timing_accuracy_and_balanced_accuracy(
            baseline_eval,
            "_prediction",
        )
    )

    metadata = {
        "enabled": True,
        "inner_train_size": int(len(inner_train)),
        "inner_validation_size": int(len(inner_validation)),
        "candidate_stage_rules": int(len(candidate_stage_rules)),
        "candidate_move_rules": int(len(candidate_move_rules)),
        "accepted_stage_rules": int(len(accepted_stage_rules)),
        "accepted_move_rules": int(len(accepted_move_rules)),
        "baseline_inner_accuracy": round(baseline_accuracy, 6),
        "safe_inner_accuracy": round(final_accuracy, 6),
        "baseline_inner_balanced_accuracy": round(
            baseline_balanced,
            6,
        ),
        "safe_inner_balanced_accuracy": round(
            final_balanced,
            6,
        ),
        "decisions": decisions,
    }

    return accepted_stage_rules, accepted_move_rules, metadata


def learn_do_no_harm_direct_rules(
    development_df: pd.DataFrame,
    min_support: int,
    min_error_rate: float,
    validation_size: float,
    random_state: int,
    min_affected: int,
    min_accuracy_gain: float,
) -> Tuple[
    Dict[Tuple[str, str, str], Dict[str, Any]],
    Dict[str, Any],
]:
    inner_train, inner_validation = split_inner_development(
        development_df=development_df,
        validation_size=validation_size,
        random_state=random_state,
    )

    candidate_rules = learn_direct_timing_rules(
        development_df=inner_train,
        min_support=min_support,
        min_error_rate=min_error_rate,
    )

    accepted_rules: Dict[
        Tuple[str, str, str],
        Dict[str, Any],
    ] = {}
    decisions: List[Dict[str, Any]] = []
    current_predictions = inner_validation[
        "auto_timing"
    ].map(norm)

    ordered = sorted(
        candidate_rules.items(),
        key=lambda item: (
            -int(item[1].get("support", 0)),
            -float(item[1].get("error_rate", 0.0)),
            str(item[0]),
        ),
    )

    for key, rule in ordered:
        trial_rules = dict(accepted_rules)
        trial_rules[key] = rule

        candidate_predictions = predict_direct_timing_on_consensus(
            dataframe=inner_validation,
            timing_rules=trial_rules,
        )

        affected_mask = (
            inner_validation["auto_stage"].map(norm)
            == norm(key[0])
        ) & (
            inner_validation["auto_move"].map(norm)
            == norm(key[1])
        ) & (
            inner_validation["auto_timing"].map(norm)
            == norm(key[2])
        )

        accepted, diagnostics = accept_candidate_without_harm(
            validation_df=inner_validation,
            current_predictions=current_predictions,
            candidate_predictions=candidate_predictions,
            affected_mask=affected_mask,
            min_affected=min_affected,
            min_accuracy_gain=min_accuracy_gain,
        )

        decisions.append(
            {
                "rule_type": "direct_timing",
                "rule_key": "|||".join(key),
                "rule_to": rule.get("to"),
                **diagnostics,
            }
        )

        if accepted:
            enriched_rule = dict(rule)
            enriched_rule["validation_affected"] = diagnostics[
                "affected_validation_examples"
            ]
            enriched_rule["validation_accuracy_gain"] = diagnostics[
                "accuracy_gain"
            ]
            enriched_rule["validation_balanced_accuracy_gain"] = diagnostics[
                "balanced_accuracy_gain"
            ]
            enriched_rule["validation_current_correct"] = diagnostics[
                "current_correct"
            ]
            enriched_rule["validation_candidate_correct"] = diagnostics[
                "candidate_correct"
            ]
            accepted_rules = dict(accepted_rules)
            accepted_rules[key] = enriched_rule
            current_predictions = candidate_predictions

    metadata = {
        "enabled": True,
        "inner_train_size": int(len(inner_train)),
        "inner_validation_size": int(len(inner_validation)),
        "candidate_rules": int(len(candidate_rules)),
        "accepted_rules": int(len(accepted_rules)),
        "decisions": decisions,
    }

    return accepted_rules, metadata

# =============================================================================
# Confidence and isotonic reliability
# =============================================================================

def compute_confidence_flags(
    dataframe: pd.DataFrame,
    stage_percentile: float,
    move_percentile: float,
) -> Tuple[pd.DataFrame, Dict[str, Any]]:
    output = dataframe.copy()

    stage_values = output.loc[
        output["stage_confidence"] >= 0,
        "stage_confidence",
    ]

    move_values = output.loc[
        output["move_confidence"] >= 0,
        "move_confidence",
    ]

    stage_threshold = (
        float(
            stage_values.quantile(
                stage_percentile / 100.0
            )
        )
        if len(stage_values)
        else None
    )

    move_threshold = (
        float(
            move_values.quantile(
                move_percentile / 100.0
            )
        )
        if len(move_values)
        else None
    )

    output["stage_low_confidence"] = (
        output["stage_confidence"] < stage_threshold
        if stage_threshold is not None
        else False
    )

    output["move_low_confidence"] = (
        output["move_confidence"] < move_threshold
        if move_threshold is not None
        else False
    )

    output["low_confidence_calibrated"] = (
        output["stage_low_confidence"]
        | output["move_low_confidence"]
    )

    meta = {
        "stage_percentile": float(stage_percentile),
        "move_percentile": float(move_percentile),
        "stage_threshold": stage_threshold,
        "move_threshold": move_threshold,
        "full_corpus_low_confidence_rate": float(
            output[
                "low_confidence_calibrated"
            ].mean()
        ),
    }

    return output, meta


def minmax_scale(
    values: np.ndarray,
    minimum: float,
    maximum: float,
) -> np.ndarray:
    denominator = maximum - minimum

    if abs(denominator) < 1e-12:
        return np.full_like(
            values,
            0.5,
            dtype=float,
        )

    return (
        np.asarray(values, dtype=float)
        - minimum
    ) / denominator


def expected_calibration_error(
    targets: np.ndarray,
    probabilities: np.ndarray,
    n_bins: int = 10,
) -> float:
    targets = np.asarray(
        targets,
        dtype=float,
    )
    probabilities = np.asarray(
        probabilities,
        dtype=float,
    )

    if len(targets) == 0:
        return float("nan")

    edges = np.linspace(
        0.0,
        1.0,
        n_bins + 1,
    )

    total = 0.0

    for index in range(n_bins):
        lower = edges[index]
        upper = edges[index + 1]

        if index == n_bins - 1:
            mask = (
                (probabilities >= lower)
                & (probabilities <= upper)
            )
        else:
            mask = (
                (probabilities >= lower)
                & (probabilities < upper)
            )

        if not np.any(mask):
            continue

        bin_accuracy = float(
            np.mean(targets[mask])
        )
        bin_confidence = float(
            np.mean(probabilities[mask])
        )

        total += float(
            np.mean(mask)
        ) * abs(
            bin_accuracy - bin_confidence
        )

    return float(total)


def fit_isotonic_model(
    development_df: pd.DataFrame,
    target_column: str,
    feature_column: str,
) -> Tuple[Optional[Any], Dict[str, Any]]:
    if not SKLEARN_AVAILABLE:
        return None, {
            "available": False,
            "reason": "scikit-learn is unavailable",
        }

    train = development_df[
        development_df[target_column].isin(
            YES_NO
        )
    ].copy()

    if (
        len(train) < 20
        or train[target_column].nunique() < 2
    ):
        return None, {
            "available": False,
            "reason": (
                "Insufficient development labels or only one target class"
            ),
            "n_train": int(len(train)),
        }

    x_raw = (
        train[feature_column]
        .astype(float)
        .to_numpy()
    )

    y = (
        train[target_column]
        .eq("yes")
        .astype(int)
        .to_numpy()
    )

    minimum = float(np.min(x_raw))
    maximum = float(np.max(x_raw))
    x = minmax_scale(
        x_raw,
        minimum,
        maximum,
    )

    model = IsotonicRegression(
        out_of_bounds="clip",
        y_min=0.0,
        y_max=1.0,
    )
    model.fit(x, y)

    probabilities = model.predict(x)

    brier = (
        float(
            brier_score_loss(
                y,
                probabilities,
            )
        )
        if brier_score_loss is not None
        else None
    )

    wrapped = {
        "model": model,
        "minimum": minimum,
        "maximum": maximum,
        "feature_column": feature_column,
        "target_column": target_column,
    }

    meta = {
        "available": True,
        "n_train": int(len(train)),
        "positive_rate": float(np.mean(y)),
        "development_brier": brier,
        "development_ece": float(
            expected_calibration_error(
                y,
                probabilities,
            )
        ),
        "feature_minimum": minimum,
        "feature_maximum": maximum,
        "note": (
            "The isotonic model was fitted only on the development subset."
        ),
    }

    return wrapped, meta


def predict_isotonic(
    wrapped_model: Optional[Dict[str, Any]],
    dataframe: pd.DataFrame,
    output_column: str,
) -> pd.DataFrame:
    output = dataframe.copy()

    if wrapped_model is None:
        output[output_column] = 0.5
        return output

    feature_column = wrapped_model[
        "feature_column"
    ]

    raw = (
        output[feature_column]
        .astype(float)
        .to_numpy()
    )

    scaled = minmax_scale(
        raw,
        wrapped_model["minimum"],
        wrapped_model["maximum"],
    )

    output[output_column] = wrapped_model[
        "model"
    ].predict(scaled)

    return output


def add_isotonic_reliability(
    base_df: pd.DataFrame,
    development_df: pd.DataFrame,
    reliability_threshold: float,
) -> Tuple[pd.DataFrame, Dict[str, Any], Dict[str, Any]]:
    stage_model, stage_meta = fit_isotonic_model(
        development_df=development_df,
        target_column="human_stage_correct",
        feature_column="stage_confidence",
    )

    move_model, move_meta = fit_isotonic_model(
        development_df=development_df,
        target_column="human_move_correct",
        feature_column="move_confidence",
    )

    timing_model, timing_meta = fit_isotonic_model(
        development_df=development_df,
        target_column="human_timing_correct",
        feature_column="combined_confidence",
    )

    output = predict_isotonic(
        stage_model,
        base_df,
        "isotonic_stage_reliability",
    )

    output = predict_isotonic(
        move_model,
        output,
        "isotonic_move_reliability",
    )

    output = predict_isotonic(
        timing_model,
        output,
        "isotonic_timing_reliability",
    )

    output["isotonic_overall_reliability"] = output[
        [
            "isotonic_stage_reliability",
            "isotonic_move_reliability",
            "isotonic_timing_reliability",
        ]
    ].min(axis=1)

    output["isotonic_high_reliability"] = (
        output[
            "isotonic_overall_reliability"
        ]
        >= reliability_threshold
    )

    meta = {
        "reliability_threshold": float(
            reliability_threshold
        ),
        "stage": stage_meta,
        "move": move_meta,
        "timing": timing_meta,
        "full_corpus_high_reliability_rate": float(
            output[
                "isotonic_high_reliability"
            ].mean()
        ),
    }

    model_bundle = {
        "stage": stage_model,
        "move": move_model,
        "timing": timing_model,
    }

    return output, meta, model_bundle


# =============================================================================
# Apply calibration methods
# =============================================================================

def apply_safe_keep_correct_review_policy(
    base_df: pd.DataFrame,
    timing_rules: Dict[
        Tuple[str, str, str],
        Dict[str, Any],
    ],
    keep_threshold: float,
    correction_max_original_reliability: float,
    min_rule_support: int,
    min_rule_error_rate: float,
    min_rule_validation_gain: float,
) -> pd.DataFrame:
    """
    Three-way safety policy:
      1. KEEP high-reliability original labels.
      2. CORRECT only with an internally validated direct rule and when the
         original label is not already highly reliable.
      3. REVIEW all remaining uncertain cases while preserving the original
         label in the exported prediction.

    Human review is an abstention decision, not an automatic relabelling.
    """
    output = base_df.copy()

    actions: List[str] = []
    final_timings: List[str] = []
    sources: List[str] = []
    review_flags: List[bool] = []
    rule_supports: List[int] = []
    rule_gains: List[float] = []

    for _, row in output.iterrows():
        stage = norm(row["auto_stage_for_calibration"])
        move = norm(row["auto_move_for_calibration"])
        original = norm(row["auto_timing_for_calibration"])
        reliability = safe_float(
            row.get("isotonic_overall_reliability", 0.5),
            0.5,
        )

        key = (stage, move, original)
        rule = timing_rules.get(key)

        if reliability >= keep_threshold:
            action = "keep"
            final = original
            source = "high_reliability_keep"
            needs_review = False
            support = 0
            gain = 0.0

        elif rule is not None:
            support = int(rule.get("support", 0))
            error_rate = float(rule.get("error_rate", 0.0))
            gain = float(rule.get("validation_accuracy_gain", 0.0))
            corrected = norm(rule.get("to", original))

            strong_rule = (
                support >= min_rule_support
                and error_rate >= min_rule_error_rate
                and gain >= min_rule_validation_gain
                and reliability <= correction_max_original_reliability
                and corrected in TIMINGS
                and corrected != original
            )

            if strong_rule:
                action = "correct"
                final = corrected
                source = "validated_direct_rule"
                needs_review = False
            else:
                action = "review"
                final = original
                source = "rule_not_strong_enough_review"
                needs_review = True

        else:
            action = "review"
            final = original
            source = "uncertain_no_validated_rule"
            needs_review = True
            support = 0
            gain = 0.0

        actions.append(action)
        final_timings.append(final)
        sources.append(source)
        review_flags.append(needs_review)
        rule_supports.append(support)
        rule_gains.append(gain)

    output["calibration_method"] = "safe_keep_correct_review"
    output["calibrated_stage"] = output[
        "auto_stage_for_calibration"
    ].map(norm)
    output["stage_calibration_source"] = "kept"
    output["calibrated_move"] = output[
        "auto_move_for_calibration"
    ].map(norm)
    output["move_calibration_source"] = "kept"
    output["calibrated_timing"] = final_timings
    output["timing_calibration_source"] = sources
    output["policy_action"] = actions
    output["needs_human_review"] = review_flags
    output["policy_accepted"] = ~output["needs_human_review"]
    output["policy_rule_support"] = rule_supports
    output["policy_rule_validation_gain"] = rule_gains
    output["calibrated_is_well_timed"] = output[
        "calibrated_timing"
    ].eq("well_timed")
    output["stage_changed"] = False
    output["move_changed"] = False
    output["timing_changed"] = (
        output["calibrated_timing"]
        != output["auto_timing_for_calibration"]
    )

    return output


def apply_calibration_method(
    base_df: pd.DataFrame,
    method: str,
    stage_rules: Optional[
        Dict[str, Dict[str, Any]]
    ] = None,
    move_rules: Optional[
        Dict[Tuple[str, str], Dict[str, Any]]
    ] = None,
    timing_rules: Optional[
        Dict[
            Tuple[str, str, str],
            Dict[str, Any],
        ]
    ] = None,
) -> pd.DataFrame:
    stage_rules = stage_rules or {}
    move_rules = move_rules or {}
    timing_rules = timing_rules or {}

    output = base_df.copy()

    calibrated_stages: List[str] = []
    stage_sources: List[str] = []
    calibrated_moves: List[str] = []
    move_sources: List[str] = []
    calibrated_timings: List[str] = []
    timing_sources: List[str] = []

    for _, row in output.iterrows():
        original_stage = norm(
            row["auto_stage_for_calibration"]
        )
        original_move = norm(
            row["auto_move_for_calibration"]
        )
        original_timing = norm(
            row["auto_timing_for_calibration"]
        )

        if method in RULE_METHODS:
            stage, stage_source = apply_stage_rule(
                original_stage,
                stage_rules,
            )

            move, move_source = (
                apply_conditional_move_rule(
                    original_stage,
                    original_move,
                    move_rules,
                )
            )
        else:
            stage = original_stage
            stage_source = "kept"
            move = original_move
            move_source = "kept"

        if method == "human_rules_direct":
            timing, timing_source = (
                apply_direct_timing_rule(
                    original_stage,
                    original_move,
                    original_timing,
                    timing_rules,
                )
            )

        elif method in RECOMPUTE_METHODS:
            timing = timing_from_stage_move(
                stage,
                move,
            )
            timing_source = (
                "recomputed_from_calibrated_stage_move"
            )

        else:
            timing = original_timing
            timing_source = "kept"

        calibrated_stages.append(stage)
        stage_sources.append(stage_source)
        calibrated_moves.append(move)
        move_sources.append(move_source)
        calibrated_timings.append(timing)
        timing_sources.append(timing_source)

    output["calibration_method"] = method
    output["calibrated_stage"] = calibrated_stages
    output["stage_calibration_source"] = stage_sources
    output["calibrated_move"] = calibrated_moves
    output["move_calibration_source"] = move_sources
    output["calibrated_timing"] = calibrated_timings
    output["timing_calibration_source"] = timing_sources
    output["calibrated_is_well_timed"] = output[
        "calibrated_timing"
    ].eq("well_timed")

    output["stage_changed"] = (
        output["calibrated_stage"]
        != output["auto_stage_for_calibration"]
    )

    output["move_changed"] = (
        output["calibrated_move"]
        != output["auto_move_for_calibration"]
    )

    output["timing_changed"] = (
        output["calibrated_timing"]
        != output["auto_timing_for_calibration"]
    )

    return output


# =============================================================================
# Full-corpus descriptive summaries
# =============================================================================

def summarize_full_corpus(
    dataframe: pd.DataFrame,
    method: str,
) -> Dict[str, Any]:
    rank1 = dataframe[
        dataframe["rank_for_calibration"] == 1
    ].copy()

    if rank1.empty:
        raise ValueError(
            "No rank-1 rows were available for full-corpus summary."
        )

    original_rate = float(
        rank1[
            "auto_is_well_timed_for_calibration"
        ].mean()
    )

    calibrated_rate = float(
        rank1[
            "calibrated_is_well_timed"
        ].mean()
    )

    high_confidence = rank1[
        ~rank1["low_confidence_calibrated"]
    ]

    high_confidence_rate = (
        float(
            high_confidence[
                "calibrated_is_well_timed"
            ].mean()
        )
        if len(high_confidence)
        else None
    )

    isotonic_subset = (
        rank1[
            rank1["isotonic_high_reliability"]
        ]
        if "isotonic_high_reliability" in rank1.columns
        else pd.DataFrame()
    )

    isotonic_rate = (
        float(
            isotonic_subset[
                "calibrated_is_well_timed"
            ].mean()
        )
        if len(isotonic_subset)
        else None
    )

    isotonic_coverage = (
        float(
            rank1[
                "isotonic_high_reliability"
            ].mean()
        )
        if "isotonic_high_reliability" in rank1.columns
        else None
    )

    return {
        "method": method,
        "original_automatic_well_timed_rate_at_1": round(
            original_rate,
            4,
        ),
        "calibrated_automatic_well_timed_rate_at_1": round(
            calibrated_rate,
            4,
        ),
        "delta_automatic_well_timed_rate_at_1": round(
            calibrated_rate - original_rate,
            4,
        ),
        "high_confidence_automatic_well_timed_rate_at_1": (
            round(high_confidence_rate, 4)
            if high_confidence_rate is not None
            else None
        ),
        "high_confidence_coverage": round(
            float(
                (
                    ~rank1[
                        "low_confidence_calibrated"
                    ]
                ).mean()
            ),
            4,
        ),
        "isotonic_high_reliability_automatic_well_timed_rate_at_1": (
            round(isotonic_rate, 4)
            if isotonic_rate is not None
            else None
        ),
        "isotonic_high_reliability_coverage": (
            round(isotonic_coverage, 4)
            if isotonic_coverage is not None
            else None
        ),
        "stage_changes_rank1": int(
            rank1["stage_changed"].sum()
        ),
        "move_changes_rank1": int(
            rank1["move_changed"].sum()
        ),
        "timing_changes_rank1": int(
            rank1["timing_changed"].sum()
        ),
        "stage_changes_all_rows": int(
            dataframe["stage_changed"].sum()
        ),
        "move_changes_all_rows": int(
            dataframe["move_changed"].sum()
        ),
        "timing_changes_all_rows": int(
            dataframe["timing_changed"].sum()
        ),
        "policy_keep_count_rank1": int(
            rank1["policy_action"].eq("keep").sum()
        ) if "policy_action" in rank1.columns else None,
        "policy_correct_count_rank1": int(
            rank1["policy_action"].eq("correct").sum()
        ) if "policy_action" in rank1.columns else None,
        "policy_review_count_rank1": int(
            rank1["policy_action"].eq("review").sum()
        ) if "policy_action" in rank1.columns else None,
        "policy_accepted_coverage_rank1": round(
            float(rank1["policy_accepted"].mean()), 4
        ) if "policy_accepted" in rank1.columns else None,
        "n_rank1": int(len(rank1)),
        "n_rows": int(len(dataframe)),
        "interpretation": (
            "These are automatic label-distribution statistics, not "
            "human-validated accuracy."
        ),
    }


# =============================================================================
# Held-out human evaluation
# =============================================================================

def build_human_reference(
    row: pd.Series,
    label_type: str,
) -> Optional[str]:
    auto_value = norm(row[f"auto_{label_type}"])
    correctness = norm(row[f"human_{label_type}_correct"])
    human_value = norm(row[f"human_{label_type}"])

    if correctness == "yes":
        return auto_value

    if (
        correctness == "no"
        and human_value
        and human_value != auto_value  # <-- FIX: a tie/no-correction fallback isn't a real reference
    ):
        return human_value

    return None


def make_heldout_auto_rows(
    auto_df: pd.DataFrame,
    heldout_df: pd.DataFrame,
) -> pd.DataFrame:
    rank1 = auto_df[
        auto_df["rank_for_calibration"] == 1
    ].copy()

    wanted_ids = set(
        heldout_df["id"].astype(str)
    )

    selected = rank1[
        rank1["id_for_calibration"].isin(
            wanted_ids
        )
    ].copy()

    if len(selected) != len(wanted_ids):
        selected_ids = set(
            selected["id_for_calibration"]
        )

        missing = sorted(
            wanted_ids - selected_ids
        )

        raise ValueError(
            "Held-out examples could not all be matched to rank-1 "
            f"automatic judgments. Missing: {missing[:10]}"
        )

    return selected


def evaluate_method_on_heldout(
    method_output: pd.DataFrame,
    heldout_df: pd.DataFrame,
    method: str,
) -> Tuple[Dict[str, Any], pd.DataFrame]:
    rank1 = method_output[
        method_output["rank_for_calibration"] == 1
    ].copy()

    human_columns = [
        "id",
        "auto_stage",
        "auto_move",
        "auto_timing",
        "human_stage_correct",
        "human_stage",
        "human_move_correct",
        "human_move",
        "human_timing_correct",
        "human_timing",
    ]

    human = heldout_df[
        human_columns
    ].copy()

    human["id_for_calibration"] = (
        human["id"]
        .astype(str)
        .str.strip()
    )

    evaluation = rank1.merge(
        human.drop(columns=["id"]),
        on="id_for_calibration",
        how="inner",
        validate="one_to_one",
        suffixes=("", "_human"),
    )

    if len(evaluation) != len(heldout_df):
        raise ValueError(
            f"Held-out evaluation matched {len(evaluation)} of "
            f"{len(heldout_df)} examples for method {method}."
        )

    for label_type in [
        "stage",
        "move",
        "timing",
    ]:
        evaluation[
            f"human_reference_{label_type}"
        ] = evaluation.apply(
            lambda row: build_human_reference(
                row,
                label_type,
            ),
            axis=1,
        )

        reference_column = (
            f"human_reference_{label_type}"
        )

        evaluation[
            f"original_{label_type}_correct"
        ] = (
            evaluation[
                f"auto_{label_type}_for_calibration"
            ]
            == evaluation[reference_column]
        )

        evaluation[
            f"calibrated_{label_type}_correct"
        ] = (
            evaluation[
                f"calibrated_{label_type}"
            ]
            == evaluation[reference_column]
        )

    def accuracy(column: str) -> float:
        return float(
            evaluation[column].mean()
        )

    original_stage_accuracy = accuracy(
        "original_stage_correct"
    )
    calibrated_stage_accuracy = accuracy(
        "calibrated_stage_correct"
    )

    original_move_accuracy = accuracy(
        "original_move_correct"
    )
    calibrated_move_accuracy = accuracy(
        "calibrated_move_correct"
    )

    original_timing_accuracy = accuracy(
        "original_timing_correct"
    )
    calibrated_timing_accuracy = accuracy(
        "calibrated_timing_correct"
    )

    high_confidence = evaluation[
        ~evaluation["low_confidence_calibrated"]
    ]

    high_confidence_accuracy = (
        float(
            high_confidence[
                "calibrated_timing_correct"
            ].mean()
        )
        if len(high_confidence)
        else None
    )

    isotonic_subset = (
        evaluation[
            evaluation["isotonic_high_reliability"]
        ]
        if "isotonic_high_reliability" in evaluation.columns
        else pd.DataFrame()
    )

    isotonic_accuracy = (
        float(
            isotonic_subset[
                "calibrated_timing_correct"
            ].mean()
        )
        if len(isotonic_subset)
        else None
    )

    isotonic_coverage = (
        float(
            evaluation[
                "isotonic_high_reliability"
            ].mean()
        )
        if "isotonic_high_reliability" in evaluation.columns
        else None
    )

    policy_metrics: Dict[str, Any] = {}
    if "policy_action" in evaluation.columns:
        accepted = evaluation[evaluation["policy_accepted"]].copy()
        reviewed = evaluation[evaluation["needs_human_review"]].copy()
        corrected = evaluation[evaluation["policy_action"] == "correct"].copy()

        policy_metrics = {
            "policy_accepted_coverage": round(
                float(len(accepted) / len(evaluation)), 4
            ),
            "policy_review_rate": round(
                float(len(reviewed) / len(evaluation)), 4
            ),
            "policy_accepted_timing_accuracy": (
                round(float(accepted["calibrated_timing_correct"].mean()), 4)
                if len(accepted) else None
            ),
            "policy_n_keep": int(
                evaluation["policy_action"].eq("keep").sum()
            ),
            "policy_n_correct": int(len(corrected)),
            "policy_n_review": int(len(reviewed)),
            "policy_corrected_case_accuracy": (
                round(float(corrected["calibrated_timing_correct"].mean()), 4)
                if len(corrected) else None
            ),
        }

    summary = {
        "method": method,
        "n_heldout": int(len(evaluation)),
        "original_stage_accuracy": round(
            original_stage_accuracy,
            4,
        ),
        "calibrated_stage_accuracy": round(
            calibrated_stage_accuracy,
            4,
        ),
        "stage_accuracy_gain": round(
            calibrated_stage_accuracy
            - original_stage_accuracy,
            4,
        ),
        "original_move_accuracy": round(
            original_move_accuracy,
            4,
        ),
        "calibrated_move_accuracy": round(
            calibrated_move_accuracy,
            4,
        ),
        "move_accuracy_gain": round(
            calibrated_move_accuracy
            - original_move_accuracy,
            4,
        ),
        "original_timing_accuracy": round(
            original_timing_accuracy,
            4,
        ),
        "calibrated_timing_accuracy": round(
            calibrated_timing_accuracy,
            4,
        ),
        "timing_accuracy_gain": round(
            calibrated_timing_accuracy
            - original_timing_accuracy,
            4,
        ),
        "high_confidence_timing_accuracy": (
            round(high_confidence_accuracy, 4)
            if high_confidence_accuracy is not None
            else None
        ),
        "high_confidence_coverage": round(
            float(
                (
                    ~evaluation[
                        "low_confidence_calibrated"
                    ]
                ).mean()
            ),
            4,
        ),
        "isotonic_high_reliability_timing_accuracy": (
            round(isotonic_accuracy, 4)
            if isotonic_accuracy is not None
            else None
        ),
        "isotonic_high_reliability_coverage": (
            round(isotonic_coverage, 4)
            if isotonic_coverage is not None
            else None
        ),
        "interpretation": (
            "These accuracies are measured on an untouched held-out "
            "human-consensus subset."
        ),
    }

    summary.update(policy_metrics)
    return summary, evaluation


# =============================================================================
# Selective held-out reliability analysis
# =============================================================================

def heldout_risk_coverage_table(
    evaluation_df: pd.DataFrame,
    score_column: str,
    coverage_points: Optional[
        Sequence[float]
    ] = None,
) -> pd.DataFrame:
    if coverage_points is None:
        coverage_points = [
            1.00,
            0.90,
            0.80,
            0.70,
            0.60,
            0.50,
            0.40,
            0.30,
            0.20,
            0.10,
        ]

    if score_column not in evaluation_df.columns:
        return pd.DataFrame()

    ranked = (
        evaluation_df
        .sort_values(
            score_column,
            ascending=False,
        )
        .reset_index(drop=True)
    )

    total = len(ranked)
    rows: List[Dict[str, Any]] = []

    for target_coverage in coverage_points:
        accepted_count = max(
            1,
            int(round(
                target_coverage * total
            )),
        )

        subset = ranked.iloc[
            :accepted_count
        ]

        accuracy = float(
            subset[
                "calibrated_timing_correct"
            ].mean()
        )

        rows.append(
            {
                "score_column": score_column,
                "target_coverage": float(
                    target_coverage
                ),
                "actual_coverage": float(
                    accepted_count / total
                ),
                "n_accepted": int(
                    accepted_count
                ),
                "n_total": int(total),
                "threshold": float(
                    subset[score_column].min()
                ),
                "selective_human_validated_accuracy": round(
                    accuracy,
                    4,
                ),
                "selective_human_validated_risk": round(
                    1.0 - accuracy,
                    4,
                ),
            }
        )

    return pd.DataFrame(rows)


Writing theratime_post_calibration.py


# **theratime_calibration_robustness**

In [4]:
%%writefile theratime_calibration_robustness.py
#!/usr/bin/env python3
"""
theratime_calibration_robustness.py
====================================

Robustness layer on top of theratime_post_calibration.py (v2).

This module does NOT reimplement the calibration pipeline. It imports the
v2 functions (consensus building, dev/held-out splitting, rule learning,
isotonic reliability, the safe_keep_correct_review policy) and adds three
things that a reviewer would reasonably ask for once you claim a calibration
improvement on a ~150-300 item human-annotated sample:

1. BOOTSTRAP CONFIDENCE INTERVALS
   Percentile bootstrap over the held-out evaluation rows, for each method's
   held-out timing/stage/move accuracy. Answers: "how much would this number
   move if we'd happened to sample a slightly different held-out set?"

2. MULTI-SEED STABILITY
   Re-runs the ENTIRE pipeline (dev/held-out split -> inner train/validation
   split -> rule learning with the do-no-harm gate -> isotonic fit ->
   held-out evaluation) under several different random seeds. Reports the
   mean / std / min / max of held-out timing accuracy per method across
   seeds. Answers: "is the reported gain a property of the method, or an
   artifact of one lucky 70/30 split?"

3. THRESHOLD SENSITIVITY SWEEP
   For the safe_keep_correct_review policy specifically, sweeps
   policy_keep_threshold and policy_correction_max_original_reliability over
   a grid and reports held-out accepted-coverage / accepted-accuracy /
   review-rate for each combination. Answers: "why these threshold values,
   and how much do results change if we'd picked slightly different ones?"

Nothing here changes labels beyond what v2 already does. This module only
adds uncertainty quantification and sensitivity analysis around v2's
existing, defensible pipeline.

Typical usage
-------------
python theratime_calibration_robustness.py \
  --auto all_judgments_mpnet.csv \
  --ann theratime_300_Hasnae.csv theratime_300_Asmae.csv theratime_300_external.csv \
  --out-dir theratime_robustness_outputs \
  --methods baseline conservative_human_recompute safe_keep_correct_review \
  --seeds 0 1 2 3 4 5 6 7 8 9 \
  --n-bootstrap 2000

Recommended paper wording
--------------------------
"Held-out timing accuracy is reported with 95% percentile bootstrap
confidence intervals (2000 resamples) over the held-out evaluation subset.
To assess sensitivity to the specific development/held-out split, the full
calibration pipeline -- including do-no-harm rule learning -- was repeated
under 10 random seeds; we report the mean and range of held-out timing
accuracy across seeds. The safe_keep_correct_review policy's reliability
and correction thresholds were selected via a sensitivity sweep reported in
the supplementary material, rather than a single untested default."
"""

from __future__ import annotations

import argparse
import json
from pathlib import Path
from typing import Any, Dict, List, Optional, Sequence, Tuple

import numpy as np
import pandas as pd

import theratime_post_calibration as tpc


# =============================================================================
# 1. Bootstrap confidence intervals
# =============================================================================

def bootstrap_ci_from_bool_series(
    correct: pd.Series,
    n_bootstrap: int = 2000,
    alpha: float = 0.05,
    seed: int = 0,
) -> Dict[str, float]:
    """Percentile bootstrap CI for a proportion (e.g. accuracy) over rows."""
    values = correct.astype(float).to_numpy()
    n = len(values)

    if n == 0:
        return {
            "point": float("nan"),
            "ci_low": float("nan"),
            "ci_high": float("nan"),
            "n": 0,
            "n_bootstrap": int(n_bootstrap),
        }

    rng = np.random.default_rng(seed)
    point = float(values.mean())

    # Resample row indices with replacement, n_bootstrap times.
    boot_indices = rng.integers(0, n, size=(n_bootstrap, n))
    boot_means = values[boot_indices].mean(axis=1)

    ci_low, ci_high = np.percentile(
        boot_means,
        [100 * alpha / 2, 100 * (1 - alpha / 2)],
    )

    return {
        "point": round(point, 4),
        "ci_low": round(float(ci_low), 4),
        "ci_high": round(float(ci_high), 4),
        "n": int(n),
        "n_bootstrap": int(n_bootstrap),
        "alpha": float(alpha),
    }


def bootstrap_ci_for_heldout_evaluation(
    evaluation_df: pd.DataFrame,
    n_bootstrap: int,
    alpha: float,
    seed: int,
) -> Dict[str, Dict[str, float]]:
    """Bootstrap CIs for stage/move/timing calibrated accuracy on one
    held-out evaluation dataframe (as returned by evaluate_method_on_heldout).
    """
    results: Dict[str, Dict[str, float]] = {}

    for label_type, seed_offset in [("stage", 0), ("move", 1), ("timing", 2)]:
        column = f"calibrated_{label_type}_correct"
        if column not in evaluation_df.columns:
            continue
        results[label_type] = bootstrap_ci_from_bool_series(
            evaluation_df[column],
            n_bootstrap=n_bootstrap,
            alpha=alpha,
            seed=seed + seed_offset,
        )

    # Also bootstrap the "accepted subset" accuracy for the safe policy,
    # since that's the number most likely to be quoted in the paper.
    if "policy_accepted" in evaluation_df.columns:
        accepted = evaluation_df[evaluation_df["policy_accepted"]]
        results["policy_accepted_timing"] = bootstrap_ci_from_bool_series(
            accepted["calibrated_timing_correct"],
            n_bootstrap=n_bootstrap,
            alpha=alpha,
            seed=seed + 3,
        )

    return results


# =============================================================================
# 2. Single full-pipeline run (one seed) -- mirrors v2's main(), as a function
# =============================================================================

def run_pipeline_once(
    auto_path: Path,
    annotation_paths: Sequence[Path],
    methods: Sequence[str],
    seed: int,
    consensus_mode: str = "auto",
    heldout_size: float = 0.30,
    standard_min_support: int = 3,
    standard_min_error_rate: float = 0.50,
    conservative_min_support: int = 5,
    conservative_min_error_rate: float = 0.60,
    do_no_harm: bool = True,
    inner_validation_size: float = 0.25,
    min_rule_affected: int = 3,
    min_rule_accuracy_gain: float = 0.0,
    stage_confidence_percentile: float = 10.0,
    move_confidence_percentile: float = 10.0,
    isotonic_reliability_threshold: float = 0.50,
    policy_keep_threshold: float = 0.75,
    policy_correction_max_original_reliability: float = 0.60,
    policy_min_rule_support: int = 5,
    policy_min_rule_error_rate: float = 0.60,
    policy_min_rule_validation_gain: float = 0.0,
) -> Dict[str, Any]:
    """Run the full v2 pipeline once under a given random seed.

    Returns a dict with:
      - 'heldout_summaries': {method: summary_dict}
      - 'heldout_evaluations': {method: evaluation_dataframe}
      - 'split_meta', 'standard_meta'
    """
    base_df = tpc.prepare_auto_df(auto_path)
    base_df, _confidence_meta = tpc.compute_confidence_flags(
        base_df,
        stage_percentile=stage_confidence_percentile,
        move_percentile=move_confidence_percentile,
    )

    standard_consensus_df, standard_meta = tpc.build_consensus(
        annotation_paths=annotation_paths,
        require_same_correction=False,
        consensus_mode=consensus_mode,
    )
    conservative_consensus_df, _conservative_meta = tpc.build_consensus(
        annotation_paths=annotation_paths,
        require_same_correction=True,
        consensus_mode=consensus_mode,
    )

    base_df, _id_config = tpc.configure_auto_id_column(
        auto_df=base_df,
        annotation_ids=standard_consensus_df["id"].tolist(),
    )
    tpc.validate_annotation_ids_against_auto(base_df, standard_consensus_df)

    standard_development, standard_heldout, split_meta = tpc.split_consensus_data(
        consensus_df=standard_consensus_df,
        test_size=heldout_size,
        random_state=seed,
    )

    development_ids = set(standard_development["id"])
    heldout_ids = set(standard_heldout["id"])

    conservative_development = (
        conservative_consensus_df[conservative_consensus_df["id"].isin(development_ids)]
        .copy()
        .reset_index(drop=True)
    )

    if do_no_harm:
        standard_stage_rules, standard_move_rules, _std_safe_meta = (
            tpc.learn_do_no_harm_recompute_rules(
                development_df=standard_development,
                min_support=standard_min_support,
                min_error_rate=standard_min_error_rate,
                validation_size=inner_validation_size,
                random_state=seed + 100,
                min_affected=min_rule_affected,
                min_accuracy_gain=min_rule_accuracy_gain,
            )
        )
        conservative_stage_rules, conservative_move_rules, _cons_safe_meta = (
            tpc.learn_do_no_harm_recompute_rules(
                development_df=conservative_development,
                min_support=conservative_min_support,
                min_error_rate=conservative_min_error_rate,
                validation_size=inner_validation_size,
                random_state=seed + 200,
                min_affected=min_rule_affected,
                min_accuracy_gain=min_rule_accuracy_gain,
            )
        )
        standard_timing_rules, _direct_safe_meta = tpc.learn_do_no_harm_direct_rules(
            development_df=standard_development,
            min_support=standard_min_support,
            min_error_rate=standard_min_error_rate,
            validation_size=inner_validation_size,
            random_state=seed + 300,
            min_affected=min_rule_affected,
            min_accuracy_gain=min_rule_accuracy_gain,
        )
        conservative_timing_rules: Dict[Any, Any] = {}
    else:
        standard_stage_rules = tpc.learn_stage_rules(
            standard_development, standard_min_support, standard_min_error_rate
        )
        standard_move_rules = tpc.learn_conditional_move_rules(
            standard_development, standard_min_support, standard_min_error_rate
        )
        standard_timing_rules = tpc.learn_direct_timing_rules(
            standard_development, standard_min_support, standard_min_error_rate
        )
        conservative_stage_rules = tpc.learn_stage_rules(
            conservative_development, conservative_min_support, conservative_min_error_rate
        )
        conservative_move_rules = tpc.learn_conditional_move_rules(
            conservative_development, conservative_min_support, conservative_min_error_rate
        )
        conservative_timing_rules = tpc.learn_direct_timing_rules(
            conservative_development, conservative_min_support, conservative_min_error_rate
        )

    development_auto_rows = tpc.make_heldout_auto_rows(base_df, standard_development)
    development_with_humans = development_auto_rows.merge(
        standard_development[
            ["id", "human_stage_correct", "human_move_correct", "human_timing_correct"]
        ].rename(columns={"id": "id_for_calibration"}),
        on="id_for_calibration",
        how="inner",
        validate="one_to_one",
    )

    isotonic_base_df, _iso_meta, _iso_models = tpc.add_isotonic_reliability(
        base_df=base_df,
        development_df=development_with_humans,
        reliability_threshold=isotonic_reliability_threshold,
    )

    heldout_summaries: Dict[str, Dict[str, Any]] = {}
    heldout_evaluations: Dict[str, pd.DataFrame] = {}

    for method in methods:
        # isotonic_base_df is base_df plus isotonic_*_reliability columns --
        # a pure addition, so using it as the source for EVERY method (not
        # just the ones whose label logic reads those columns) is safe and
        # ensures isotonic_overall_reliability is always available for
        # coverage_target_report, regardless of which methods were requested.
        method_base = isotonic_base_df.copy()

        if method == "human_rules_direct":
            stage_rules, move_rules, timing_rules = (
                standard_stage_rules, standard_move_rules, standard_timing_rules
            )
        elif method == "human_rules_recompute":
            stage_rules, move_rules, timing_rules = (
                standard_stage_rules, standard_move_rules, {}
            )
        elif method in {
            "conservative_human_recompute",
            "conservative_recompute_with_isotonic_reliability",
        }:
            stage_rules, move_rules, timing_rules = (
                conservative_stage_rules, conservative_move_rules, conservative_timing_rules
            )
        else:
            stage_rules, move_rules, timing_rules = {}, {}, {}

        if method == "safe_keep_correct_review":
            method_output = tpc.apply_safe_keep_correct_review_policy(
                base_df=method_base,
                timing_rules=standard_timing_rules,
                keep_threshold=policy_keep_threshold,
                correction_max_original_reliability=policy_correction_max_original_reliability,
                min_rule_support=policy_min_rule_support,
                min_rule_error_rate=policy_min_rule_error_rate,
                min_rule_validation_gain=policy_min_rule_validation_gain,
            )
        else:
            method_output = tpc.apply_calibration_method(
                base_df=method_base,
                method=method,
                stage_rules=stage_rules,
                move_rules=move_rules,
                timing_rules=timing_rules,
            )

        summary, evaluation = tpc.evaluate_method_on_heldout(
            method_output=method_output,
            heldout_df=standard_heldout,
            method=method,
        )
        heldout_summaries[method] = summary
        heldout_evaluations[method] = evaluation

    return {
        "seed": seed,
        "heldout_summaries": heldout_summaries,
        "heldout_evaluations": heldout_evaluations,
        "split_meta": split_meta,
        "standard_meta": standard_meta,
        "standard_timing_rules": standard_timing_rules,
        "isotonic_base_df": isotonic_base_df,
        "standard_development": standard_development,
        "standard_heldout": standard_heldout,
    }


# =============================================================================
# 2.5 K-fold pooled held-out evaluation (tightens CI WITHOUT new annotation)
# =============================================================================
#
# A single 70/30 development/held-out split only ever EVALUATES the 30% held
# out; the other 70% is only ever used to LEARN rules. That means a 300-item
# annotated sample gives an accuracy estimate with the precision of n=90,
# not n=300 -- most of the collected labels never contribute to the reported
# confidence interval.
#
# K-fold cross-validation fixes this without collecting a single new label:
# rotate which 1/K of the consensus data is held out, learn rules on the
# other (K-1)/K each time (with the same do-no-harm gate as a single split),
# and evaluate each fold's held-out portion. Every item is held out exactly
# once, by a model that never saw it during rule learning, so pooling all K
# folds' predictions gives a legitimate out-of-fold accuracy estimate over
# the FULL annotated sample (~300, not ~90) -- roughly halving the
# confidence interval width for the same annotation effort already spent.

def run_kfold_pooled_evaluation(
    auto_path: Path,
    annotation_paths: Sequence[Path],
    methods: Sequence[str],
    k_folds: int = 5,
    seed: int = 42,
    n_bootstrap: int = 2000,
    alpha: float = 0.05,
    consensus_mode: str = "auto",
    standard_min_support: int = 3,
    standard_min_error_rate: float = 0.50,
    conservative_min_support: int = 5,
    conservative_min_error_rate: float = 0.60,
    do_no_harm: bool = True,
    inner_validation_size: float = 0.25,
    min_rule_affected: int = 3,
    min_rule_accuracy_gain: float = 0.0,
    stage_confidence_percentile: float = 10.0,
    move_confidence_percentile: float = 10.0,
    isotonic_reliability_threshold: float = 0.50,
    policy_keep_threshold: float = 0.75,
    policy_correction_max_original_reliability: float = 0.60,
    policy_min_rule_support: int = 5,
    policy_min_rule_error_rate: float = 0.60,
    policy_min_rule_validation_gain: float = 0.0,
) -> Tuple[pd.DataFrame, Dict[str, pd.DataFrame]]:
    """Rotate a K-fold split over the human consensus data, pooling
    out-of-fold held-out predictions across all folds.

    Returns:
      summary_df  -- one row per method: pooled n, pooled accuracy, and its
                      bootstrap CI (this is the number that goes in the
                      paper -- computed over ~all annotated items, not just
                      one fold's 1/K share).
      pooled_evaluations -- {method: concatenated per-fold evaluation
                      dataframe}, for further analysis (e.g. feeding into
                      coverage_target_report at full pooled n).
    """
    if not tpc.SKLEARN_AVAILABLE:
        raise RuntimeError("scikit-learn is required for K-fold splitting.")

    from sklearn.model_selection import StratifiedKFold

    base_df = tpc.prepare_auto_df(auto_path)
    base_df, _cm = tpc.compute_confidence_flags(
        base_df, stage_confidence_percentile, move_confidence_percentile
    )

    standard_consensus_df, _sm = tpc.build_consensus(
        annotation_paths, require_same_correction=False, consensus_mode=consensus_mode
    )
    conservative_consensus_df, _cm2 = tpc.build_consensus(
        annotation_paths, require_same_correction=True, consensus_mode=consensus_mode
    )
    base_df, _idc = tpc.configure_auto_id_column(
        base_df, standard_consensus_df["id"].tolist()
    )
    tpc.validate_annotation_ids_against_auto(base_df, standard_consensus_df)

    valid = standard_consensus_df[
        standard_consensus_df["human_timing_correct"].isin(tpc.YES_NO)
    ].copy().reset_index(drop=True)

    if len(valid) < k_folds * 10:
        raise ValueError(
            f"Only {len(valid)} items have valid timing consensus -- too few "
            f"for a reliable {k_folds}-fold split (need at least ~{k_folds*10})."
        )

    # Stratify folds by auto_timing so each fold's held-out portion has a
    # similar label mix, mirroring the stratification used for the single
    # 70/30 split.
    skf = StratifiedKFold(n_splits=k_folds, shuffle=True, random_state=seed)
    fold_assignment = np.zeros(len(valid), dtype=int)
    for fold_index, (_train_idx, test_idx) in enumerate(
        skf.split(valid, valid["auto_timing"])
    ):
        fold_assignment[test_idx] = fold_index
    valid["_fold"] = fold_assignment

    pooled_evaluations: Dict[str, List[pd.DataFrame]] = {m: [] for m in methods}

    for fold_index in range(k_folds):
        fold_development = valid[valid["_fold"] != fold_index].reset_index(drop=True)
        fold_heldout = valid[valid["_fold"] == fold_index].reset_index(drop=True)

        fold_dev_ids = set(fold_development["id"])
        fold_conservative_development = (
            conservative_consensus_df[conservative_consensus_df["id"].isin(fold_dev_ids)]
            .copy()
            .reset_index(drop=True)
        )

        if do_no_harm:
            fold_stage_rules, fold_move_rules, _meta1 = tpc.learn_do_no_harm_recompute_rules(
                development_df=fold_development,
                min_support=standard_min_support,
                min_error_rate=standard_min_error_rate,
                validation_size=inner_validation_size,
                random_state=seed + 100 + fold_index,
                min_affected=min_rule_affected,
                min_accuracy_gain=min_rule_accuracy_gain,
            )
            fold_cons_stage_rules, fold_cons_move_rules, _meta2 = (
                tpc.learn_do_no_harm_recompute_rules(
                    development_df=fold_conservative_development,
                    min_support=conservative_min_support,
                    min_error_rate=conservative_min_error_rate,
                    validation_size=inner_validation_size,
                    random_state=seed + 200 + fold_index,
                    min_affected=min_rule_affected,
                    min_accuracy_gain=min_rule_accuracy_gain,
                )
            )
            fold_timing_rules, _meta3 = tpc.learn_do_no_harm_direct_rules(
                development_df=fold_development,
                min_support=standard_min_support,
                min_error_rate=standard_min_error_rate,
                validation_size=inner_validation_size,
                random_state=seed + 300 + fold_index,
                min_affected=min_rule_affected,
                min_accuracy_gain=min_rule_accuracy_gain,
            )
        else:
            fold_stage_rules = tpc.learn_stage_rules(
                fold_development, standard_min_support, standard_min_error_rate
            )
            fold_move_rules = tpc.learn_conditional_move_rules(
                fold_development, standard_min_support, standard_min_error_rate
            )
            fold_timing_rules = tpc.learn_direct_timing_rules(
                fold_development, standard_min_support, standard_min_error_rate
            )
            fold_cons_stage_rules = tpc.learn_stage_rules(
                fold_conservative_development, conservative_min_support, conservative_min_error_rate
            )
            fold_cons_move_rules = tpc.learn_conditional_move_rules(
                fold_conservative_development, conservative_min_support, conservative_min_error_rate
            )

        fold_dev_auto_rows = tpc.make_heldout_auto_rows(base_df, fold_development)
        fold_dev_with_humans = fold_dev_auto_rows.merge(
            fold_development[
                ["id", "human_stage_correct", "human_move_correct", "human_timing_correct"]
            ].rename(columns={"id": "id_for_calibration"}),
            on="id_for_calibration",
            how="inner",
            validate="one_to_one",
        )

        fold_isotonic_base_df, _iso_meta, _iso_models = tpc.add_isotonic_reliability(
            base_df=base_df,
            development_df=fold_dev_with_humans,
            reliability_threshold=isotonic_reliability_threshold,
        )

        for method in methods:
            if method == "human_rules_direct":
                stage_rules, move_rules, timing_rules = (
                    fold_stage_rules, fold_move_rules, fold_timing_rules
                )
            elif method == "human_rules_recompute":
                stage_rules, move_rules, timing_rules = (
                    fold_stage_rules, fold_move_rules, {}
                )
            elif method in {
                "conservative_human_recompute",
                "conservative_recompute_with_isotonic_reliability",
            }:
                stage_rules, move_rules, timing_rules = (
                    fold_cons_stage_rules, fold_cons_move_rules, {}
                )
            else:
                stage_rules, move_rules, timing_rules = {}, {}, {}

            if method == "safe_keep_correct_review":
                method_output = tpc.apply_safe_keep_correct_review_policy(
                    base_df=fold_isotonic_base_df,
                    timing_rules=fold_timing_rules,
                    keep_threshold=policy_keep_threshold,
                    correction_max_original_reliability=policy_correction_max_original_reliability,
                    min_rule_support=policy_min_rule_support,
                    min_rule_error_rate=policy_min_rule_error_rate,
                    min_rule_validation_gain=policy_min_rule_validation_gain,
                )
            else:
                method_output = tpc.apply_calibration_method(
                    base_df=fold_isotonic_base_df,
                    method=method,
                    stage_rules=stage_rules,
                    move_rules=move_rules,
                    timing_rules=timing_rules,
                )

            _fold_summary, fold_evaluation = tpc.evaluate_method_on_heldout(
                method_output=method_output,
                heldout_df=fold_heldout,
                method=method,
            )
            fold_evaluation["_fold"] = fold_index
            pooled_evaluations[method].append(fold_evaluation)

    summary_rows: List[Dict[str, Any]] = []
    pooled_dfs: Dict[str, pd.DataFrame] = {}

    for method in methods:
        pooled = pd.concat(pooled_evaluations[method], ignore_index=True)
        pooled_dfs[method] = pooled

        boot = bootstrap_ci_from_bool_series(
            pooled["calibrated_timing_correct"],
            n_bootstrap=n_bootstrap,
            alpha=alpha,
            seed=seed,
        )

        summary_rows.append(
            {
                "method": method,
                "k_folds": k_folds,
                "n_pooled": boot["n"],
                "pooled_timing_accuracy": boot["point"],
                "ci_low": boot["ci_low"],
                "ci_high": boot["ci_high"],
                "ci_half_width": round((boot["ci_high"] - boot["ci_low"]) / 2, 4),
            }
        )

    return pd.DataFrame(summary_rows), pooled_dfs


# =============================================================================
# 2.6 Paired bootstrap difference test (the correct test for two methods
# evaluated on the SAME held-out items -- more powerful than eyeballing
# whether two separate marginal CIs overlap)
# =============================================================================

def paired_bootstrap_difference(
    pooled_evaluations: Dict[str, pd.DataFrame],
    method_a: str,
    method_b: str,
    id_column: str = "id_for_calibration",
    correct_column: str = "calibrated_timing_correct",
    n_bootstrap: int = 2000,
    alpha: float = 0.05,
    seed: int = 42,
) -> Dict[str, Any]:
    """Test whether method_b's accuracy differs from method_a's, on the same
    set of items, using a paired percentile bootstrap on the difference.

    Two separate marginal CIs (as in run_kfold_pooled_evaluation's summary)
    can look "barely overlapping" while actually understating how confident
    a paired comparison is, because they ignore that both methods were
    scored on the identical items -- some of that per-item variance cancels
    out in the paired difference and does not need to be double-counted.
    This is the more appropriate test once you have two methods' pooled
    K-fold (or any) evaluation on a shared item set.

    Returns point difference (b - a), its bootstrap CI, and whether that CI
    excludes zero (a simple, standard significance check at the given alpha).
    """
    df_a = pooled_evaluations[method_a][[id_column, correct_column]].rename(
        columns={correct_column: "correct_a"}
    )
    df_b = pooled_evaluations[method_b][[id_column, correct_column]].rename(
        columns={correct_column: "correct_b"}
    )

    merged = df_a.merge(df_b, on=id_column, how="inner", validate="one_to_one")

    if len(merged) == 0:
        raise ValueError(
            f"No shared items between '{method_a}' and '{method_b}' pooled "
            f"evaluations -- check that both were run with the same methods "
            f"list and the same K-fold split."
        )

    a = merged["correct_a"].astype(float).to_numpy()
    b = merged["correct_b"].astype(float).to_numpy()
    n = len(merged)

    point_diff = float(b.mean() - a.mean())

    rng = np.random.default_rng(seed)
    boot_indices = rng.integers(0, n, size=(n_bootstrap, n))
    # Resample ROW indices jointly for both arrays -- this is what makes it
    # a paired bootstrap: each resample keeps (a_i, b_i) pairs intact.
    boot_diffs = b[boot_indices].mean(axis=1) - a[boot_indices].mean(axis=1)

    ci_low, ci_high = np.percentile(
        boot_diffs, [100 * alpha / 2, 100 * (1 - alpha / 2)]
    )

    significant = bool(ci_low > 0 or ci_high < 0)

    return {
        "method_a": method_a,
        "method_b": method_b,
        "n_paired": int(n),
        "accuracy_a": round(float(a.mean()), 4),
        "accuracy_b": round(float(b.mean()), 4),
        "point_difference": round(point_diff, 4),
        "diff_ci_low": round(float(ci_low), 4),
        "diff_ci_high": round(float(ci_high), 4),
        "significant_at_alpha": significant,
        "alpha": alpha,
        "n_bootstrap": n_bootstrap,
        "note": (
            "significant_at_alpha=True means the paired-bootstrap CI on "
            "(accuracy_b - accuracy_a) excludes zero at the given alpha. "
            "This is a standard, defensible significance statement for two "
            "methods scored on the same held-out items."
        ),
    }



def run_multi_seed(
    auto_path: Path,
    annotation_paths: Sequence[Path],
    methods: Sequence[str],
    seeds: Sequence[int],
    n_bootstrap: int,
    alpha: float,
    **pipeline_kwargs: Any,
) -> Tuple[pd.DataFrame, pd.DataFrame]:
    """Repeat the full pipeline under several seeds.

    Returns:
      per_seed_df   -- one row per (method, seed) with held-out accuracy and
                        that seed's own bootstrap CI.
      aggregate_df  -- one row per method, summarizing across-seed mean/std/
                        min/max of held-out timing accuracy, i.e. how much
                        the reported number moves with the train/held-out
                        split itself (as opposed to sampling noise within one
                        held-out set, which the bootstrap CI already covers).
    """
    per_seed_rows: List[Dict[str, Any]] = []

    for seed in seeds:
        run = run_pipeline_once(
            auto_path=auto_path,
            annotation_paths=annotation_paths,
            methods=methods,
            seed=seed,
            **pipeline_kwargs,
        )

        for method in methods:
            summary = run["heldout_summaries"][method]
            evaluation = run["heldout_evaluations"][method]

            boot = bootstrap_ci_for_heldout_evaluation(
                evaluation_df=evaluation,
                n_bootstrap=n_bootstrap,
                alpha=alpha,
                seed=seed,
            )

            row = {
                "seed": seed,
                "method": method,
                "n_heldout": summary["n_heldout"],
                "calibrated_timing_accuracy": summary["calibrated_timing_accuracy"],
                "timing_accuracy_gain": summary["timing_accuracy_gain"],
                "calibrated_stage_accuracy": summary["calibrated_stage_accuracy"],
                "calibrated_move_accuracy": summary["calibrated_move_accuracy"],
                "timing_ci_low": boot.get("timing", {}).get("ci_low"),
                "timing_ci_high": boot.get("timing", {}).get("ci_high"),
            }

            if "policy_accepted_timing_accuracy" in summary:
                row["policy_accepted_coverage"] = summary.get("policy_accepted_coverage")
                row["policy_accepted_timing_accuracy"] = summary.get(
                    "policy_accepted_timing_accuracy"
                )
                row["policy_review_rate"] = summary.get("policy_review_rate")

            per_seed_rows.append(row)

    per_seed_df = pd.DataFrame(per_seed_rows)

    aggregate_rows: List[Dict[str, Any]] = []
    for method, group in per_seed_df.groupby("method"):
        values = group["calibrated_timing_accuracy"].to_numpy(dtype=float)
        aggregate_rows.append(
            {
                "method": method,
                "n_seeds": int(len(group)),
                "mean_timing_accuracy": round(float(np.mean(values)), 4),
                "std_timing_accuracy": round(float(np.std(values, ddof=1)), 4)
                if len(values) > 1
                else 0.0,
                "min_timing_accuracy": round(float(np.min(values)), 4),
                "max_timing_accuracy": round(float(np.max(values)), 4),
                "range_timing_accuracy": round(
                    float(np.max(values) - np.min(values)), 4
                ),
                "mean_timing_accuracy_gain": round(
                    float(group["timing_accuracy_gain"].mean()), 4
                ),
            }
        )

    aggregate_df = pd.DataFrame(aggregate_rows)

    return per_seed_df, aggregate_df


# =============================================================================
# 3.5 Reliability diagnostics and auto-computed threshold grids
# =============================================================================
#
# The isotonic_overall_reliability score is min(stage, move, timing)
# reliability, fit on a modest development set. Its range is data-dependent
# and can sit well below the [0, 1] interval you might naively grid-search
# (e.g. we observed a corpus with p99 = 0.50 -- every "keep_threshold >= 0.5"
# grid point was silently dead). These helpers inspect the *observed*
# distribution and build a threshold grid around it, instead of assuming a
# canonical 0-1 range.

def summarize_reliability_components(
    isotonic_base_df: pd.DataFrame,
    quantiles: Sequence[float] = (0.25, 0.50, 0.75, 0.90, 0.95, 0.99),
) -> pd.DataFrame:
    """Describe each isotonic reliability column (stage/move/timing/overall).

    Run this before choosing --keep-thresholds / --correction-thresholds by
    hand, or just let --auto-thresholds do it for you (see main()).
    """
    columns = [
        "isotonic_stage_reliability",
        "isotonic_move_reliability",
        "isotonic_timing_reliability",
        "isotonic_overall_reliability",
    ]

    rows: List[Dict[str, Any]] = []
    for column in columns:
        if column not in isotonic_base_df.columns:
            continue

        values = isotonic_base_df[column].dropna()
        if values.empty:
            continue

        row: Dict[str, Any] = {
            "component": column,
            "n": int(len(values)),
            "mean": round(float(values.mean()), 4),
            "std": round(float(values.std()), 4),
            "min": round(float(values.min()), 4),
            "max": round(float(values.max()), 4),
        }
        for q in quantiles:
            row[f"p{int(q * 100)}"] = round(float(values.quantile(q)), 4)
        rows.append(row)

    return pd.DataFrame(rows)


def compute_auto_threshold_grid(
    scores: pd.Series,
    n_points: int,
    low_quantile: float,
    high_quantile: float,
) -> List[float]:
    """Build a threshold grid from the OBSERVED quantiles of `scores`,
    instead of an assumed 0-1 range. Deduplicates and sorts ascending.

    Falls back to [median] if the scores are degenerate (empty, constant,
    or too few unique values to span the requested quantile range).
    """
    clean = scores.dropna()

    if clean.empty:
        return [0.5]

    if clean.nunique() <= 1:
        return [round(float(clean.iloc[0]), 4)]

    quantile_points = np.linspace(low_quantile, high_quantile, n_points)
    raw_values = clean.quantile(quantile_points).tolist()
    grid = sorted({round(float(v), 4) for v in raw_values})

    if not grid:
        return [round(float(clean.median()), 4)]

    return grid


def compute_auto_keep_and_correction_grids(
    isotonic_base_df: pd.DataFrame,
    n_keep_points: int = 8,
    n_correction_points: int = 5,
    score_column: str = "isotonic_overall_reliability",
) -> Tuple[List[float], List[float]]:
    """Auto-derive sensible keep/correction threshold grids from the
    observed isotonic_overall_reliability distribution.

    Rationale:
      - KEEP should sweep the upper half of the observed distribution
        (p50 -> p99): these are the candidate "confident enough to accept
        automatically" cutoffs.
      - CORRECTION should sweep the lower half (p5 -> p50): a rule should
        only fire on originally low-reliability cases, by policy design
        (correction_max_original_reliability caps how reliable the ORIGINAL
        label can be and still get corrected).

    Because both grids are derived from the same observed scale, this
    self-adjusts if the reliability distribution shifts across re-annotation
    rounds or dataset versions, instead of silently testing thresholds that
    sit outside the data (as a fixed [0.5 ... 0.9] grid would if p99 is
    itself only 0.5, which is what we observed on the 300-item round).
    """
    scores = isotonic_base_df[score_column]

    keep_grid = compute_auto_threshold_grid(
        scores, n_points=n_keep_points, low_quantile=0.50, high_quantile=0.99
    )
    correction_grid = compute_auto_threshold_grid(
        scores, n_points=n_correction_points, low_quantile=0.05, high_quantile=0.50
    )

    return keep_grid, correction_grid


# =============================================================================
# 3.6 Rank-based coverage targeting (fixes threshold-based coverage failure)
# =============================================================================
#
# The safe_keep_correct_review policy's KEEP decision uses an ABSOLUTE
# reliability threshold. That fails to hit a coverage target reliably when
# the reliability distribution is plateaued (many tied scores) and/or the
# held-out set's score distribution doesn't match the full corpus's -- both
# of which we observed. An absolute cutoff of "reliability >= 0.34" can only
# ever accept whatever fraction of THIS SPECIFIC held-out set happens to
# clear 0.34; it cannot be dialed to "accept exactly 60%".
#
# The fix used here is rank-based coverage (identical in spirit to v1's
# original selective-reliability design, and already implemented in v2 as
# heldout_risk_coverage_table): sort the held-out set by reliability score
# and take the top K% BY COUNT. This hits any target coverage exactly,
# regardless of plateaus, ties, or small-sample mismatch between the
# held-out set and the full corpus.

def coverage_target_report(
    auto_path: Path,
    annotation_paths: Sequence[Path],
    seed: int,
    methods: Sequence[str],
    coverage_points: Sequence[float] = (0.90, 0.80, 0.70, 0.60, 0.50, 0.40, 0.30, 0.20),
    score_column: str = "isotonic_overall_reliability",
    n_bootstrap: int = 2000,
    alpha: float = 0.05,
    consensus_mode: str = "auto",
    heldout_size: float = 0.30,
) -> pd.DataFrame:
    """For each method, rank the held-out evaluation set by `score_column`
    and report accuracy at each target coverage, with a bootstrap CI on the
    accepted subset's accuracy at every point. This is the direct,
    honest answer to "how do I get back to 60% coverage": rank-select the
    top 60% by count on the held-out set, then read off its accuracy --
    rather than searching for an absolute threshold that happens to produce
    60% (which may not exist, as we saw).

    Note this reports what accuracy you'd get by trusting the automatic
    label (no rule-based correction) for the top-K% most reliable cases,
    which is the same "keep-only" selective-prediction framing as v1's
    original selective reliability table. It intentionally does not mix in
    the safe_keep_correct_review policy's separate CORRECT branch -- that
    branch is orthogonal and can still be layered on top of whatever is
    sent to review, if desired.
    """
    run = run_pipeline_once(
        auto_path=auto_path,
        annotation_paths=annotation_paths,
        methods=methods,
        seed=seed,
        consensus_mode=consensus_mode,
        heldout_size=heldout_size,
    )

    rows: List[Dict[str, Any]] = []

    for method in methods:
        evaluation = run["heldout_evaluations"][method]

        if score_column not in evaluation.columns:
            continue

        table = tpc.heldout_risk_coverage_table(
            evaluation_df=evaluation,
            score_column=score_column,
            coverage_points=list(coverage_points),
        )

        ranked = evaluation.sort_values(score_column, ascending=False).reset_index(drop=True)
        total = len(ranked)

        for _, table_row in table.iterrows():
            accepted_count = int(table_row["n_accepted"])
            subset = ranked.iloc[:accepted_count]

            boot = bootstrap_ci_from_bool_series(
                subset["calibrated_timing_correct"],
                n_bootstrap=n_bootstrap,
                alpha=alpha,
                seed=seed,
            )

            rows.append(
                {
                    "method": method,
                    "target_coverage": float(table_row["target_coverage"]),
                    "actual_coverage": float(table_row["actual_coverage"]),
                    "n_accepted": accepted_count,
                    "n_total": total,
                    "score_threshold_at_cutoff": float(table_row["threshold"]),
                    "accepted_timing_accuracy": float(
                        table_row["selective_human_validated_accuracy"]
                    ),
                    "accuracy_ci_low": boot["ci_low"],
                    "accuracy_ci_high": boot["ci_high"],
                    "review_rate": round(1.0 - float(table_row["actual_coverage"]), 4),
                }
            )

    return pd.DataFrame(rows)



def threshold_sensitivity_sweep(
    auto_path: Path,
    annotation_paths: Sequence[Path],
    seed: int,
    keep_thresholds: Optional[Sequence[float]] = None,
    correction_thresholds: Optional[Sequence[float]] = None,
    auto_n_keep_points: int = 8,
    auto_n_correction_points: int = 5,
    consensus_mode: str = "auto",
    heldout_size: float = 0.30,
    standard_min_support: int = 3,
    standard_min_error_rate: float = 0.50,
    inner_validation_size: float = 0.25,
    min_rule_affected: int = 3,
    min_rule_accuracy_gain: float = 0.0,
    stage_confidence_percentile: float = 10.0,
    move_confidence_percentile: float = 10.0,
    isotonic_reliability_threshold: float = 0.50,
    policy_min_rule_support: int = 5,
    policy_min_rule_error_rate: float = 0.60,
    policy_min_rule_validation_gain: float = 0.0,
    verbose: bool = True,
) -> pd.DataFrame:
    """Sweep policy_keep_threshold x policy_correction_max_original_reliability
    for the safe_keep_correct_review policy, at a fixed seed. Rules and the
    dev/held-out split are computed once (they don't depend on the policy
    thresholds); only the KEEP/CORRECT/REVIEW decision is re-applied per grid
    point, so this is cheap even for a fine grid.

    Combinations where correction_threshold > keep_threshold are skipped:
    that configuration would try to "correct" cases already reliable enough
    to keep automatically, which is not a coherent policy.

    If keep_thresholds / correction_thresholds are left as None (the
    default), grids are auto-derived from the OBSERVED
    isotonic_overall_reliability distribution on this corpus via
    compute_auto_keep_and_correction_grids, rather than assuming a canonical
    [0, 1] range. This avoids the failure mode of testing a grid that sits
    entirely above the data (e.g. all "keep_threshold >= 0.5" points being
    dead because the corpus's p99 reliability is itself only 0.5).

    The returned DataFrame carries the grids actually used in
    `.attrs["keep_thresholds_used"]` / `.attrs["correction_thresholds_used"]`
    for logging/reproducibility.
    """
    base_df = tpc.prepare_auto_df(auto_path)
    base_df, _cm = tpc.compute_confidence_flags(
        base_df, stage_confidence_percentile, move_confidence_percentile
    )

    standard_consensus_df, _sm = tpc.build_consensus(
        annotation_paths, require_same_correction=False, consensus_mode=consensus_mode
    )
    base_df, _idc = tpc.configure_auto_id_column(
        base_df, standard_consensus_df["id"].tolist()
    )
    tpc.validate_annotation_ids_against_auto(base_df, standard_consensus_df)

    standard_development, standard_heldout, _split_meta = tpc.split_consensus_data(
        standard_consensus_df, test_size=heldout_size, random_state=seed
    )

    standard_timing_rules, _direct_meta = tpc.learn_do_no_harm_direct_rules(
        development_df=standard_development,
        min_support=standard_min_support,
        min_error_rate=standard_min_error_rate,
        validation_size=inner_validation_size,
        random_state=seed + 300,
        min_affected=min_rule_affected,
        min_accuracy_gain=min_rule_accuracy_gain,
    )

    development_auto_rows = tpc.make_heldout_auto_rows(base_df, standard_development)
    development_with_humans = development_auto_rows.merge(
        standard_development[
            ["id", "human_stage_correct", "human_move_correct", "human_timing_correct"]
        ].rename(columns={"id": "id_for_calibration"}),
        on="id_for_calibration",
        how="inner",
        validate="one_to_one",
    )

    isotonic_base_df, _iso_meta, _iso_models = tpc.add_isotonic_reliability(
        base_df=base_df,
        development_df=development_with_humans,
        reliability_threshold=isotonic_reliability_threshold,
    )

    if verbose:
        diagnostics = summarize_reliability_components(isotonic_base_df)
        print("Observed isotonic reliability distribution (used to build the grid):")
        print(diagnostics.to_string(index=False))
        print()

    auto_keep_grid, auto_correction_grid = compute_auto_keep_and_correction_grids(
        isotonic_base_df=isotonic_base_df,
        n_keep_points=auto_n_keep_points,
        n_correction_points=auto_n_correction_points,
    )

    resolved_keep_thresholds = (
        list(keep_thresholds) if keep_thresholds is not None else auto_keep_grid
    )
    resolved_correction_thresholds = (
        list(correction_thresholds)
        if correction_thresholds is not None
        else auto_correction_grid
    )

    if verbose:
        source = "user-specified" if keep_thresholds is not None else "auto (observed quantiles)"
        print(f"keep_thresholds ({source}): {resolved_keep_thresholds}")
        source = (
            "user-specified" if correction_thresholds is not None else "auto (observed quantiles)"
        )
        print(f"correction_thresholds ({source}): {resolved_correction_thresholds}")
        print()

    rows: List[Dict[str, Any]] = []

    for keep_threshold in resolved_keep_thresholds:
        for correction_threshold in resolved_correction_thresholds:
            if correction_threshold > keep_threshold:
                continue

            method_output = tpc.apply_safe_keep_correct_review_policy(
                base_df=isotonic_base_df,
                timing_rules=standard_timing_rules,
                keep_threshold=keep_threshold,
                correction_max_original_reliability=correction_threshold,
                min_rule_support=policy_min_rule_support,
                min_rule_error_rate=policy_min_rule_error_rate,
                min_rule_validation_gain=policy_min_rule_validation_gain,
            )

            summary, _evaluation = tpc.evaluate_method_on_heldout(
                method_output=method_output,
                heldout_df=standard_heldout,
                method="safe_keep_correct_review",
            )

            rows.append(
                {
                    "keep_threshold": keep_threshold,
                    "correction_threshold": correction_threshold,
                    "n_heldout": summary["n_heldout"],
                    "policy_accepted_coverage": summary.get("policy_accepted_coverage"),
                    "policy_accepted_timing_accuracy": summary.get(
                        "policy_accepted_timing_accuracy"
                    ),
                    "policy_review_rate": summary.get("policy_review_rate"),
                    "policy_n_keep": summary.get("policy_n_keep"),
                    "policy_n_correct": summary.get("policy_n_correct"),
                    "policy_n_review": summary.get("policy_n_review"),
                    "calibrated_timing_accuracy": summary["calibrated_timing_accuracy"],
                }
            )

    result_df = pd.DataFrame(rows)
    result_df.attrs["keep_thresholds_used"] = resolved_keep_thresholds
    result_df.attrs["correction_thresholds_used"] = resolved_correction_thresholds
    result_df.attrs["reliability_diagnostics"] = summarize_reliability_components(
        isotonic_base_df
    ).to_dict("records")

    if result_df.empty and verbose:
        print(
            "WARNING: threshold sweep produced 0 rows -- every "
            "correction_threshold exceeded every keep_threshold in the grid. "
            "This should not happen with auto-computed grids; check that "
            "isotonic_overall_reliability is not entirely NaN or constant."
        )

    return result_df


# =============================================================================
# Main
# =============================================================================

def main() -> None:
    parser = argparse.ArgumentParser(
        description="Robustness analysis (bootstrap CIs, multi-seed stability, "
        "threshold sweep) for TheraTime post-calibration v2."
    )
    parser.add_argument("--auto", required=True, help="Automatic judgments CSV.")
    parser.add_argument(
        "--ann", nargs="+", required=True, help="Two or more annotation CSV/JSON files."
    )
    parser.add_argument("--out-dir", default="theratime_robustness_outputs")
    parser.add_argument(
        "--methods",
        nargs="+",
        default=["baseline", "conservative_human_recompute", "safe_keep_correct_review"],
    )
    parser.add_argument(
        "--seeds",
        nargs="+",
        type=int,
        default=[0, 1, 2, 3, 4, 5, 6, 7, 8, 9],
        help="Random seeds for the multi-seed stability run.",
    )
    parser.add_argument(
        "--primary-seed",
        type=int,
        default=42,
        help="Seed used for the headline bootstrap CI and threshold sweep.",
    )
    parser.add_argument("--n-bootstrap", type=int, default=2000)
    parser.add_argument("--alpha", type=float, default=0.05, help="1 - confidence level.")
    parser.add_argument("--heldout-size", type=float, default=0.30)
    parser.add_argument("--consensus-mode", choices=["auto", "unanimous", "majority"], default="auto")
    parser.add_argument(
        "--keep-thresholds",
        nargs="+",
        type=float,
        default=None,
        help=(
            "Explicit keep-threshold grid. If omitted, a grid is auto-derived "
            "from the observed isotonic_overall_reliability quantiles "
            "(p50 -> p99) on this corpus -- recommended, since the "
            "reliability score's usable range is data-dependent and a fixed "
            "0-1 grid can silently test only dead thresholds."
        ),
    )
    parser.add_argument(
        "--correction-thresholds",
        nargs="+",
        type=float,
        default=None,
        help=(
            "Explicit correction-threshold grid. If omitted, auto-derived "
            "from observed quantiles (p5 -> p50). See --keep-thresholds."
        ),
    )
    parser.add_argument(
        "--n-keep-points",
        type=int,
        default=8,
        help="Number of auto-grid points for keep_threshold when --keep-thresholds is omitted.",
    )
    parser.add_argument(
        "--n-correction-points",
        type=int,
        default=5,
        help="Number of auto-grid points for correction_threshold when --correction-thresholds is omitted.",
    )
    parser.add_argument(
        "--coverage-points",
        nargs="+",
        type=float,
        default=[0.90, 0.80, 0.70, 0.60, 0.50, 0.40, 0.30, 0.20],
        help=(
            "Target coverage fractions for the rank-based coverage report "
            "(e.g. 0.60 = accept the top 60%% most reliable held-out cases "
            "by count). Unlike --keep-thresholds, this always hits the "
            "requested coverage exactly."
        ),
    )
    parser.add_argument(
        "--k-folds",
        type=int,
        default=5,
        help=(
            "Number of folds for the pooled K-fold held-out evaluation. "
            "This tightens the confidence interval by evaluating every "
            "annotated item exactly once (out-of-fold), without requiring "
            "any additional annotation. Set to 0 to skip this step."
        ),
    )
    args = parser.parse_args()

    auto_path = Path(args.auto)
    annotation_paths = [Path(p) for p in args.ann]
    out_dir = Path(args.out_dir)
    out_dir.mkdir(parents=True, exist_ok=True)

    print("=" * 88)
    print("1/6  Headline bootstrap CIs at primary seed", args.primary_seed)
    print("=" * 88)

    primary_run = run_pipeline_once(
        auto_path=auto_path,
        annotation_paths=annotation_paths,
        methods=args.methods,
        seed=args.primary_seed,
        consensus_mode=args.consensus_mode,
        heldout_size=args.heldout_size,
    )

    bootstrap_rows = []
    for method in args.methods:
        evaluation = primary_run["heldout_evaluations"][method]
        boot = bootstrap_ci_for_heldout_evaluation(
            evaluation_df=evaluation,
            n_bootstrap=args.n_bootstrap,
            alpha=args.alpha,
            seed=args.primary_seed,
        )
        for label_type, ci in boot.items():
            bootstrap_rows.append({"method": method, "metric": label_type, **ci})

    bootstrap_df = pd.DataFrame(bootstrap_rows)
    bootstrap_path = out_dir / "theratime_bootstrap_ci.csv"
    bootstrap_df.to_csv(bootstrap_path, index=False)
    print(bootstrap_df.to_string(index=False))

    print()
    print("=" * 88)
    print(f"2/6  Multi-seed stability across {len(args.seeds)} seeds")
    print("=" * 88)

    per_seed_df, aggregate_df = run_multi_seed(
        auto_path=auto_path,
        annotation_paths=annotation_paths,
        methods=args.methods,
        seeds=args.seeds,
        n_bootstrap=args.n_bootstrap,
        alpha=args.alpha,
        consensus_mode=args.consensus_mode,
        heldout_size=args.heldout_size,
    )

    per_seed_path = out_dir / "theratime_multiseed_per_seed.csv"
    aggregate_path = out_dir / "theratime_multiseed_aggregate.csv"
    per_seed_df.to_csv(per_seed_path, index=False)
    aggregate_df.to_csv(aggregate_path, index=False)
    print(aggregate_df.to_string(index=False))

    print()
    print("=" * 88)
    print("3/6  Threshold sensitivity sweep (safe_keep_correct_review)")
    print("=" * 88)

    if "safe_keep_correct_review" in args.methods:
        sweep_df = threshold_sensitivity_sweep(
            auto_path=auto_path,
            annotation_paths=annotation_paths,
            seed=args.primary_seed,
            keep_thresholds=args.keep_thresholds,
            correction_thresholds=args.correction_thresholds,
            auto_n_keep_points=args.n_keep_points,
            auto_n_correction_points=args.n_correction_points,
            consensus_mode=args.consensus_mode,
            heldout_size=args.heldout_size,
            verbose=True,
        )
        sweep_path = out_dir / "theratime_threshold_sweep.csv"
        sweep_df.to_csv(sweep_path, index=False)
        print(sweep_df.to_string(index=False))

        reliability_diagnostics = sweep_df.attrs.get("reliability_diagnostics")
        keep_thresholds_used = sweep_df.attrs.get("keep_thresholds_used")
        correction_thresholds_used = sweep_df.attrs.get("correction_thresholds_used")

        diagnostics_path = out_dir / "theratime_reliability_diagnostics.csv"
        if reliability_diagnostics:
            pd.DataFrame(reliability_diagnostics).to_csv(diagnostics_path, index=False)
    else:
        sweep_df = pd.DataFrame()
        sweep_path = None
        reliability_diagnostics = None
        keep_thresholds_used = None
        correction_thresholds_used = None
        diagnostics_path = None
        print("Skipped: 'safe_keep_correct_review' not in --methods.")

    print()
    print("=" * 88)
    print("4/6  Rank-based coverage report (recovers a target coverage, e.g. 60%)")
    print("=" * 88)

    coverage_methods = [m for m in args.methods if m != "safe_keep_correct_review"] or [
        "baseline"
    ]
    coverage_df = coverage_target_report(
        auto_path=auto_path,
        annotation_paths=annotation_paths,
        seed=args.primary_seed,
        methods=coverage_methods,
        coverage_points=args.coverage_points,
        n_bootstrap=args.n_bootstrap,
        alpha=args.alpha,
        consensus_mode=args.consensus_mode,
        heldout_size=args.heldout_size,
    )
    coverage_path = out_dir / "theratime_coverage_target_report.csv"
    coverage_df.to_csv(coverage_path, index=False)
    print(coverage_df.to_string(index=False))

    print()
    print("=" * 88)
    print(f"5/6  K-fold pooled held-out evaluation (k={args.k_folds}) -- tightens CI, no new annotation")
    print("=" * 88)

    if args.k_folds and args.k_folds >= 2:
        kfold_summary_df, kfold_pooled = run_kfold_pooled_evaluation(
            auto_path=auto_path,
            annotation_paths=annotation_paths,
            methods=args.methods,
            k_folds=args.k_folds,
            seed=args.primary_seed,
            n_bootstrap=args.n_bootstrap,
            alpha=args.alpha,
            consensus_mode=args.consensus_mode,
        )
        kfold_summary_path = out_dir / "theratime_kfold_pooled_summary.csv"
        kfold_summary_df.to_csv(kfold_summary_path, index=False)
        print(kfold_summary_df.to_string(index=False))
        print()
        print(
            "Compare ci_half_width above to the single-split bootstrap CI "
            "half-width in step 1/5 -- pooling across folds uses every "
            "annotated item as a held-out test case exactly once, so this "
            "is normally substantially tighter for the same annotation effort."
        )

        for method, pooled_df in kfold_pooled.items():
            pooled_path = out_dir / f"theratime_kfold_pooled_{method}.csv"
            pooled_df.to_csv(pooled_path, index=False)
    else:
        kfold_summary_df = pd.DataFrame()
        kfold_summary_path = None
        kfold_pooled = {}
        print("Skipped: --k-folds < 2.")

    print()
    print("=" * 88)
    print("6/6  Paired significance test on pooled K-fold results")
    print("=" * 88)

    if kfold_pooled and len(kfold_pooled) >= 2 and "baseline" in kfold_pooled:
        pair_rows = []
        for method in args.methods:
            if method == "baseline" or method not in kfold_pooled:
                continue
            result = paired_bootstrap_difference(
                pooled_evaluations=kfold_pooled,
                method_a="baseline",
                method_b=method,
                n_bootstrap=args.n_bootstrap,
                alpha=args.alpha,
                seed=args.primary_seed,
            )
            pair_rows.append(result)

        pair_df = pd.DataFrame(pair_rows)
        pair_path = out_dir / "theratime_paired_significance.csv"
        if not pair_df.empty:
            pair_df.to_csv(pair_path, index=False)
            print(
                pair_df[
                    [
                        "method_b",
                        "n_paired",
                        "accuracy_a",
                        "accuracy_b",
                        "point_difference",
                        "diff_ci_low",
                        "diff_ci_high",
                        "significant_at_alpha",
                    ]
                ].to_string(index=False)
            )
            print()
            print(
                "significant_at_alpha=True means the paired-bootstrap CI on "
                "(method_b accuracy - baseline accuracy) excludes zero -- a "
                "stronger, more appropriate test than comparing two separate "
                "marginal CIs, since both methods were scored on the same "
                "held-out items."
            )
        else:
            print("No comparable method pairs found.")
    else:
        pair_path = None
        print("Skipped: requires --k-folds >= 2 and 'baseline' plus at least one other method in --methods.")




    report = {
        "purpose": (
            "Robustness analysis layered on theratime_post_calibration.py v2: "
            "bootstrap CIs, multi-seed stability, and threshold sensitivity."
        ),
        "primary_seed": args.primary_seed,
        "seeds_for_stability": args.seeds,
        "n_bootstrap": args.n_bootstrap,
        "alpha": args.alpha,
        "methods": args.methods,
        "output_files": {
            "bootstrap_ci": str(bootstrap_path),
            "multiseed_per_seed": str(per_seed_path),
            "multiseed_aggregate": str(aggregate_path),
            "threshold_sweep": str(sweep_path) if sweep_path else None,
            "reliability_diagnostics": str(diagnostics_path) if diagnostics_path else None,
            "coverage_target_report": str(coverage_path),
            "kfold_pooled_summary": str(kfold_summary_path) if kfold_summary_path else None,
            "paired_significance": str(pair_path) if pair_path else None,
        },
        "threshold_grid": {
            "keep_thresholds_used": keep_thresholds_used,
            "correction_thresholds_used": correction_thresholds_used,
            "grid_source": (
                "user-specified"
                if args.keep_thresholds is not None
                else "auto (observed isotonic_overall_reliability quantiles)"
            ),
        },
        "recommended_paper_wording": (
            "Held-out timing accuracy is reported with 95% percentile "
            "bootstrap confidence intervals (2000 resamples). To assess "
            "sensitivity to the specific development/held-out split, the "
            "full calibration pipeline was repeated under multiple random "
            "seeds; we report the mean and range of held-out timing "
            "accuracy across seeds rather than a single point estimate. "
            "The safe_keep_correct_review policy's keep/correction "
            "thresholds were selected via a sensitivity sweep over a grid "
            "derived from the observed isotonic reliability distribution on "
            "this corpus, rather than an untested fixed default."
        ),
    }
    report_path = out_dir / "theratime_robustness_report.json"
    report_path.write_text(json.dumps(report, indent=2, ensure_ascii=False), encoding="utf-8")

    print()
    print("=" * 88)
    print("Robustness analysis complete.")
    print(f"Output directory: {out_dir}")
    print(f"Report          : {report_path}")
    print("=" * 88)


if __name__ == "__main__":
    main()

Writing theratime_calibration_robustness.py


In [7]:
!python theratime_calibration_robustness.py \
  --auto /kaggle/input/datasets/asmaeassmaebriouya/all-judgments-mpnet/all_judgments_mpnet.csv \
  --ann \
    /kaggle/input/datasets/asmaeassmaebriouya/anotator-recalibrated/theratime_annotation_sample_300_asmae_FINAL_RECALIBRATED.csv \
    /kaggle/input/datasets/asmaeassmaebriouya/anotator-recalibrated/theratime_annotation_sample_300_external_rechecked_with_move_unsure.csv \
    /kaggle/input/datasets/asmaeassmaebriouya/anotator-recalibrated/theratime_annotation_sample_300_hasnae_FINAL_RECALIBRATED.csv \
  --out-dir theratime_robustness_outputs_v3 \
  --methods baseline conservative_human_recompute safe_keep_correct_review \
  --k-folds 5

1/6  Headline bootstrap CIs at primary seed 42
                      method                 metric  point  ci_low  ci_high  n  n_bootstrap  alpha
                    baseline                  stage 0.3333  0.2333   0.4333 90         2000   0.05
                    baseline                   move 0.3444  0.2444   0.4444 90         2000   0.05
                    baseline                 timing 0.5333  0.4333   0.6333 90         2000   0.05
conservative_human_recompute                  stage 0.3333  0.2333   0.4333 90         2000   0.05
conservative_human_recompute                   move 0.3444  0.2444   0.4444 90         2000   0.05
conservative_human_recompute                 timing 0.5333  0.4333   0.6333 90         2000   0.05
    safe_keep_correct_review                  stage 0.3333  0.2333   0.4333 90         2000   0.05
    safe_keep_correct_review                   move 0.3444  0.2444   0.4444 90         2000   0.05
    safe_keep_correct_review                 timing 0.7222  0.

# **theratime_selective_reliability**

In [28]:
%%writefile theratime_selective_reliability.py
"""
theratime_selective_reliability.py
──────────────────────────────────
Selective reliability analysis for TheraTime.

Purpose
-------
This script does not retrain TheraTime.
It analyzes calibrated TheraTime outputs and asks:

  1. Which predictions should we trust?
  2. What TTA@1 do we get if we keep only high-confidence cases?
  3. How much coverage do we keep?
  4. Which cases should be sent to human review?

Inputs
------
Use one calibrated output file from theratime_post_calibration.py.

Recommended input:
  theratime_hybrid_isotonic_conservative.csv

or:
  theratime_conservative_human_recompute.csv

Outputs
-------
  - theratime_selective_risk_coverage.csv
  - theratime_selective_summary.json
  - theratime_selective_review_flags.csv
  - theratime_risk_coverage_curve.png
  - theratime_coverage_tta_curve.png

Kaggle usage
------------
!python theratime_selective_reliability.py \
  --input /kaggle/working/theratime_post_calibration_outputs/theratime_hybrid_isotonic_conservative.csv \
  --out-dir /kaggle/working/theratime_selective_reliability_outputs
"""

import argparse
import json
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt


def as_bool_series(series: pd.Series) -> pd.Series:
    return series.astype(str).str.lower().isin(["true", "1", "yes"])


def safe_numeric(series: pd.Series, default: float = 0.0) -> pd.Series:
    return pd.to_numeric(series, errors="coerce").fillna(default)


def prepare_df(path: Path) -> tuple[pd.DataFrame, str]:
    df = pd.read_csv(path).fillna("")

    if "rank" in df.columns:
        df["rank_for_analysis"] = safe_numeric(df["rank"], -1).astype(int)
    elif "rank_for_calibration" in df.columns:
        df["rank_for_analysis"] = safe_numeric(
            df["rank_for_calibration"], -1
        ).astype(int)
    else:
        df["rank_for_analysis"] = 1

    if "calibrated_is_well_timed" in df.columns:
        df["final_is_well_timed"] = as_bool_series(
            df["calibrated_is_well_timed"]
        )
    elif "is_well_timed" in df.columns:
        df["final_is_well_timed"] = as_bool_series(df["is_well_timed"])
    elif "calibrated_timing" in df.columns:
        df["final_is_well_timed"] = (
            df["calibrated_timing"].astype(str).str.strip().str.lower()
            == "well_timed"
        )
    elif "timing_label" in df.columns:
        df["final_is_well_timed"] = (
            df["timing_label"].astype(str).str.strip().str.lower()
            == "well_timed"
        )
    else:
        raise ValueError(
            "Could not find calibrated_is_well_timed, is_well_timed, "
            "calibrated_timing, or timing_label."
        )

    # Prefer the human-validated correctness column from pooled held-out
    # evaluation files. Fall back to a descriptive well-timed indicator only
    # for legacy output files that do not contain human correctness.
    if "calibrated_timing_correct" in df.columns:
        df["analysis_outcome"] = as_bool_series(
            df["calibrated_timing_correct"]
        )
        outcome_type = "human_validated_timing_accuracy"
    elif "original_timing_correct" in df.columns:
        df["analysis_outcome"] = as_bool_series(
            df["original_timing_correct"]
        )
        outcome_type = "human_validated_original_timing_accuracy"
    else:
        df["analysis_outcome"] = df["final_is_well_timed"].astype(bool)
        outcome_type = "descriptive_well_timed_rate"

    if "stage_confidence" not in df.columns:
        df["stage_confidence"] = 0.0
    if "move_confidence" not in df.columns:
        df["move_confidence"] = 0.0
    if "retrieval_score" not in df.columns:
        df["retrieval_score"] = 0.0

    df["stage_confidence"] = safe_numeric(df["stage_confidence"], 0.0)
    df["move_confidence"] = safe_numeric(df["move_confidence"], 0.0)
    df["retrieval_score"] = safe_numeric(df["retrieval_score"], 0.0)

    df["margin_reliability_score"] = df[
        ["stage_confidence", "move_confidence"]
    ].min(axis=1)

    if "isotonic_overall_reliability" in df.columns:
        df["isotonic_overall_reliability"] = safe_numeric(
            df["isotonic_overall_reliability"], 0.0
        )
    else:
        df["isotonic_overall_reliability"] = np.nan

    valid_iso = (
        df["isotonic_overall_reliability"].notna()
        & (df["isotonic_overall_reliability"] > 0)
    )
    df["hybrid_reliability_score"] = df["margin_reliability_score"]
    df.loc[valid_iso, "hybrid_reliability_score"] = df.loc[
        valid_iso, "isotonic_overall_reliability"
    ]

    margin = df["margin_reliability_score"].to_numpy(dtype=float)
    if len(margin) and np.max(margin) > np.min(margin):
        df["margin_reliability_score_01"] = (
            margin - np.min(margin)
        ) / (np.max(margin) - np.min(margin))
    else:
        df["margin_reliability_score_01"] = 0.5

    return df, outcome_type

def risk_coverage_table(
    df_rank1: pd.DataFrame,
    score_col: str,
    coverage_points=None,
) -> pd.DataFrame:
    if coverage_points is None:
        coverage_points = [1.00, 0.95, 0.90, 0.85, 0.80, 0.75, 0.70, 0.65, 0.60, 0.50, 0.40, 0.30, 0.20, 0.10]

    rows = []
    n = len(df_rank1)

    if n == 0:
        return pd.DataFrame()

    ranked = df_rank1.sort_values(score_col, ascending=False).reset_index(drop=True)

    for cov in coverage_points:
        k = max(1, int(round(cov * n)))
        subset = ranked.iloc[:k]

        outcome = float(subset["analysis_outcome"].mean())
        risk = 1.0 - outcome
        threshold = float(subset[score_col].min())

        rows.append({
            "score_column": score_col,
            "target_coverage": round(cov, 4),
            "actual_coverage": round(k / n, 4),
            "n_accepted": int(k),
            "n_total": int(n),
            "threshold": round(threshold, 6),
            "selective_outcome": round(outcome, 4),
            "selective_risk": round(risk, 4),
        })

    return pd.DataFrame(rows)


def area_under_risk_coverage(rc_df: pd.DataFrame) -> float:
    if rc_df.empty:
        return float("nan")

    d = rc_df.sort_values("actual_coverage")
    x = d["actual_coverage"].values
    y = d["selective_risk"].values
    return float(np.trapezoid(y, x))


def add_review_flags(
    df: pd.DataFrame,
    score_col: str,
    threshold: float,
    crisis_review: bool = True,
) -> pd.DataFrame:
    out = df.copy()

    out["review_reliability_score"] = safe_numeric(out[score_col], 0.0)
    out["review_threshold"] = threshold

    out["review_recommendation"] = np.where(
        out["review_reliability_score"] >= threshold,
        "accept",
        "human_review",
    )

    # Safety-sensitive override.
    if crisis_review:
        stage_col = None
        for candidate in ["calibrated_stage", "predicted_stage", "auto_stage", "auto_stage_for_calibration"]:
            if candidate in out.columns:
                stage_col = candidate
                break

        timing_col = None
        for candidate in ["calibrated_timing", "timing_label", "auto_timing", "auto_timing_for_calibration"]:
            if candidate in out.columns:
                timing_col = candidate
                break

        if stage_col is not None:
            crisis_mask = out[stage_col].astype(str).str.lower().eq("crisis_safety")
            out.loc[crisis_mask, "review_recommendation"] = "high_risk_review"

        if timing_col is not None:
            safety_mask = out[timing_col].astype(str).str.lower().eq("delayed_safety")
            out.loc[safety_mask, "review_recommendation"] = "high_risk_review"

    return out


def plot_risk_coverage(rc_df: pd.DataFrame, out_dir: Path):
    if rc_df.empty:
        return

    plt.figure(figsize=(8, 5))
    for score_col, group in rc_df.groupby("score_column"):
        group = group.sort_values("actual_coverage")
        plt.plot(group["actual_coverage"], group["selective_risk"], marker="o", label=score_col)

    plt.xlabel("Coverage")
    plt.ylabel("Selective risk, lower is better")
    plt.title("TheraTime selective risk-coverage curve")
    plt.grid(True, alpha=0.3)
    plt.legend()
    plt.tight_layout()
    path = out_dir / "theratime_risk_coverage_curve.png"
    plt.savefig(path, dpi=200)
    plt.close()


def plot_coverage_tta(rc_df: pd.DataFrame, out_dir: Path):
    if rc_df.empty:
        return

    plt.figure(figsize=(8, 5))
    for score_col, group in rc_df.groupby("score_column"):
        group = group.sort_values("actual_coverage")
        plt.plot(group["actual_coverage"], group["selective_outcome"], marker="o", label=score_col)

    plt.xlabel("Coverage")
    plt.ylabel("Selective outcome, higher is better")
    plt.title("TheraTime selective outcome by coverage")
    plt.grid(True, alpha=0.3)
    plt.legend()
    plt.tight_layout()
    path = out_dir / "theratime_coverage_tta_curve.png"
    plt.savefig(path, dpi=200)
    plt.close()


def main():
    parser = argparse.ArgumentParser(
        description="Selective reliability analysis for calibrated TheraTime outputs."
    )
    parser.add_argument("--input", required=True, help="Calibrated TheraTime CSV file.")
    parser.add_argument("--out-dir", default="theratime_selective_reliability_outputs")
    parser.add_argument(
        "--preferred-coverage",
        type=float,
        default=0.80,
        help="Coverage level used to choose the human-review threshold.",
    )
    parser.add_argument(
        "--score",
        default="margin_reliability_score",
        choices=[
            "margin_reliability_score",
            "margin_reliability_score_01",
            "hybrid_reliability_score",
            "isotonic_overall_reliability",
        ],
        help="Reliability score used for the final review flag.",
    )
    args = parser.parse_args()

    input_path = Path(args.input)
    out_dir = Path(args.out_dir)
    out_dir.mkdir(parents=True, exist_ok=True)

    df, outcome_type = prepare_df(input_path)
    rank1 = df[df["rank_for_analysis"] == 1].copy()

    if len(rank1) == 0:
        raise ValueError("No rank-1 rows found for selective reliability analysis.")

    score_columns = [
        "margin_reliability_score",
        "margin_reliability_score_01",
        "hybrid_reliability_score",
    ]

    # Only include isotonic if it actually has useful non-zero values.
    if "isotonic_overall_reliability" in rank1.columns:
        if rank1["isotonic_overall_reliability"].notna().sum() > 0:
            score_columns.append("isotonic_overall_reliability")

    tables = []
    for score_col in score_columns:
        if score_col not in rank1.columns:
            continue
        rc = risk_coverage_table(rank1, score_col)
        if not rc.empty:
            rc["AURC"] = round(area_under_risk_coverage(rc), 6)
            tables.append(rc)

    if tables:
        rc_all = pd.concat(tables, ignore_index=True)
    else:
        rc_all = pd.DataFrame()

    rc_path = out_dir / "theratime_selective_risk_coverage.csv"
    rc_all.to_csv(rc_path, index=False)

    # Select threshold for review flags at preferred coverage.
    selected_score = args.score
    if selected_score not in rank1.columns:
        selected_score = "margin_reliability_score"

    selected_rc = risk_coverage_table(rank1, selected_score, coverage_points=[args.preferred_coverage])
    if selected_rc.empty:
        threshold = float(rank1[selected_score].median())
        selected_tta = float(rank1["final_is_well_timed"].mean())
        selected_coverage = 1.0
    else:
        threshold = float(selected_rc.iloc[0]["threshold"])
        selected_tta = float(selected_rc.iloc[0]["selective_outcome"])
        selected_coverage = float(selected_rc.iloc[0]["actual_coverage"])

    flagged = add_review_flags(df, selected_score, threshold)
    flags_path = out_dir / "theratime_selective_review_flags.csv"
    flagged.to_csv(flags_path, index=False)

    rank1_flagged = flagged[flagged["rank_for_analysis"] == 1].copy()
    review_counts = rank1_flagged["review_recommendation"].value_counts().to_dict()

    n_rank1 = int(len(rank1_flagged))
    n_accept = int(review_counts.get("accept", 0))
    n_human_review = int(review_counts.get("human_review", 0))
    n_high_risk_review = int(review_counts.get("high_risk_review", 0))

    final_acceptance_coverage = (
        float(n_accept / n_rank1) if n_rank1 else 0.0
    )
    final_review_rate = (
        float((n_human_review + n_high_risk_review) / n_rank1)
        if n_rank1 else 0.0
    )

    baseline_tta = float(rank1["analysis_outcome"].mean())

    summary = {
        "input_file": str(input_path),
        "n_rows": int(len(df)),
        "n_rank1": int(len(rank1)),
        "outcome_type": outcome_type,
        "all_rank1_outcome": round(baseline_tta, 4),
        "preferred_coverage": args.preferred_coverage,
        "selected_score": selected_score,
        "selected_threshold": round(threshold, 6),
        "selective_outcome_at_preferred_coverage": round(selected_tta, 4),
        "nominal_rank_selected_coverage": round(selected_coverage, 4),
        "final_acceptance_coverage_after_safety_override": round(
            final_acceptance_coverage, 4
        ),
        "final_review_rate_after_safety_override": round(
            final_review_rate, 4
        ),
        "selective_gain_vs_all_rank1": round(selected_tta - baseline_tta, 4),
        "review_counts_rank1": review_counts,
        "risk_coverage_csv": str(rc_path),
        "review_flags_csv": str(flags_path),
        "paper_safe_interpretation": (
            "Selective reliability analysis evaluates when TheraTime judgments are more trustworthy. "
            "The nominal rank-selected coverage is reported separately from final acceptance coverage "
            "after safety-sensitive overrides. Low-reliability or safety-sensitive cases are sent to "
            "human review."
        ),
    }

    summary_path = out_dir / "theratime_selective_summary.json"
    summary_path.write_text(json.dumps(summary, indent=2), encoding="utf-8")

    plot_risk_coverage(rc_all, out_dir)
    plot_coverage_tta(rc_all, out_dir)

    print("=" * 80)
    print("TheraTime selective reliability analysis complete")
    print("=" * 80)
    print(f"Input file                   : {input_path}")
    print(f"Output directory             : {out_dir}")
    print(f"Risk-coverage CSV            : {rc_path}")
    print(f"Review flags CSV             : {flags_path}")
    print(f"Summary JSON                 : {summary_path}")
    print()
    print(f"All rank-1 TTA@1             : {baseline_tta:.4f}")
    print(f"Selected reliability score   : {selected_score}")
    print(f"Preferred coverage           : {args.preferred_coverage:.2f}")
    print(f"Selected threshold           : {threshold:.6f}")
    print(f"Selective TTA@1              : {selected_tta:.4f}")
    print(f"Selective gain               : {selected_tta - baseline_tta:.4f}")
    print(f"Nominal rank-selected coverage: {selected_coverage:.4f}")
    print(f"Final acceptance coverage     : {final_acceptance_coverage:.4f}")
    print(f"Final review rate             : {final_review_rate:.4f}")
    print()
    print("Rank-1 review counts:")
    for key, value in review_counts.items():
        print(f"  {key:18s}: {value}")
    print("=" * 80)


if __name__ == "__main__":
    main()

Overwriting theratime_selective_reliability.py


In [29]:
!python theratime_selective_reliability.py \
  --input /kaggle/working/theratime_robustness_outputs_v3/theratime_kfold_pooled_safe_keep_correct_review.csv \
  --out-dir /kaggle/working/theratime_selective_reliability_outputs \
  --score isotonic_overall_reliability

TheraTime selective reliability analysis complete
Input file                   : /kaggle/working/theratime_robustness_outputs_v3/theratime_kfold_pooled_safe_keep_correct_review.csv
Output directory             : /kaggle/working/theratime_selective_reliability_outputs
Risk-coverage CSV            : /kaggle/working/theratime_selective_reliability_outputs/theratime_selective_risk_coverage.csv
Review flags CSV             : /kaggle/working/theratime_selective_reliability_outputs/theratime_selective_review_flags.csv
Summary JSON                 : /kaggle/working/theratime_selective_reliability_outputs/theratime_selective_summary.json

All rank-1 TTA@1             : 0.6167
Selected reliability score   : isotonic_overall_reliability
Preferred coverage           : 0.80
Selected threshold           : 0.259259
Selective TTA@1              : 0.6333
Selective gain               : 0.0166
Nominal rank-selected coverage: 0.8000
Final acceptance coverage     : 0.7900
Final review rate             : 0.

In [30]:
!python theratime_selective_reliability.py \
  --input /kaggle/working/theratime_robustness_outputs_v3/theratime_kfold_pooled_safe_keep_correct_review.csv \
  --out-dir /kaggle/working/theratime_selective_reliability_outputs_conservative \
  --preferred-coverage 0.80 \
  --score margin_reliability_score

TheraTime selective reliability analysis complete
Input file                   : /kaggle/working/theratime_robustness_outputs_v3/theratime_kfold_pooled_safe_keep_correct_review.csv
Output directory             : /kaggle/working/theratime_selective_reliability_outputs_conservative
Risk-coverage CSV            : /kaggle/working/theratime_selective_reliability_outputs_conservative/theratime_selective_risk_coverage.csv
Review flags CSV             : /kaggle/working/theratime_selective_reliability_outputs_conservative/theratime_selective_review_flags.csv
Summary JSON                 : /kaggle/working/theratime_selective_reliability_outputs_conservative/theratime_selective_summary.json

All rank-1 TTA@1             : 0.6167
Selected reliability score   : margin_reliability_score
Preferred coverage           : 0.80
Selected threshold           : 0.004200
Selective TTA@1              : 0.6292
Selective gain               : 0.0125
Nominal rank-selected coverage: 0.8000
Final acceptance coverage

# **theratime_error_recall**

In [41]:
%%writefile theratime_error_recall.py
"""
theratime_error_recall.py  v2.2
────────────────────────────────
Compute TheraTime automatic timing reliability from human annotations.

Answers four questions for the paper:

  Q1. SUBSET RECALL (original metric, now correctly scoped)
      Among cases where both annotators rejected the auto timing label
      AND both provided corrections, what fraction was still genuinely
      mistimed (possibly with a different label)?
      → Answers: does the auto system catch real errors even when wrong?

  Q2. CORRECTION DIRECTION (full-150 picture)
      Across all human corrections, what is the distribution of
      correction targets (well_timed vs which error types)?
      → Answers: which direction does the auto system err?

  Q3. AUTO PRECISION (most important for the paper)
      Of cases auto labeled as mistimed, what fraction did both
      annotators confirm as genuinely mistimed?
      → Answers: how trustworthy is an auto timing flag?

  Q4. PER-ERROR-TYPE BREAKDOWN
      For each auto error category, how often did humans agree it
      was genuinely that error type vs. corrected to well_timed?
      → Answers: which error types are most reliable?

Usage:
    python theratime_error_recall.py \\
        --disagreements theratime_disagreements.csv \\
        --annotations1  theratime_annotations_annotator1.csv \\
        --annotations2  theratime_annotations_annotator2.csv \\
        --out           theratime_recall_report.csv

    The --annotations1 and --annotations2 flags are optional.
    If provided, they enable Q2 and Q3 (full-150 analysis).
    If only --disagreements is provided, only Q1 is computed.

Output:
    - Console report with paper-ready sentences
    - CSV with per-case detail
    - theratime_recall_summary.json with all key numbers
"""

import argparse
import json
import sys
import csv
from pathlib import Path
from collections import Counter

import pandas as pd


# ── Label sets ────────────────────────────────────────────────────────────────

MISTIMED_LABELS = {
    "premature_advice",
    "delayed_safety",
    "over_validation",
    "missing_clarification",
    "stage_mismatch",
}

WELL_TIMED_LABELS = {"well_timed"}

ALL_TIMING_LABELS = MISTIMED_LABELS | WELL_TIMED_LABELS


# ── Utilities ─────────────────────────────────────────────────────────────────

def norm(x: str) -> str:
    return str(x or "").strip().lower().replace("-", "_").replace(" ", "_")


def find_col(df: pd.DataFrame, must_contain: list, must_exclude: list = None) -> str:
    """
    Return the first column whose lowercase name contains all strings in
    must_contain and none of the strings in must_exclude.
    """
    must_exclude = must_exclude or []
    for col in df.columns:
        low = col.lower()
        if (all(k.lower() in low for k in must_contain)
                and not any(k.lower() in low for k in must_exclude)):
            return col
    return None



def drop_duplicate_columns(df: pd.DataFrame) -> pd.DataFrame:
    """
    Remove duplicate column names while preserving the first occurrence.

    Some CSV exports contain repeated auto_timing or correction columns.
    In pandas, selecting a duplicated name returns a DataFrame rather than
    a Series, which breaks .map() operations.
    """
    return df.loc[:, ~df.columns.duplicated()].copy()


def resolve_column(
    df: pd.DataFrame,
    exact_names: list[str],
    must_contain: list[str] | None = None,
    must_exclude: list[str] | None = None,
) -> str | None:
    """
    Resolve a column using exact-name priority, then a conservative fallback.
    Matching is case-insensitive.
    """
    lookup = {str(col).strip().lower(): col for col in df.columns}

    for name in exact_names:
        match = lookup.get(name.strip().lower())
        if match is not None:
            return match

    if must_contain:
        return find_col(
            df,
            must_contain=must_contain,
            must_exclude=must_exclude or [],
        )

    return None


def resolve_annotation_columns(df: pd.DataFrame) -> dict:
    """
    Resolve the canonical columns required for Q2/Q3 from a full annotation
    file. Exact canonical names are preferred.
    """
    return {
        "id": resolve_column(
            df,
            ["id", "query_id", "example_id"],
            must_contain=["id"],
        ),
        "timing_correct": resolve_column(
            df,
            [
                "timing_correct",
                "timing correctness",
                "timing_correctness",
                "is_timing_correct",
            ],
            must_contain=["timing", "correct"],
            must_exclude=["correction", "notes", "auto", "predicted"],
        ),
        "timing_correction": resolve_column(
            df,
            [
                "timing_correction",
                "corrected_timing",
                "timing correction",
                "human_timing_correction",
            ],
            must_contain=["timing", "correction"],
            must_exclude=["notes"],
        ),
        "auto_timing": resolve_column(
            df,
            [
                "auto_timing",
                "timing_label",
                "predicted_timing",
                "automatic_timing",
            ],
            must_contain=["timing"],
            must_exclude=[
                "correct",
                "correction",
                "notes",
                "human",
                "annotator",
            ],
        ),
    }


def unique_columns(columns: list[str | None]) -> list[str]:
    """Return non-empty column names once, preserving order."""
    result: list[str] = []
    for column in columns:
        if column and column not in result:
            result.append(column)
    return result


def is_mistimed(label: str) -> bool | None:
    """Return True if label is a timing error, False if well_timed, None if unknown."""
    l = norm(label)
    if l in MISTIMED_LABELS:
        return True
    if l in WELL_TIMED_LABELS:
        return False
    return None


def detect_annotator_prefixes(df: pd.DataFrame) -> tuple[str, str]:
    """
    Detect the two annotator prefixes in a disagreements CSV generated by
    theratime_kappa.py.

    Expected columns include:
      <annotator>_timing
      <annotator>_timing_correction

    Generic columns such as auto_timing and timing_disagree are ignored.
    """
    suffix = "_timing"
    excluded = {
        "auto",
        "predicted",
        "human",
        "stage",
        "move",
        "timing",
    }

    prefixes = []
    for col in df.columns:
        low = col.lower()
        if not low.endswith(suffix):
            continue
        if low in {"auto_timing", "predicted_timing"}:
            continue

        prefix = col[:-len(suffix)]
        if not prefix or prefix.lower() in excluded:
            continue

        correction_col = f"{prefix}_timing_correction"
        if correction_col in df.columns:
            prefixes.append(prefix)

    # Preserve column order while removing duplicates.
    unique = []
    for prefix in prefixes:
        if prefix not in unique:
            unique.append(prefix)

    if len(unique) != 2:
        raise ValueError(
            "Could not uniquely detect two annotator prefixes from the "
            f"disagreements CSV. Detected: {unique}. "
            "Expected columns like '<annotator>_timing' and "
            "'<annotator>_timing_correction'."
        )

    return unique[0], unique[1]


# ── Q1: Subset recall (disagreements CSV only) ────────────────────────────────

def compute_subset_recall(df: pd.DataFrame,
                           h_timing_col: str,
                           a_timing_col: str,
                           h_corr_col: str,
                           a_corr_col: str) -> dict:
    """
    Compute recall on the subset where both annotators said timing was wrong.

    SCOPE: This metric applies only to cases in theratime_disagreements.csv
    (cases with at least one stage or move disagreement). It is NOT the
    system's overall precision or recall. Report with explicit scope.
    """
    df = df.copy()
    df["_h_timing"] = df[h_timing_col].map(norm)
    df["_a_timing"] = df[a_timing_col].map(norm)
    df["_h_corr"]   = df[h_corr_col].map(norm)
    df["_a_corr"]   = df[a_corr_col].map(norm)

    both_wrong = df[(df["_h_timing"] == "no") & (df["_a_timing"] == "no")].copy()

    rows = []
    for _, row in both_wrong.iterrows():
        h = row["_h_corr"]
        a = row["_a_corr"]
        corrections = [x for x in [h, a] if x and x in ALL_TIMING_LABELS]

        h_mt = is_mistimed(h)
        a_mt = is_mistimed(a)

        if not corrections:
            status = "unknown_no_correction"
            consensus_mt = None
        elif h == a and h in ALL_TIMING_LABELS:
            status = ("mistimed_same_label" if h_mt
                      else "well_timed_same_label")
            consensus_mt = h_mt
        elif h_mt is True and a_mt is True:
            status = "mistimed_different_label"
            consensus_mt = True
        elif h_mt is False and a_mt is False:
            status = "well_timed_different_label"
            consensus_mt = False
        elif h_mt is not None and a_mt is not None and h_mt != a_mt:
            status = "human_disagree_mistimed_vs_well_timed"
            consensus_mt = None
        else:
            status = "unknown_or_invalid_correction"
            consensus_mt = None

        rows.append({
            "id":               row.get("id", ""),
            "auto_timing":      norm(row.get("auto_timing", row.get(
                                    "timing_label", row.get("predicted_timing","")))),
            "annotator_1_correction": h,
            "annotator_2_correction": a,
            "human_status":      status,
            "consensus_mistimed": consensus_mt,
        })

    result = pd.DataFrame(rows)
    valid  = result[result["consensus_mistimed"].isin([True, False])]
    n_both_wrong = len(both_wrong)
    n_valid      = len(valid)
    n_mistimed   = int((valid["consensus_mistimed"] == True).sum())
    n_well_timed = int((valid["consensus_mistimed"] == False).sum())
    recall = n_mistimed / n_valid if n_valid else None

    return {
        "n_both_said_wrong":   n_both_wrong,
        "n_valid_corrections": n_valid,
        "n_actually_mistimed": n_mistimed,
        "n_actually_well_timed": n_well_timed,
        "subset_recall":       round(recall, 4) if recall is not None else None,
        "status_breakdown":    result["human_status"].value_counts().to_dict(),
        "detail_df":           result,
    }


# ── Q2: Correction direction (full annotation files) ─────────────────────────

def compute_correction_direction(
    ann1: pd.DataFrame,
    ann2: pd.DataFrame,
) -> dict:
    """
    Across all human timing corrections where an annotator rejected the
    automatic timing label, report the correction-target distribution.

    This is a pairwise annotation diagnostic. It counts annotator-level
    corrections, not three-annotator majority-consensus corrections.
    """
    ann1 = drop_duplicate_columns(ann1)
    ann2 = drop_duplicate_columns(ann2)

    corrections = []
    diagnostics = []

    for df, annotator in [
        (ann1, "annotator1"),
        (ann2, "annotator2"),
    ]:
        columns = resolve_annotation_columns(df)
        tc_col = columns["timing_correct"]
        corr_col = columns["timing_correction"]

        diagnostics.append(
            {
                "annotator": annotator,
                "timing_correct_column": tc_col,
                "timing_correction_column": corr_col,
            }
        )

        if tc_col is None or corr_col is None:
            continue

        wrong = df[df[tc_col].map(norm) == "no"]
        for _, row in wrong.iterrows():
            correction = norm(row[corr_col])
            if correction in ALL_TIMING_LABELS:
                corrections.append(
                    {
                        "annotator": annotator,
                        "correction": correction,
                    }
                )

    if not corrections:
        return {
            "error": (
                "Could not find usable timing-correction rows in the full "
                "annotation files."
            ),
            "column_diagnostics": diagnostics,
        }

    direction = Counter(row["correction"] for row in corrections)
    total = sum(direction.values())

    return {
        "total_corrections": int(total),
        "well_timed_count": int(direction.get("well_timed", 0)),
        "well_timed_pct": (
            round(direction.get("well_timed", 0) / total, 4)
            if total
            else 0.0
        ),
        "mistimed_corrections": {
            key: int(value)
            for key, value in direction.items()
            if key in MISTIMED_LABELS
        },
        "full_breakdown": {
            key: int(value)
            for key, value in direction.most_common()
        },
        "column_diagnostics": diagnostics,
    }


# ── Q3: Auto precision (needs full annotation + auto labels) ──────────────────

def compute_auto_precision(
    ann1: pd.DataFrame,
    ann2: pd.DataFrame,
) -> dict:
    """
    Among cases automatically labelled as mistimed, estimate strict pairwise
    precision: both annotators must confirm that the automatic timing label
    is correct.

    Ambiguous or incomplete pairwise judgments remain in the denominator,
    making this a conservative pairwise estimate.
    """
    ann1 = drop_duplicate_columns(ann1)
    ann2 = drop_duplicate_columns(ann2)

    cols1 = resolve_annotation_columns(ann1)
    cols2 = resolve_annotation_columns(ann2)

    id1 = cols1["id"]
    id2 = cols2["id"]
    tc1 = cols1["timing_correct"]
    tc2 = cols2["timing_correct"]
    corr1 = cols1["timing_correction"]
    corr2 = cols2["timing_correction"]
    auto1 = cols1["auto_timing"]
    auto2 = cols2["auto_timing"]

    required = {
        "annotator1_id": id1,
        "annotator2_id": id2,
        "annotator1_timing_correct": tc1,
        "annotator2_timing_correct": tc2,
        "annotator1_timing_correction": corr1,
        "annotator2_timing_correction": corr2,
    }

    missing = [name for name, value in required.items() if value is None]
    if missing:
        return {
            "error": f"Missing required columns: {missing}",
            "annotator1_columns": cols1,
            "annotator2_columns": cols2,
        }

    if auto1 is None and auto2 is None:
        return {
            "error": "Could not locate an automatic timing-label column.",
            "annotator1_columns": cols1,
            "annotator2_columns": cols2,
        }

    left_columns = unique_columns([id1, tc1, corr1, auto1])
    right_columns = unique_columns([id2, tc2, corr2, auto2])

    left = ann1[left_columns].copy()
    right = ann2[right_columns].copy()

    left_rename = {
        id1: "id",
        tc1: "h1_tc",
        corr1: "h1_corr",
    }
    if auto1 is not None:
        left_rename[auto1] = "auto_timing"

    right_rename = {
        id2: "id",
        tc2: "h2_tc",
        corr2: "h2_corr",
    }
    if auto2 is not None:
        right_rename[auto2] = "auto_timing_from_annotator2"

    left = left.rename(columns=left_rename)
    right = right.rename(columns=right_rename)

    merged = pd.merge(
        left,
        right,
        on="id",
        how="inner",
        validate="one_to_one",
    )

    if "auto_timing" not in merged.columns:
        merged["auto_timing"] = merged["auto_timing_from_annotator2"]
    elif "auto_timing_from_annotator2" in merged.columns:
        left_auto = merged["auto_timing"].map(norm)
        right_auto = merged["auto_timing_from_annotator2"].map(norm)

        mismatch = (
            left_auto.ne("")
            & right_auto.ne("")
            & left_auto.ne(right_auto)
        )
        if mismatch.any():
            examples = merged.loc[
                mismatch,
                ["id", "auto_timing", "auto_timing_from_annotator2"],
            ].head(10)
            return {
                "error": (
                    "Automatic timing labels disagree between the two full "
                    "annotation files."
                ),
                "n_auto_timing_mismatches": int(mismatch.sum()),
                "examples": examples.to_dict("records"),
            }

        merged["auto_timing"] = merged["auto_timing"].where(
            left_auto.ne(""),
            merged["auto_timing_from_annotator2"],
        )

    # At this point auto_timing is guaranteed to be one Series.
    merged["auto_mt"] = merged["auto_timing"].map(norm).map(is_mistimed)
    merged["h1_tc_n"] = merged["h1_tc"].map(norm)
    merged["h2_tc_n"] = merged["h2_tc"].map(norm)
    merged["h1_corr_n"] = merged["h1_corr"].map(norm)
    merged["h2_corr_n"] = merged["h2_corr"].map(norm)

    auto_mistimed = merged[merged["auto_mt"] == True].copy()
    n_auto_mistimed = int(len(auto_mistimed))

    if n_auto_mistimed == 0:
        return {
            "error": "No automatically mistimed cases were found.",
            "n_merged": int(len(merged)),
        }

    both_confirmed_mask = (
        (auto_mistimed["h1_tc_n"] == "yes")
        & (auto_mistimed["h2_tc_n"] == "yes")
    )

    both_rejected_to_wt_mask = (
        (auto_mistimed["h1_tc_n"] == "no")
        & (auto_mistimed["h2_tc_n"] == "no")
        & (auto_mistimed["h1_corr_n"] == "well_timed")
        & (auto_mistimed["h2_corr_n"] == "well_timed")
    )

    n_both_confirmed = int(both_confirmed_mask.sum())
    n_both_rejected_to_wt = int(both_rejected_to_wt_mask.sum())
    n_ambiguous = (
        n_auto_mistimed
        - n_both_confirmed
        - n_both_rejected_to_wt
    )

    precision = n_both_confirmed / n_auto_mistimed

    return {
        "n_merged": int(len(merged)),
        "n_auto_mistimed": n_auto_mistimed,
        "n_both_confirmed_tp": n_both_confirmed,
        "n_both_rejected_to_wt_fp": n_both_rejected_to_wt,
        "n_ambiguous": int(n_ambiguous),
        "precision": round(float(precision), 4),
        "precision_pct": round(float(precision * 100), 1),
        "annotator1_columns": cols1,
        "annotator2_columns": cols2,
        "scope_note": (
            "Strict pairwise precision: both annotators must confirm the "
            "automatic mistimed label. Ambiguous cases remain in the "
            "denominator."
        ),
    }


# ── Q4: Per-error-type breakdown ──────────────────────────────────────────────

def compute_per_error_breakdown(disagreements: pd.DataFrame,
                                 h_timing_col: str,
                                 a_timing_col: str,
                                 h_corr_col: str,
                                 a_corr_col: str,
                                 auto_col: str) -> pd.DataFrame:
    """
    For each auto error type, compute:
    - How many cases had that auto label
    - How many both annotators confirmed correct
    - How many were corrected to well_timed
    - How many were corrected to a different error type

    Clinical note: delayed_safety false positives are more concerning
    than stage_mismatch false positives. This breakdown shows which
    error types are most and least reliable.
    """
    df = disagreements.copy()
    df["_auto"]   = df[auto_col].map(norm) if auto_col in df.columns else ""
    df["_h_tc"]   = df[h_timing_col].map(norm)
    df["_a_tc"]   = df[a_timing_col].map(norm)
    df["_h_corr"] = df[h_corr_col].map(norm)
    df["_a_corr"] = df[a_corr_col].map(norm)

    rows = []
    for error_type in sorted(MISTIMED_LABELS):
        subset = df[df["_auto"] == error_type]
        if len(subset) == 0:
            continue
        n_total = len(subset)
        n_both_correct = ((subset["_h_tc"] == "yes") & (subset["_a_tc"] == "yes")).sum()
        n_both_to_wt   = (
            (subset["_h_tc"] == "no") & (subset["_a_tc"] == "no") &
            (subset["_h_corr"] == "well_timed") & (subset["_a_corr"] == "well_timed")
        ).sum()
        rows.append({
            "auto_error_type":          error_type,
            "n_in_disagreements":       n_total,
            "n_both_confirmed_correct": int(n_both_correct),
            "n_both_corrected_to_wt":   int(n_both_to_wt),
            "n_other":                  n_total - int(n_both_correct) - int(n_both_to_wt),
            "confirmation_rate":        round(int(n_both_correct)/n_total, 3),
            "fp_rate_to_wt":            round(int(n_both_to_wt)/n_total, 3),
            "clinical_note":            (
                "HIGHEST RISK — false positives hide safety failures"
                if error_type == "delayed_safety"
                else "")
        })
    return pd.DataFrame(rows)


# ── Paper reporting template ──────────────────────────────────────────────────

def paper_template(q1: dict, q2: dict, q3: dict) -> str:
    r1  = q1.get("subset_recall")
    nm  = q1.get("n_actually_mistimed", 0)
    nv  = q1.get("n_valid_corrections", 0)
    nwt = q1.get("n_actually_well_timed", 0)

    q2_available = bool(q2) and "error" not in q2 and q2.get("total_corrections", 0) > 0
    wt_count = q2.get("well_timed_count", 0)
    wt_pct   = q2.get("well_timed_pct", 0)
    total_c  = q2.get("total_corrections", 0)

    prec     = q3.get("precision_pct")
    n_tp     = q3.get("n_both_confirmed_tp")
    n_auto_m = q3.get("n_auto_mistimed")

    lines = []
    lines.append("── PAPER REPORTING TEMPLATE ─────────────────────────────────────────")
    lines.append("")
    lines.append("Section 3.3 paragraph (paste directly):")
    lines.append("")
    lines.append(
        "Analysis of correction patterns provides further diagnostic evidence."
    )
    if q2_available:
        lines.append(
            f" Across all {total_c} annotator-level timing corrections made when"
            f" annotators judged the automatic label incorrect, {wt_count}"
            f" ({wt_pct*100:.1f}\\%) were directed toward"
            f" \\texttt{{well\\_timed}}."
        )
    if prec is not None:
        lines.append(
            f" Among the {n_auto_m} rank-1 cases where the automatic system predicted"
            f" a timing error, both annotators confirmed the prediction correct in"
            f" {n_tp} cases ({prec:.1f}\\%), giving an estimated automatic timing"
            f" precision of {prec:.1f}\\%."
        )
    if r1 is not None:
        lines.append(
            f" Among the {nv} disagreement cases where both annotators provided"
            f" explicit corrections, {nm} correction{'s' if nm != 1 else ''}"
            f" ({nm/nv*100:.1f}\\%) confirmed a genuine timing error"
            f" (assigned a different label by the automatic system),"
            f" while {nwt} ({nwt/nv*100:.1f}\\%) were corrected to"
            f" \\texttt{{well\\_timed}}."
        )
    lines.append(
        " These results indicate that the automatic timing component should be"
        " treated as a conservative first-pass screener: it identifies candidate"
        " timing problems for human review rather than making reliable final judgments."
    )
    lines.append("")
    lines.append("── Limitations bullet (paste directly):")
    lines.append("")
    if prec is not None and q2_available:
        lines.append(
            f"\\item \\textbf{{Conservative automatic timing bias.}}"
            f" Strict pairwise validation estimated automatic timing precision"
            f" at {prec:.1f}\\%: both annotators confirmed {n_tp} of"
            f" {n_auto_m} automatically flagged timing errors."
            f" The dominant correction direction was"
            f" \\texttt{{well\\_timed}} ({wt_count} of {total_c}"
            f" annotator-level corrections, {wt_pct*100:.1f}\\%)."
            f" Final timing judgments require human review."
        )
    elif prec is not None:
        lines.append(
            f"\\item \\textbf{{Pairwise automatic timing precision.}}"
            f" Both annotators confirmed {n_tp} of {n_auto_m}"
            f" automatically flagged timing errors ({prec:.1f}\\%)."
            f" This pairwise diagnostic does not replace the primary"
            f" three-annotator majority-consensus evaluation."
        )
    elif q2_available:
        lines.append(
            f"\\item \\textbf{{Conservative automatic timing bias.}}"
            f" The dominant correction direction was"
            f" \\texttt{{well\\_timed}} ({wt_count} of {total_c}"
            f" annotator-level corrections, {wt_pct*100:.1f}\\%)."
            f" Automatic timing labels require human review."
        )
    else:
        lines.append(
            "\\item The pairwise disagreement-subset analysis is exploratory "
            "and does not replace the primary three-annotator "
            "majority-consensus evaluation."
        )
    return "\n".join(lines)


# ── Main ──────────────────────────────────────────────────────────────────────

def main():
    parser = argparse.ArgumentParser(
        description="Compute TheraTime automatic timing reliability from human annotations."
    )
    parser.add_argument("--disagreements", required=True,
                        help="theratime_disagreements.csv from theratime_kappa.py")
    parser.add_argument("--annotations1",  default=None,
                        help="Full annotation CSV for annotator 1 (optional, enables Q2/Q3)")
    parser.add_argument("--annotations2",  default=None,
                        help="Full annotation CSV for annotator 2 (optional, enables Q2/Q3)")
    parser.add_argument("--out",           default="theratime_recall_report.csv")
    parser.add_argument("--json",          default="theratime_recall_summary.json")
    args = parser.parse_args()

    SEP = "=" * 80

    # ── Load disagreements ────────────────────────────────────────────────────
    disagree_df = pd.read_csv(args.disagreements).fillna("")
    print(SEP)
    print("TheraTime Automatic Timing Reliability Report  v2.2")
    print(SEP)
    print(f"Disagreements file : {args.disagreements}")
    print(f"Disagreement rows  : {len(disagree_df)}")
    print()

    # Detect the two annotators dynamically from the column names generated
    # by theratime_kappa.py. This supports pairs such as:
    #   Asmae / external
    #   Hasnae / Asmae
    #   Internal_1 / External
    try:
        annotator_1, annotator_2 = detect_annotator_prefixes(disagree_df)
    except ValueError as exc:
        print(f"ERROR: {exc}")
        print(f"Columns found: {list(disagree_df.columns)}")
        sys.exit(1)

    h_tc = f"{annotator_1}_timing"
    a_tc = f"{annotator_2}_timing"
    h_corr = f"{annotator_1}_timing_correction"
    a_corr = f"{annotator_2}_timing_correction"

    auto_c = next(
        (
            c
            for c in ["auto_timing", "timing_label", "predicted_timing"]
            if c in disagree_df.columns
        ),
        None,
    )

    print(f"Detected annotator pair: {annotator_1} vs {annotator_2}")
    print()

    # ── Q1: Subset recall ─────────────────────────────────────────────────────
    print("── Q1: Subset Recall (disagreements CSV only) ──────────────────────────")
    print("   SCOPE: Cases with stage/move disagreement where both annotators")
    print("   rejected the timing label AND provided corrections.")
    print("   NOT the system's overall precision — scoped to this subset only.")
    print()

    q1 = compute_subset_recall(disagree_df, h_tc, a_tc, h_corr, a_corr)

    print(f"   Both annotators rejected auto timing label : {q1['n_both_said_wrong']}")
    print(f"   Valid correction cases (both gave labels)  : {q1['n_valid_corrections']}")
    print(f"   Still genuinely mistimed (any error type)  : {q1['n_actually_mistimed']}")
    print(f"   Corrected to well_timed                    : {q1['n_actually_well_timed']}")
    if q1['subset_recall'] is not None:
        print(f"   Subset recall (within this scope)          : "
              f"{q1['subset_recall']:.4f} ({q1['subset_recall']*100:.1f}%)")
    print()
    print("   Status breakdown:")
    for k, v in sorted(q1['status_breakdown'].items(), key=lambda x: -x[1]):
        print(f"     {k:45s}: {v}")
    print()

    # ── Q2: Correction direction ──────────────────────────────────────────────
    q2 = {}
    if args.annotations1 and args.annotations2:
        print("── Q2: Correction Direction (full annotation files) ────────────────────")
        ann1 = drop_duplicate_columns(
            pd.read_csv(args.annotations1).fillna("")
        )
        ann2 = drop_duplicate_columns(
            pd.read_csv(args.annotations2).fillna("")
        )
        print(f"   Annotator 1 file : {args.annotations1} ({len(ann1)} rows)")
        print(f"   Annotator 2 file : {args.annotations2} ({len(ann2)} rows)")
        q2 = compute_correction_direction(ann1, ann2)
        if "error" in q2:
            print(f"   Warning: {q2['error']}")
            for diagnostic in q2.get("column_diagnostics", []):
                print(
                    "   Column detection "
                    f"({diagnostic['annotator']}): "
                    f"timing_correct={diagnostic['timing_correct_column']!r}, "
                    f"timing_correction={diagnostic['timing_correction_column']!r}"
                )
        else:
            print(f"   Total timing corrections  : {q2['total_corrections']}")
            print(f"   Toward well_timed         : {q2['well_timed_count']} "
                  f"({q2['well_timed_pct']*100:.1f}%)")
            print(f"   Toward other error types  :")
            for lbl, cnt in sorted(q2['mistimed_corrections'].items(),
                                    key=lambda x: -x[1]):
                print(f"     {lbl:35s}: {cnt}")
            print()

        # ── Q3: Auto precision ────────────────────────────────────────────────
        print("── Q3: Auto Timing Precision (most important for paper) ────────────────")
        q3 = compute_auto_precision(ann1, ann2)
        if "error" in q3:
            print(f"   Warning: {q3['error']}")
            q3 = {}
        else:
            print(f"   Auto-mistimed cases           : {q3['n_auto_mistimed']}")
            print(f"   Both confirmed correct (TP)   : {q3['n_both_confirmed_tp']}")
            print(f"   Both corrected to well_timed  : {q3['n_both_rejected_to_wt_fp']}")
            print(f"   Ambiguous                     : {q3['n_ambiguous']}")
            print(f"   AUTO TIMING PRECISION         : {q3['precision_pct']:.1f}%")
            print()
    else:
        print("── Q2 and Q3 skipped (no --annotations1 / --annotations2 provided) ────")
        print("   Provide full annotation CSVs to compute correction direction")
        print("   and auto precision — the most important numbers for the paper.")
        print()
        q3 = {}

    # ── Q4: Per-error breakdown ───────────────────────────────────────────────
    if auto_c:
        print("── Q4: Per-Error-Type Breakdown ────────────────────────────────────────")
        q4_df = compute_per_error_breakdown(
            disagree_df, h_tc, a_tc, h_corr, a_corr, auto_c)
        if not q4_df.empty:
            print(q4_df[["auto_error_type","n_in_disagreements",
                          "n_both_confirmed_correct","n_both_corrected_to_wt",
                          "confirmation_rate","fp_rate_to_wt",
                          "clinical_note"]].to_string(index=False))
            print()
            print("   Clinical priority note:")
            print("   delayed_safety false positives are MOST CONCERNING because they")
            print("   indicate the system flags safety responses as errors when they are not.")
            print("   Check these cases manually regardless of aggregate statistics.")
    else:
        q4_df = pd.DataFrame()
        print("── Q4 skipped (no auto_timing column in disagreements CSV) ─────────────")
    print()

    # ── Paper template ────────────────────────────────────────────────────────
    print(SEP)
    print(paper_template(q1, q2, q3))
    print(SEP)

    # ── Save outputs ──────────────────────────────────────────────────────────
    q1["detail_df"].to_csv(args.out, index=False)
    print(f"\nSaved detail CSV : {args.out}")

    summary = {
        "q1_subset_recall": {k: v for k, v in q1.items() if k != "detail_df"},
        "q2_correction_direction": q2,
        "q3_auto_precision": q3,
        "q4_per_error_rows": q4_df.to_dict(orient="records") if not q4_df.empty else [],
        "scope_note": (
            "subset_recall in Q1 applies only to cases in theratime_disagreements.csv "
            "where both annotators rejected the timing label. "
            "It is NOT the system's overall precision. "
            "Use Q3 auto_precision for the overall number."
        ),
    }
    with open(args.json, "w") as f:
        json.dump(summary, f, indent=2, default=str)
    print(f"Saved summary JSON: {args.json}")


if __name__ == "__main__":
    main()

Overwriting theratime_error_recall.py


In [32]:
!python theratime_kappa.py -f /kaggle/input/datasets/asmaeassmaebriouya/anotator-recalibrated/theratime_annotation_sample_300_hasnae_FINAL_RECALIBRATED.csv /kaggle/input/datasets/asmaeassmaebriouya/anotator-recalibrated/theratime_annotation_sample_300_asmae_FINAL_RECALIBRATED.csv --out-prefix hasnae_asmae

!python theratime_kappa.py -f /kaggle/input/datasets/asmaeassmaebriouya/anotator-recalibrated/theratime_annotation_sample_300_hasnae_FINAL_RECALIBRATED.csv /kaggle/input/datasets/asmaeassmaebriouya/anotator-recalibrated/theratime_annotation_sample_300_external_rechecked_with_move_unsure.csv --out-prefix hasnae_external

!python theratime_kappa.py -f /kaggle/input/datasets/asmaeassmaebriouya/anotator-recalibrated/theratime_annotation_sample_300_asmae_FINAL_RECALIBRATED.csv /kaggle/input/datasets/asmaeassmaebriouya/anotator-recalibrated/theratime_annotation_sample_300_external_rechecked_with_move_unsure.csv --out-prefix asmae_external


TheraTime Inter-Annotator Agreement Report
Annotator 1 : Hasnae (300 labelled)
Annotator 2 : Asmae (300 labelled)
File 1      : /kaggle/input/datasets/asmaeassmaebriouya/anotator-recalibrated/theratime_annotation_sample_300_hasnae_FINAL_RECALIBRATED.csv
File 2      : /kaggle/input/datasets/asmaeassmaebriouya/anotator-recalibrated/theratime_annotation_sample_300_asmae_FINAL_RECALIBRATED.csv
Common IDs  : 300

── Q1: Stage Classification Agreement ──────────────────
  Cohen's kappa : 0.624 (Substantial)
  Observed po   : 0.82
  Expected pe   : 0.5213
  Valid pairs   : 300 (skipped unsure/missing: 0)

── Q2: Move Classification Agreement ───────────────────
  Cohen's kappa : 0.8512 (Almost perfect)
  Observed po   : 0.93
  Expected pe   : 0.5296
  Valid pairs   : 300 (skipped unsure/missing: 0)

── Q3: Timing Label Agreement ──────────────────────────
  Cohen's kappa : 0.7328 (Substantial)
  Observed po   : 0.8667
  Expected pe   : 0.5011
  Valid pairs   : 300 (skipped unsure/missing: 0)

In [42]:
!python theratime_error_recall.py \
  --disagreements /kaggle/working/hasnae_external_disagreements.csv \
  --annotations1 /kaggle/input/datasets/asmaeassmaebriouya/anotator-recalibrated/theratime_annotation_sample_300_hasnae_FINAL_RECALIBRATED.csv \
  --annotations2 /kaggle/input/datasets/asmaeassmaebriouya/anotator-recalibrated/theratime_annotation_sample_300_external_rechecked_with_move_unsure.csv \
  --out hasnae_external_theratime_screening_recall_report.csv

TheraTime Automatic Timing Reliability Report  v2.2
Disagreements file : /kaggle/working/hasnae_external_disagreements.csv
Disagreement rows  : 145

Detected annotator pair: Hasnae vs external

── Q1: Subset Recall (disagreements CSV only) ──────────────────────────
   SCOPE: Cases with stage/move disagreement where both annotators
   rejected the timing label AND provided corrections.
   NOT the system's overall precision — scoped to this subset only.

   Both annotators rejected auto timing label : 52
   Valid correction cases (both gave labels)  : 37
   Still genuinely mistimed (any error type)  : 4
   Corrected to well_timed                    : 33
   Subset recall (within this scope)          : 0.1081 (10.8%)

   Status breakdown:
     well_timed_same_label                        : 33
     human_disagree_mistimed_vs_well_timed        : 15
     mistimed_different_label                     : 2
     mistimed_same_label                          : 2

── Q2: Correction Direction (full a